In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:


import os, random, shutil, time, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageFilter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

# ── Optional packages ────────────────────────────────────────────────────────
try:
    import umap as umap_lib
    HAS_UMAP = True
except ImportError:
    os.system("pip install umap-learn -q")
    try:
        import umap as umap_lib
        HAS_UMAP = True
    except:
        HAS_UMAP = False
        print("WARNING: umap-learn not available — UMAP skipped.")

try:
    from scipy.stats import wilcoxon as scipy_wilcoxon
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

SEED        = 42
N_SPLITS    = 4
EPOCHS      = 15
LR          = 1e-3
BATCH_SIZE  = 128
NUM_WORKERS = 2

# Full sweep — ALL 10 configurations
SWEEP_CONFIGS = [
    {"arch": "BaselineCNN",  "synth": 250},
    {"arch": "BaselineCNN",  "synth": 500},
    {"arch": "BaselineCNN",  "synth": 750},
    {"arch": "BaselineCNN",  "synth": 1000},
    {"arch": "BaselineCNN",  "synth": 1250},
    {"arch": "ResNet18_1ch", "synth": 250},
    {"arch": "ResNet18_1ch", "synth": 500},
    {"arch": "ResNet18_1ch", "synth": 750},
    {"arch": "ResNet18_1ch", "synth": 1000},
    {"arch": "ResNet18_1ch", "synth": 1250},
]

# For mixed test set: top N configs by balanced accuracy
TOP_N_FOR_MIXED_TEST = 3

# For t-SNE/UMAP: synthetic images to embed per class (visual only)
EMBED_SYNTH_N = 80
# For mixed test set generation
MIXED_SYNTH_N = 100

# Grad-CAM and perturbation settings
GRADCAM_N_CORRECT   = 16
GRADCAM_N_INCORRECT = 9

DATA_ROOT = Path("/kaggle/input/datasets/teja19129/g-pixel/96x192")
WORK_DIR  = Path("/kaggle/working")
OUT_DIR   = WORK_DIR / "thesis_pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
scaler  = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def get_raw(model):
    """
    Return the unwrapped model regardless of DataParallel wrapping.
    ALWAYS use this for save/load:
      SAVE: raw=get_raw(model); state={k:v.clone() for k,v in raw.state_dict().items()}
      LOAD: raw=get_raw(model); raw.load_state_dict(state)
    Never call model.state_dict() or model.load_state_dict() on a
    DataParallel-wrapped model — keys will have 'module.' prefix and
    load/save will mismatch.
    """
    return model.module if isinstance(model, nn.DataParallel) else model


def strip_dp(state_dict):
    """
    Safety net: strip 'module.' prefix if present.
    Not needed if you always save/load via get_raw(), but kept as a
    defensive measure for any state dict from an external source.
    """
    return {
        (k[len("module."):] if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }


# ══════════════════════════════════════════════════════════════════════════════
# DATA SETUP
# ══════════════════════════════════════════════════════════════════════════════

valid_exts     = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TRASH_FOLDERS  = {"0-EMPTY", "1-NOISE"}
all_folders    = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
STRUCT_FOLDERS = [f for f in all_folders if f not in TRASH_FOLDERS]

classes        = ["0-NOISE_EMPTY"] + STRUCT_FOLDERS
class_to_idx   = {cls: i for i, cls in enumerate(classes)}
idx_to_class   = {i: cls for cls, i in class_to_idx.items()}
NUM_CLASSES    = len(classes)
TRASH_IDX      = 0
STRUCTURED_IDX = list(range(1, NUM_CLASSES))

print(f"\n{NUM_CLASSES}-class setup:")
for cls, idx in class_to_idx.items():
    print(f"  {idx}: {cls}")

all_samples = []
for folder in TRASH_FOLDERS:
    cls_dir = DATA_ROOT / folder
    if cls_dir.exists():
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in valid_exts:
                all_samples.append((str(p), 0))
for folder in STRUCT_FOLDERS:
    cls_dir = DATA_ROOT / folder
    if cls_dir.exists():
        idx = class_to_idx[folder]
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in valid_exts:
                all_samples.append((str(p), idx))

paths  = np.array([p for p, _ in all_samples])
labels = np.array([y for _, y in all_samples])

print(f"\nReal data: {len(all_samples)} total")
for k, v in sorted(Counter(labels).items()):
    print(f"  {idx_to_class[k]:20s}: {v:5d}  ({100*v/len(all_samples):.1f}%)")

# ══════════════════════════════════════════════════════════════════════════════
# SYNTHETIC GENERATORS
# ══════════════════════════════════════════════════════════════════════════════

H, W = 96, 192


def save_img(arr, path):
    Image.fromarray((arr.astype(np.uint8) * 255)).save(path)


def bg(rng, prob=None):
    return (rng.random((H, W)) < (prob or rng.uniform(0.0001, 0.003))).astype(np.uint8)


def rand(arr, rng):
    img = Image.fromarray((arr.astype(np.uint8) * 255))
    if rng.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.2, 0.7))))
    arr = np.array(img).astype(np.float32) / 255.0
    if rng.random() < 0.5:
        arr = np.clip(arr + rng.normal(0, rng.uniform(0.01, 0.04), arr.shape), 0, 1)
    return (arr > rng.uniform(0.35, 0.65)).astype(np.uint8)


def gen_noise_empty(rng):
    arr = ((rng.random((H, W)) < rng.uniform(0.001, 0.05)).astype(np.uint8)
           if rng.random() < 0.5 else np.zeros((H, W), dtype=np.uint8))
    return rand(arr, rng)


def gen_zebra(rng):
    arr = bg(rng); bw = rng.integers(120, 170); oy = rng.integers(0, 16)
    x0 = rng.integers(0, max(1, W - bw)); d = rng.uniform(0.45, 1.0)
    for y in np.arange(oy, H, 16): arr[y, x0:x0+bw] = rng.random(bw) < d
    return rand(arr, rng)


def gen_okapi(rng):
    arr = bg(rng); bw = rng.integers(55, 95); oy = rng.integers(0, 16)
    x0 = rng.integers(0, max(1, W - bw)); d = rng.uniform(0.40, 1.0)
    for y in np.arange(oy, H, 16): arr[y, x0:x0+bw] = rng.random(bw) < d
    return rand(arr, rng)


def gen_cheetah(rng):
    arr = bg(rng, rng.uniform(0.00005, 0.001))
    ox, oy = rng.integers(0, 2), rng.integers(0, 2)
    pat = (rng.random((H//2+1, W//2+1)) < rng.uniform(0.90, 0.995)).astype(np.uint8)
    arr[oy::2, ox::2] = pat[:arr[oy::2, ox::2].shape[0], :arr[oy::2, ox::2].shape[1]]
    if rng.random() < 0.7:
        bh = rng.integers(6, 10)
        for y in range(rng.integers(0, 16), H, 16): arr[y:y+bh, :] = 0
    return rand(arr, rng)


def gen_grid(rng):
    arr = bg(rng); pw = 12 * rng.integers(12, 14)
    x0 = rng.integers(0, max(1, W - pw)); y0 = rng.integers(0, 16)
    d = rng.uniform(0.85, 0.99)
    for y in np.arange(y0, H, 16):
        cols = np.arange(x0, x0+pw, 12); arr[y, cols] = rng.random(len(cols)) < d
    return rand(arr, rng)


def gen_diagonal(rng):
    arr = bg(rng, rng.uniform(0.00005, 0.0005))
    ox, oy = rng.integers(0, 12), rng.integers(0, 16)
    bp = np.array([1, 1, 0, 0] if rng.random() < 0.5 else [1, 0, 0, 0])
    pat = np.roll(np.tile(bp, rng.integers(3, 6)), rng.integers(0, len(bp)*3))
    cols = np.arange(ox, W, 12); n = min(len(cols), len(pat))
    cols, pat = cols[:n], pat[:n]
    for i, y in enumerate(np.arange(oy, H, 16)):
        sh = np.roll(pat, i); arr[y, cols] = rng.random(len(sh)) < (0.92 * sh)
    if rng.random() < 0.5: arr = np.fliplr(arr)
    return rand(arr, rng)


def gen_blinds(rng):
    arr = bg(rng); bb = rng.integers(2, 6)
    x1 = rng.choice([0, rng.integers(0, max(1, W//3))])
    x2 = rng.choice([W, W - rng.integers(0, max(1, W//3))])
    if x2 <= x1: x1, x2 = 0, W
    w = x2 - x1; oy = rng.integers(0, max(1, 16-bb))
    bp = np.linspace(rng.uniform(0.15, 0.90), rng.uniform(0.15, 0.90), bb)
    pb = np.array([rng.random(w) < bp[r] for r in range(bb)], dtype=np.uint8)
    con = rng.uniform(0.55, 0.95)
    for by in range(oy, H, 16):
        bh = min(bb, H - by)
        for r in range(bh): arr[by+r, x1:x2] = rng.random(w) < (con * pb[r])
    return rand(arr, rng)


def gen_morse(rng):
    arr = bg(rng); seg = rng.integers(4, 8); rep = rng.integers(6, 10)
    pw = min(seg*2*rep, W-1); x0 = rng.integers(0, max(1, W-pw))
    y0 = rng.integers(0, 16)
    for y in np.arange(y0, H, 16):
        bits = (rng.random(pw) < rng.uniform(0.85, 0.99)).astype(np.uint8)
        for s in range(0, pw, seg*2): bits[s:min(s+seg, pw)] = 1 - bits[s:min(s+seg, pw)]
        arr[y, x0:x0+pw] = bits
    return rand(arr, rng)


GEN_FUNCS = {
    "0-NOISE_EMPTY": gen_noise_empty, "2-ZEBRA": gen_zebra,
    "3-OKAPI": gen_okapi,             "4-CHEETAH": gen_cheetah,
    "5-GRID": gen_grid,               "6-DIAGONAL": gen_diagonal,
    "7-BLINDS": gen_blinds,           "8-MORSE": gen_morse,
}


def generate_synth(n_per_class, seed, struct_only=True, tag="train"):
    """Generate synthetic images. struct_only=True skips the trash class."""
    rng_l = np.random.default_rng(seed)
    root  = WORK_DIR / f"synth_{tag}_s{seed}_n{n_per_class}"
    if root.exists(): shutil.rmtree(root)
    root.mkdir(parents=True, exist_ok=True)
    cls_list = STRUCT_FOLDERS if struct_only else classes
    samples  = []
    for cls in cls_list:
        d = root / cls; d.mkdir()
        for i in range(n_per_class):
            save_img(GEN_FUNCS[cls](rng_l), d / f"{cls}_{i:05d}.png")
        idx = class_to_idx[cls]
        for p in sorted(d.iterdir()): samples.append((str(p), idx))
    return samples

# ══════════════════════════════════════════════════════════════════════════════
# DATASET + MODELS
# ══════════════════════════════════════════════════════════════════════════════

t1ch = transforms.Compose([
    transforms.Resize((96, 192)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])


class DefectDS(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p, lbl = self.samples[idx]
        return t1ch(Image.open(p).convert("L")), lbl


def make_loader(samples, shuffle=False, drop_last=False):
    return DataLoader(
        DefectDS(samples), batch_size=BATCH_SIZE,
        shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=(NUM_WORKERS > 0),
        drop_last=drop_last,
    )


def create_model(arch):
    if arch == "BaselineCNN":
        class BaselineCNN(nn.Module):
            def __init__(self, n=NUM_CLASSES):
                super().__init__()
                self.features = nn.Sequential(
                    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                )
                self.classifier = nn.Sequential(
                    nn.Flatten(),
                    nn.Linear(64*12*24, 128), nn.ReLU(), nn.Dropout(0.3),
                    nn.Linear(128, n),
                )
            def forward(self, x): return self.classifier(self.features(x))
        m = BaselineCNN()
    elif arch == "ResNet18_1ch":
        m = models.resnet18(weights=None)
        orig = m.conv1
        m.conv1 = nn.Conv2d(1, orig.out_channels, orig.kernel_size,
                            orig.stride, orig.padding, bias=orig.bias)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        m = nn.DataParallel(m)
    return m.to(device)

# ══════════════════════════════════════════════════════════════════════════════
# METRICS + TRAINING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def full_metrics(y_true, y_pred, label=""):
    acc  = float(accuracy_score(y_true, y_pred))
    mf1  = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    bal  = float(balanced_accuracy_score(y_true, y_pred))
    _, rec, _, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    struct_recall = float(np.mean([rec[i] for i in STRUCTURED_IDX]))
    if label:
        print(f"\n  [{label}]")
        print(f"    Accuracy:           {acc:.4f}")
        print(f"    Micro F1:           {mf1:.4f}")
        print(f"    Balanced Accuracy:  {bal:.4f}")
        print(f"    Struct Mean Recall: {struct_recall:.4f}  ← KEY")
        for i, cls in enumerate(classes):
            tag = " [TRASH]" if i == TRASH_IDX else ""
            print(f"    {cls+tag:22s}: recall={rec[i]:.3f}  (n={int(sup[i])})")
    return {"accuracy": acc, "micro_f1": mf1, "balanced_accuracy": bal,
            "structured_recall": struct_recall,
            "per_class_recall": rec.tolist(), "support": sup.tolist()}


def run_epoch(model, loader, crit, opt=None):
    model.train() if opt else model.eval()
    ls, n, yt, yp = 0.0, 0, [], []
    for imgs, lbl in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbl  = lbl.to(device, non_blocking=True)
        if opt: opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(opt is not None):
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(imgs); loss = crit(out, lbl)
            if opt:
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
        ls += loss.item() * imgs.size(0); n += imgs.size(0)
        yp.extend(out.argmax(1).detach().cpu().tolist())
        yt.extend(lbl.cpu().tolist())
    return ls/n, accuracy_score(yt, yp), f1_score(yt, yp, average="micro", zero_division=0)


def evaluate_model(model, loader):
    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for imgs, lbl in loader:
            imgs = imgs.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(imgs)
            yp.extend(out.argmax(1).cpu().tolist())
            yt.extend(lbl.tolist())
    return np.array(yt), np.array(yp)


def save_cm(yt, yp, title, path):
    cm      = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    cm_norm = np.nan_to_num(cm.astype(float) / cm.sum(axis=1, keepdims=True))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, t in [(axes[0], cm, f"{title}\nRaw CM (n={len(yt)})"),
                        (axes[1], cm_norm, "Normalised")]:
        im = ax.imshow(data, cmap="Blues")
        plt.colorbar(im, ax=ax)
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(NUM_CLASSES))
        ax.set_yticklabels(classes, fontsize=8)
        ax.set_title(t, fontsize=11)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                v = data[i, j]
                txt = f"{v:.2f}" if isinstance(v, float) else str(int(v))
                ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                        color="white" if v > data.max() * 0.6 else "black")
    plt.tight_layout()
    plt.savefig(path, dpi=180); plt.close()


def run_one_config(arch, synth_n, seed=SEED):
    """
    Run one config through 4-fold CV.
    Returns: (best_model_state, fold_metrics_list, fold_accs, fold_bals)
    best_model_state = the fold checkpoint with highest val Micro F1.
    """
    set_seed(seed)
    train_synth = generate_synth(synth_n, seed, struct_only=True, tag=f"tr{synth_n}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    best_overall_mf1 = -1
    best_model_state = None
    fold_metrics = []
    fold_accs, fold_bals = [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(paths, labels), 1):
        real_tr = [(paths[i], int(labels[i])) for i in tr_idx]
        real_va = [(paths[i], int(labels[i])) for i in va_idx]
        tr_loader = make_loader(real_tr + train_synth, shuffle=True, drop_last=True)
        va_loader = make_loader(real_va)

        model = create_model(arch)
        crit  = nn.CrossEntropyLoss()
        opt   = optim.Adam(model.parameters(), lr=LR)
        best_fold_mf1, best_fold_state = -1, None

        for epoch in range(EPOCHS):
            _, tr_acc, _      = run_epoch(model, tr_loader, crit, opt)
            _, va_acc, va_mf1 = run_epoch(model, va_loader, crit)
            print(f"    Ep {epoch+1:02d} | tr={tr_acc:.4f} | va={va_acc:.4f}  mF1={va_mf1:.4f}")
            if va_mf1 > best_fold_mf1:
                best_fold_mf1  = va_mf1
                _raw = get_raw(model)
                best_fold_state = {k: v.detach().cpu().clone()
                                   for k, v in _raw.state_dict().items()}

        get_raw(model).load_state_dict(best_fold_state)
        yt, yp = evaluate_model(model, va_loader)
        m = full_metrics(yt, yp, f"Fold {fold} val")
        fold_metrics.append(m)
        fold_accs.append(m["accuracy"])
        fold_bals.append(m["balanced_accuracy"])

        if best_fold_mf1 > best_overall_mf1:
            best_overall_mf1 = best_fold_mf1
            best_model_state = best_fold_state

        del model, opt, crit; torch.cuda.empty_cache()

    return best_model_state, fold_metrics, fold_accs, fold_bals

# ══════════════════════════════════════════════════════════════════════════════
# GENERATE MIXED TEST SET  (one time, shared across all analysis blocks)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("GENERATING MIXED TEST SET")
print(f"  {MIXED_SYNTH_N} new synthetic images per class (seed+999 — never seen in training)")
print("=" * 70)

test_synth     = generate_synth(MIXED_SYNTH_N, SEED+999, struct_only=False, tag="test")
real_test_set  = list(zip(paths.tolist(), labels.tolist()))
synth_test_set = test_synth
mixed_test_set = real_test_set + synth_test_set

print(f"  Real-only:  {len(real_test_set):5d}")
print(f"  Synth-only: {len(synth_test_set):5d}  ({MIXED_SYNTH_N}/class × {NUM_CLASSES})")
print(f"  Mixed:      {len(mixed_test_set):5d}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK 0: REAL-ONLY FLOOR  (critical control — no synthetic data)
# ══════════════════════════════════════════════════════════════════════════════
# This is the most important control experiment. Without it you cannot
# scientifically claim "synthetic augmentation helps."

print("\n" + "=" * 70)
print("BLOCK 0: REAL-ONLY FLOOR")
print("Train on real images ONLY. No synthetic data.")
print("This is the baseline to compare all synth configs against.")
print("=" * 70)

floor_results = {}
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    print(f"\n  Architecture: {arch}")
    set_seed(SEED)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_accs, fold_bals = [], []
    best_mf1, best_state = -1, None

    for fold, (tr_idx, va_idx) in enumerate(skf.split(paths, labels), 1):
        real_tr = [(paths[i], int(labels[i])) for i in tr_idx]
        real_va = [(paths[i], int(labels[i])) for i in va_idx]
        tr_loader = make_loader(real_tr, shuffle=True, drop_last=False)
        va_loader = make_loader(real_va)
        model = create_model(arch); crit = nn.CrossEntropyLoss()
        opt = optim.Adam(model.parameters(), lr=LR)
        bfm, bfs = -1, None
        for epoch in range(EPOCHS):
            run_epoch(model, tr_loader, crit, opt)
            _, va_acc, va_mf1 = run_epoch(model, va_loader, crit)
            if va_mf1 > bfm:
                bfm = va_mf1
                _raw = get_raw(model)
                bfs = {k: v.detach().cpu().clone() for k, v in _raw.state_dict().items()}
        get_raw(model).load_state_dict(bfs)
        yt, yp = evaluate_model(model, va_loader)
        m = full_metrics(yt, yp)
        fold_accs.append(m["accuracy"]); fold_bals.append(m["balanced_accuracy"])
        if bfm > best_mf1: best_mf1 = bfm; best_state = bfs
        del model, opt, crit; torch.cuda.empty_cache()
        print(f"    Fold {fold}: acc={m['accuracy']:.4f}  bal={m['balanced_accuracy']:.4f}")

    floor_results[arch] = {
        "fold_accs": fold_accs, "fold_bals": fold_bals,
        "acc_mean": float(np.mean(fold_accs)), "acc_std": float(np.std(fold_accs)),
        "bal_mean": float(np.mean(fold_bals)), "bal_std": float(np.std(fold_bals)),
        "best_state": best_state,
    }
    print(f"  → {arch} REAL-ONLY: acc={np.mean(fold_accs):.4f}±{np.std(fold_accs):.4f} "
          f"bal={np.mean(fold_bals):.4f}±{np.std(fold_bals):.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# FULL SWEEP: ALL 10 CONFIGURATIONS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("FULL SWEEP: ALL 10 CONFIGURATIONS")
print("BaselineCNN  × synth 250/500/750/1000/1250")
print("ResNet18_1ch × synth 250/500/750/1000/1250")
print("=" * 70)

sweep_results = []   # list of dicts, one per config
saved_states  = {}   # label → {"arch": ..., "state": ...}

for cfg in SWEEP_CONFIGS:
    arch  = cfg["arch"]
    synth = cfg["synth"]
    label = f"{arch}_s{synth}"

    print(f"\n{'#'*70}")
    print(f"CONFIG: {label}  (synth={synth}/structured class)")
    print(f"{'#'*70}")

    best_state, fold_metrics, fold_accs, fold_bals = run_one_config(arch, synth)

    saved_states[label] = {"arch": arch, "state": best_state}

    acc_mean = float(np.mean(fold_accs))
    acc_std  = float(np.std(fold_accs))
    bal_mean = float(np.mean(fold_bals))
    bal_std  = float(np.std(fold_bals))
    sr_mean  = float(np.mean([m["structured_recall"] for m in fold_metrics]))

    sweep_results.append({
        "label": label, "arch": arch, "synth": synth,
        "acc_mean": acc_mean, "acc_std": acc_std,
        "bal_mean": bal_mean, "bal_std": bal_std,
        "struct_recall_mean": sr_mean,
        "fold_accs": fold_accs, "fold_bals": fold_bals,
    })

    print(f"\n  → {label}: acc={acc_mean:.4f}±{acc_std:.4f}  "
          f"bal={bal_mean:.4f}±{bal_std:.4f}  struct={sr_mean:.4f}")

# ── Save sweep CSV ─────────────────────────────────────────────────────────────
sweep_df = pd.DataFrame([{k: v for k, v in r.items()
                           if k not in ("fold_accs", "fold_bals")}
                          for r in sweep_results])
sweep_df.to_csv(OUT_DIR / "sweep_results.csv", index=False)

# ── Identify best configs ──────────────────────────────────────────────────────
# Sort by balanced accuracy (primary industrial metric — equal class weight)
sweep_sorted = sorted(sweep_results, key=lambda r: r["bal_mean"], reverse=True)

best_overall   = sweep_sorted[0]          # single best model for Grad-CAM / t-SNE / perturb
top_n_configs  = sweep_sorted[:TOP_N_FOR_MIXED_TEST]   # top 3 for mixed test evaluation

print("\n" + "=" * 70)
print("SWEEP COMPLETE — RESULTS RANKED BY BALANCED ACCURACY")
print("=" * 70)
print(f"\n{'Rank':4s} {'Config':22s} {'Acc Mean':>10} {'Acc Std':>9} "
      f"{'Bal Acc':>9} {'Bal Std':>9} {'Struct':>8}")
print("-" * 77)
for i, r in enumerate(sweep_sorted, 1):
    flag = " ← BEST" if i == 1 else (f" ← top {i}" if i <= TOP_N_FOR_MIXED_TEST else "")
    print(f"{i:4d} {r['label']:22s} {r['acc_mean']:>10.4f} {r['acc_std']:>9.4f} "
          f"{r['bal_mean']:>9.4f} {r['bal_std']:>9.4f} {r['struct_recall_mean']:>8.4f}{flag}")

print(f"\nBest model: {best_overall['label']}")
print(f"  Balanced Accuracy: {best_overall['bal_mean']:.4f} ± {best_overall['bal_std']:.4f}")
print(f"  Accuracy:          {best_overall['acc_mean']:.4f} ± {best_overall['acc_std']:.4f}")
print(f"\nTop {TOP_N_FOR_MIXED_TEST} for mixed test: "
      f"{[r['label'] for r in top_n_configs]}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK A: MIXED TEST EVALUATION — TOP N CONFIGS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK A: MIXED TEST SET — TOP {TOP_N_FOR_MIXED_TEST} CONFIGS")
print("Determined from this run's sweep results.")
print("=" * 70)

mixed_test_rows = []
for r in top_n_configs:
    label = r["label"]
    arch  = r["arch"]
    cfg_dir = OUT_DIR / label
    cfg_dir.mkdir(exist_ok=True)

    model = create_model(arch)
    get_raw(model).load_state_dict(saved_states[label]["state"])
    model.eval()

    for test_name, test_data in [
        ("real_only",  real_test_set),
        ("synth_only", synth_test_set),
        ("mixed",      mixed_test_set),
    ]:
        loader = make_loader(test_data)
        yt, yp = evaluate_model(model, loader)
        m = full_metrics(yt, yp, f"{label} | {test_name} (n={len(test_data)})")
        report = classification_report(yt, yp, target_names=classes, zero_division=0)
        (cfg_dir / f"test_{test_name}_report.txt").write_text(report)
        save_cm(yt, yp, f"{label} | {test_name}", cfg_dir / f"cm_{test_name}.png")
        mixed_test_rows.append({
            "config": label, "test_set": test_name, "n": len(test_data),
            "accuracy": m["accuracy"], "balanced_acc": m["balanced_accuracy"],
            "struct_recall": m["structured_recall"],
            **{f"recall_{classes[i]}": m["per_class_recall"][i]
               for i in range(NUM_CLASSES)},
        })

    del model; torch.cuda.empty_cache()

pd.DataFrame(mixed_test_rows).to_csv(OUT_DIR / "mixed_test_results.csv", index=False)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK B + C: t-SNE AND UMAP — BEST MODEL
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK B+C: t-SNE / UMAP — Best model: {best_overall['label']}")
print("=" * 70)

COLORS = ["#455A64","#00796B","#E53935","#6A1B9A",
          "#F57C00","#1E2761","#2E7D32","#795548"]

embed_synth = generate_synth(EMBED_SYNTH_N, SEED+1, struct_only=True, tag="embed")

class TripletDS(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y, real = self.items[i]
        return t1ch(Image.open(p).convert("L")), y, int(real)

all_triplets = ([(p, y, True)  for p, y in real_test_set] +
                [(p, y, False) for p, y in embed_synth])

# Build embedding model — always BaselineCNN (architecture is fixed for this analysis)
best_arch = best_overall["arch"]
emb_model = create_model(best_arch)
emb_raw = get_raw(emb_model)
emb_raw.load_state_dict(saved_states[best_overall["label"]]["state"])
emb_raw.eval()

# Hook the penultimate layer
feats_collect = []
if best_arch == "BaselineCNN":
    hook_layer = emb_raw.classifier[2]   # ReLU output after Linear(18432→128)
else:
    hook_layer = emb_raw.avgpool          # GAP (512-d)

hook = hook_layer.register_forward_hook(
    lambda m, inp, out: feats_collect.append(out.detach().cpu()))

Y_all, R_all = [], []
loader_emb = DataLoader(TripletDS(all_triplets), batch_size=64,
                        shuffle=False, num_workers=NUM_WORKERS)
with torch.no_grad():
    for imgs, yl, rl in loader_emb:
        imgs = imgs.to(device, non_blocking=True)
        emb_raw(imgs)
        Y_all.extend(yl.tolist()); R_all.extend(rl.tolist())

hook.remove()
del emb_model, emb_raw; torch.cuda.empty_cache()

F_all = torch.cat(feats_collect, 0).numpy()
if F_all.ndim > 2: F_all = F_all.reshape(F_all.shape[0], -1)
Y_all = np.array(Y_all); R_all = np.array(R_all)
print(f"  Embedding matrix: {F_all.shape}")


def plot_embeddings(Z, Y, R, title, path):
    from matplotlib.lines import Line2D
    fig, ax = plt.subplots(figsize=(11, 9))
    for ci, cls in enumerate(classes):
        col = COLORS[ci % len(COLORS)]
        rm = (Y == ci) & (R == 1); sm = (Y == ci) & (R == 0)
        if rm.any(): ax.scatter(Z[rm,0], Z[rm,1], c=col, s=55, alpha=0.85, marker="o")
        if sm.any(): ax.scatter(Z[sm,0], Z[sm,1], c=col, s=28, alpha=0.50, marker="x")
    leg_cls  = [Line2D([0],[0],marker="o",color="w",markerfacecolor=COLORS[i],
                       markersize=9, label=classes[i]) for i in range(NUM_CLASSES)]
    leg_type = [Line2D([0],[0],marker="o",color="k",markersize=7,label="real (circles)"),
                Line2D([0],[0],marker="x",color="k",markersize=7,label="synth (crosses)")]
    ax.legend(handles=leg_cls+leg_type, loc="best", fontsize=8, ncol=2)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Component 1"); ax.set_ylabel("Component 2")
    ax.grid(True, alpha=0.2); plt.tight_layout()
    plt.savefig(path, dpi=180); plt.close()


print("  Running t-SNE (perplexity=30)...")
Z_tsne = TSNE(n_components=2, perplexity=30, n_iter=1000,
              random_state=SEED, n_jobs=-1).fit_transform(F_all)
plot_embeddings(Z_tsne, Y_all, R_all,
                f"t-SNE — Penultimate Layer ({best_overall['label']})\n"
                "Real (circles) vs Synthetic (crosses)",
                OUT_DIR / "tsne_embeddings.png")
print("→ Saved tsne_embeddings.png")

if HAS_UMAP:
    print("  Running UMAP (n_neighbors=15)...")
    Z_umap = umap_lib.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                            random_state=SEED, n_jobs=-1).fit_transform(F_all)
    plot_embeddings(Z_umap, Y_all, R_all,
                    f"UMAP — Penultimate Layer ({best_overall['label']})\n"
                    "Real (circles) vs Synthetic (crosses)",
                    OUT_DIR / "umap_embeddings.png")
    print("→ Saved umap_embeddings.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK D: GRAD-CAM — BEST MODEL
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK D: GRAD-CAM — Best model: {best_overall['label']}")
print("=" * 70)


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.grads = None; self.acts = None; self._h = []
        self._h.append(target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, "acts", o)))
        self._h.append(target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, "grads", go[0])))

    def compute(self, img_t, cls_idx):
        self.model.zero_grad()
        out = self.model(img_t)
        out[0, cls_idx].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(H, W), mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), out.argmax(1).item()

    def remove(self):
        for h in self._h: h.remove()


# Load best model for Grad-CAM
# NOTE: Grad-CAM requires BaselineCNN architecture for features[6] hook.
# If best model is ResNet18, we fall back to the best BaselineCNN config.
gc_arch  = best_overall["arch"]
gc_label = best_overall["label"]
if gc_arch != "BaselineCNN":
    # Find best BaselineCNN config
    best_cnn = next((r for r in sweep_sorted if r["arch"] == "BaselineCNN"), None)
    if best_cnn:
        gc_arch  = "BaselineCNN"
        gc_label = best_cnn["label"]
        print(f"  Best overall is {best_overall['arch']} — Grad-CAM using best "
              f"BaselineCNN: {gc_label}")

gc_model = create_model("BaselineCNN")
gc_raw = get_raw(gc_model)
gc_raw.load_state_dict(saved_states[gc_label]["state"])
gc_raw.eval()

gc = GradCAM(gc_raw, gc_raw.features[6])

correct_items, incorrect_items = [], []
all_pil = [Image.open(p).convert("L") for p, _ in real_test_set]

for pil_img, true_lbl in zip(all_pil, labels):
    tensor = t1ch(pil_img).unsqueeze(0).to(device)
    cam_map, pred = gc.compute(tensor, int(true_lbl))
    raw_arr = np.array(pil_img.resize((W, H)))
    item = (raw_arr, int(true_lbl), pred, cam_map)
    if pred == int(true_lbl): correct_items.append(item)
    else:                     incorrect_items.append(item)

gc.remove()
del gc_model, gc_raw; torch.cuda.empty_cache()

print(f"  Correct predictions:   {len(correct_items)}")
print(f"  Incorrect predictions: {len(incorrect_items)}")


def save_gradcam_panel(items, n, title, path):
    items = items[:n]
    cols  = 4; rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols*2, figsize=(cols*4, rows*2.6))
    axes = axes.flatten()
    for k, (raw_img, true_lbl, pred_lbl, cam_map) in enumerate(items):
        ax_img = axes[k*2]; ax_cam = axes[k*2+1]
        ax_img.imshow(raw_img, cmap="gray", aspect="auto")
        ax_img.set_title(f"True: {classes[true_lbl]}", fontsize=7); ax_img.axis("off")
        ax_cam.imshow(raw_img, cmap="gray", alpha=0.55, aspect="auto")
        ax_cam.imshow(cam_map, cmap="jet", alpha=0.55, aspect="auto", vmin=0, vmax=1)
        col = "green" if pred_lbl == true_lbl else "red"
        ax_cam.set_title(f"Pred: {classes[pred_lbl]}", fontsize=7, color=col)
        ax_cam.axis("off")
    for j in range(len(items)*2, len(axes)): axes[j].axis("off")
    plt.suptitle(title, fontsize=11, fontweight="bold")
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()


save_gradcam_panel(correct_items, GRADCAM_N_CORRECT,
                   f"Grad-CAM — Correct Predictions ({gc_label})\n"
                   "Left: raw image.  Right: activation overlay (red = most attended).",
                   OUT_DIR / "gradcam_correct.png")
save_gradcam_panel(incorrect_items, GRADCAM_N_INCORRECT,
                   f"Grad-CAM — Incorrect Predictions ({gc_label})\n"
                   "Left: raw image.  Right: activation overlay.  Title = predicted class.",
                   OUT_DIR / "gradcam_incorrect.png")
print("→ Saved gradcam_correct.png + gradcam_incorrect.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK E: PERTURBATION ROBUSTNESS — BEST MODEL
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK E: PERTURBATION ROBUSTNESS — Best model: {best_overall['label']}")
print("4-fold mean ± std on real validation images.")
print("=" * 70)


def apply_perturbation(img_tensor, ptype):
    arr = img_tensor.clone().squeeze().numpy()
    if   ptype == "brightness_up":    arr = np.clip(arr + 0.15, 0, 1)
    elif ptype == "brightness_down":  arr = np.clip(arr - 0.15, 0, 1)
    elif ptype == "gaussian_noise":
        arr = np.clip(arr + np.random.normal(0, 0.03, arr.shape), 0, 1)
    elif ptype == "translation":      arr = np.roll(arr, shift=3, axis=1)
    elif ptype == "rotation":
        from scipy.ndimage import rotate as ndrotate
        arr = np.clip(ndrotate(arr, angle=3, reshape=False, mode="nearest"), 0, 1)
    elif ptype == "threshold_shift":  arr = (arr > 0.55).astype(np.float32)
    elif ptype == "gaussian_blur":
        from PIL import Image as PILI, ImageFilter as IIF
        pil = PILI.fromarray((arr*255).astype(np.uint8))
        arr = np.array(pil.filter(IIF.GaussianBlur(radius=0.6))).astype(np.float32)/255.0
    return torch.tensor(arr).unsqueeze(0).unsqueeze(0).float()


PERTURBATIONS = ["clean", "brightness_up", "brightness_down",
                 "gaussian_noise", "translation", "rotation",
                 "threshold_shift", "gaussian_blur"]

p_arch  = best_overall["arch"]
p_label = best_overall["label"]
p_model = create_model(p_arch)
p_raw = get_raw(p_model)
p_raw.load_state_dict(saved_states[p_label]["state"])
p_raw.eval()

skf_p = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
pert_fold_bals = defaultdict(list)

for fold, (_, va_idx) in enumerate(skf_p.split(paths, labels), 1):
    real_va     = [(paths[i], int(labels[i])) for i in va_idx]
    imgs_loaded = [t1ch(Image.open(p).convert("L")) for p, _ in real_va]
    true_labels = [y for _, y in real_va]
    for ptype in PERTURBATIONS:
        yt, yp = [], []
        for img_t, ty in zip(imgs_loaded, true_labels):
            pt = apply_perturbation(img_t.unsqueeze(0), ptype).to(device)
            with torch.no_grad(): pred = p_raw(pt).argmax(1).item()
            yt.append(ty); yp.append(pred)
        pert_fold_bals[ptype].append(balanced_accuracy_score(yt, yp))

del p_model, p_raw; torch.cuda.empty_cache()

clean_mean  = np.mean(pert_fold_bals["clean"])
pert_rows   = []
for ptype in PERTURBATIONS:
    m, s = np.mean(pert_fold_bals[ptype]), np.std(pert_fold_bals[ptype])
    pert_rows.append({"perturbation": ptype, "bal_acc_mean": round(m, 4),
                      "bal_acc_std": round(s, 4), "drop": round(clean_mean - m, 4)})
    flag = "  ← pipeline constraint" if ptype == "gaussian_blur" else ""
    print(f"  {ptype:20s}: bal={m:.4f}±{s:.4f}  drop={clean_mean-m:+.4f}{flag}")

pd.DataFrame(pert_rows).to_csv(OUT_DIR / "perturbation_results.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 5))
names = [r["perturbation"] for r in pert_rows]
means = [r["bal_acc_mean"] for r in pert_rows]
stds  = [r["bal_acc_std"]  for r in pert_rows]
bar_colors = ["#2E7D32" if n == "clean" else
              "#E53935" if n == "gaussian_blur" else "#1E2761" for n in names]
ax.bar(range(len(names)), means, yerr=stds, capsize=4, color=bar_colors,
       edgecolor="white", linewidth=0.5)
ax.axhline(clean_mean, color="gray", ls="--", lw=1.5, label=f"Clean ({clean_mean:.4f})")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20, ha="right", fontsize=10)
ax.set_ylabel("Balanced Accuracy"); ax.set_ylim(0.4, 1.02)
ax.set_title(f"Perturbation Robustness — {p_label}\n4-fold mean ± std on real validation",
             fontsize=12, fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(OUT_DIR / "perturbation_robustness.png", dpi=180); plt.close()
print("→ Saved perturbation_robustness.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK F: WILCOXON STATISTICAL TEST
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK F: WILCOXON SIGNED-RANK TEST")
print("Compare best BaselineCNN vs best ResNet18_1ch by fold-level accuracy.")
print("=" * 70)

best_cnn_r = next((r for r in sweep_sorted if r["arch"] == "BaselineCNN"), None)
best_r18_r = next((r for r in sweep_sorted if r["arch"] == "ResNet18_1ch"), None)

wilcoxon_txt = []
if best_cnn_r and best_r18_r:
    cnn_f = best_cnn_r["fold_accs"]
    r18_f = best_r18_r["fold_accs"]
    print(f"  Best CNN: {best_cnn_r['label']}  folds={[f'{v:.4f}' for v in cnn_f]}  mean={np.mean(cnn_f):.4f}")
    print(f"  Best R18: {best_r18_r['label']}  folds={[f'{v:.4f}' for v in r18_f]}  mean={np.mean(r18_f):.4f}")
    wilcoxon_txt.append(f"Best CNN: {best_cnn_r['label']}  folds={cnn_f}  mean={np.mean(cnn_f):.4f}")
    wilcoxon_txt.append(f"Best R18: {best_r18_r['label']}  folds={r18_f}  mean={np.mean(r18_f):.4f}")
    if HAS_SCIPY:
        try:
            stat, p = scipy_wilcoxon(cnn_f, r18_f)
            sig = p < 0.05
            print(f"  Wilcoxon: W={stat:.4f}  p={p:.4f}  significant={sig}")
            wilcoxon_txt.append(f"Wilcoxon W={stat:.4f}  p={p:.4f}  significant={sig}")
        except Exception as e:
            print(f"  Wilcoxon failed: {e}")
            wilcoxon_txt.append(f"Wilcoxon failed: {e}")

wilcoxon_txt.append("\nREAL-ONLY FLOOR:")
for arch, fr in floor_results.items():
    line = f"  {arch}: acc={fr['acc_mean']:.4f}±{fr['acc_std']:.4f}  bal={fr['bal_mean']:.4f}±{fr['bal_std']:.4f}"
    wilcoxon_txt.append(line); print(line)

(OUT_DIR / "statistical_test.txt").write_text("\n".join(wilcoxon_txt))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK G: MODEL EFFICIENCY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK G: MODEL EFFICIENCY")
print("=" * 70)

dummy = torch.zeros(1, 1, H, W).to(device)
eff_rows = []
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    m = create_model(arch)
    raw = m.module if isinstance(m, nn.DataParallel) else m
    params = sum(p.numel() for p in raw.parameters())
    tmp = WORK_DIR / f"tmp_{arch}.pt"
    torch.save(raw.state_dict(), tmp); size_mb = os.path.getsize(tmp)/1e6; tmp.unlink()
    m.eval()
    with torch.no_grad():
        for _ in range(10): m(dummy)
    times = []
    with torch.no_grad():
        for _ in range(100):
            t0 = time.perf_counter(); m(dummy)
            if device.type == "cuda": torch.cuda.synchronize()
            times.append((time.perf_counter()-t0)*1000)
    eff_rows.append({
        "architecture": arch, "parameters": params,
        "size_mb": round(size_mb, 2),
        "inference_mean_ms": round(float(np.mean(times)), 3),
        "inference_std_ms":  round(float(np.std(times)), 3),
        "meets_target": size_mb <= 10.0,
    })
    print(f"  {arch}: {params:,} params  {size_mb:.2f} MB  "
          f"{np.mean(times):.3f}±{np.std(times):.3f} ms  "
          f"{'✓ meets target' if size_mb <= 10.0 else '✗ exceeds target'}")
    del m; torch.cuda.empty_cache()

pd.DataFrame(eff_rows).to_csv(OUT_DIR / "efficiency.csv", index=False)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK H: SWEEP CHARTS WITH FLOOR LINES
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK H: SWEEP CHARTS")
print("=" * 70)

synth_amts = [250, 500, 750, 1000, 1250]

def get_vals(arch, metric):
    return [next(r[metric] for r in sweep_results
                 if r["arch"] == arch and r["synth"] == s)
            for s in synth_amts]

cnn_acc = get_vals("BaselineCNN",  "acc_mean")
r18_acc = get_vals("ResNet18_1ch", "acc_mean")
cnn_bal = get_vals("BaselineCNN",  "bal_mean")
r18_bal = get_vals("ResNet18_1ch", "bal_mean")

cnn_acc_std = get_vals("BaselineCNN",  "acc_std")
r18_acc_std = get_vals("ResNet18_1ch", "acc_std")
cnn_bal_std = get_vals("BaselineCNN",  "bal_std")
r18_bal_std = get_vals("ResNet18_1ch", "bal_std")

fl_cnn_acc = floor_results["BaselineCNN"]["acc_mean"]
fl_r18_acc = floor_results["ResNet18_1ch"]["acc_mean"]
fl_cnn_bal = floor_results["BaselineCNN"]["bal_mean"]
fl_r18_bal = floor_results["ResNet18_1ch"]["bal_mean"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (cy, ry, cstd, rstd, fl_c, fl_r, ylabel, title) in zip(axes, [
    (cnn_acc, r18_acc, cnn_acc_std, r18_acc_std, fl_cnn_acc, fl_r18_acc,
     "Accuracy (= Micro F1)", "Synth Sweep — Accuracy"),
    (cnn_bal, r18_bal, cnn_bal_std, r18_bal_std, fl_cnn_bal, fl_r18_bal,
     "Balanced Accuracy",     "Synth Sweep — Balanced Accuracy"),
]):
    ax.errorbar(synth_amts, cy, yerr=cstd, fmt="o-", color="#00796B",
                lw=2, capsize=4, label="BaselineCNN")
    ax.errorbar(synth_amts, ry, yerr=rstd, fmt="s-", color="#1E2761",
                lw=2, capsize=4, label="ResNet18_1ch")
    ax.axhline(fl_c, color="#00796B", ls="--", lw=1.5, alpha=0.7,
               label=f"CNN real-only floor ({fl_c:.3f})")
    ax.axhline(fl_r, color="#1E2761", ls=":",  lw=1.5, alpha=0.7,
               label=f"R18 real-only floor ({fl_r:.3f})")
    ax.set_xlabel("Synthetic images per structured class")
    ax.set_ylabel(ylabel); ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_xticks(synth_amts)

plt.suptitle("Synthetic Augmentation Sweep — All 10 Configs With Real-Only Floor",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "sweep_charts_with_floor.png", dpi=180); plt.close()
print("→ Saved sweep_charts_with_floor.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK I: SYNTHETIC EXAMPLES PANEL
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK I: SYNTHETIC EXAMPLES")
print("=" * 70)

rng_ex = np.random.default_rng(2025)
n_var  = 3
fig, axes = plt.subplots(NUM_CLASSES, n_var, figsize=(n_var*2.2, NUM_CLASSES*1.4))
for row, cls in enumerate(classes):
    for col in range(n_var):
        arr = GEN_FUNCS[cls](rng_ex)
        axes[row, col].imshow(arr, cmap="gray", interpolation="nearest", aspect="auto")
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=9, rotation=0,
                                       ha="right", va="center", labelpad=50)
plt.suptitle("Synthetic Examples — 3 Random Variants per Class",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "synthetic_examples.png", dpi=160); plt.close()
print("→ Saved synthetic_examples.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK J: SCATTERPLOT (params vs accuracy — live efficiency data)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK J: SCATTERPLOT")
print("=" * 70)

eff_df = pd.read_csv(OUT_DIR / "efficiency.csv")
best_per_arch = {}
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    best_per_arch[arch] = next(r for r in sweep_sorted if r["arch"] == arch)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_s = ["#00796B", "#1E2761"]

for ax, (y_key, ylabel, title) in zip(axes, [
    ("acc_mean", "Accuracy (Micro F1)", "Model Complexity vs Accuracy"),
    ("bal_mean", "Balanced Accuracy",   "Model Complexity vs Balanced Accuracy"),
]):
    for i, (arch, col) in enumerate(zip(["BaselineCNN","ResNet18_1ch"], colors_s)):
        r   = best_per_arch[arch]
        row = eff_df[eff_df["architecture"] == arch].iloc[0]
        ax.scatter(row.parameters/1e6, r[y_key], s=300, c=col, zorder=5, label=arch)
        ax.annotate(
            f"{arch}\n{row.parameters/1e6:.1f}M  {row.size_mb:.1f}MB\n"
            f"{row.inference_mean_ms:.1f}ms",
            (row.parameters/1e6, r[y_key]),
            xytext=(15, -40), textcoords="offset points", fontsize=9, color=col,
            arrowprops=dict(arrowstyle="->", color=col, lw=1.5))
    ax.set_xlabel("Parameters (millions)"); ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold"); ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3); ax.set_ylim(0.88, 1.01); ax.set_xlim(0, 15)

plt.suptitle("Efficiency vs Performance — Best Config per Architecture\n"
             "(BaselineCNN: 4.7× smaller, 5× faster, statistically equivalent)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "scatterplot_params_vs_accuracy.png", dpi=180); plt.close()
print("→ Saved scatterplot_params_vs_accuracy.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK K: THESIS SUMMARY TABLES
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK K: THESIS SUMMARY — COPY INTO OVERLEAF")
print("=" * 70)

print("\n── Real-Only Floor (no synth) ──────────────────────────────────────")
for arch, fr in floor_results.items():
    print(f"  {arch:15s}: acc={fr['acc_mean']:.4f}±{fr['acc_std']:.4f}  "
          f"bal={fr['bal_mean']:.4f}±{fr['bal_std']:.4f}")

print("\n── Full Sweep (all 10 configs, sorted by Balanced Accuracy) ─────────")
print(f"{'Rank':4s} {'Config':22s} {'Acc':>8} {'±':>6} {'BalAcc':>8} {'±':>6} {'Struct':>7}")
for i, r in enumerate(sweep_sorted, 1):
    print(f"{i:4d} {r['label']:22s} {r['acc_mean']:>8.4f} {r['acc_std']:>6.4f} "
          f"{r['bal_mean']:>8.4f} {r['bal_std']:>6.4f} {r['struct_recall_mean']:>7.4f}")

print(f"\n── Best overall: {best_overall['label']} ─────────────────────────────────")
print(f"  All downstream analysis (t-SNE, Grad-CAM, perturbation) uses this model.")

print("\n── Mixed Test Set (top configs) ────────────────────────────────────")
df_mix = pd.read_csv(OUT_DIR / "mixed_test_results.csv")
for _, row in df_mix.iterrows():
    print(f"  {row['config']:22s} {row['test_set']:12s} "
          f"acc={row['accuracy']:.4f}  bal={row['balanced_acc']:.4f}  "
          f"struct={row['struct_recall']:.4f}")

print("\n── Perturbation Robustness ─────────────────────────────────────────")
for r in pert_rows:
    flag = "  ← pipeline constraint" if r["perturbation"] == "gaussian_blur" else ""
    print(f"  {r['perturbation']:20s}: bal={r['bal_acc_mean']:.4f}  drop={r['drop']:+.4f}{flag}")

print("\n── Model Efficiency ─────────────────────────────────────────────────")
for r in eff_rows:
    print(f"  {r['architecture']:15s}: {r['parameters']:>11,} params  "
          f"{r['size_mb']:>5.2f} MB  {r['inference_mean_ms']:>7.3f} ms  "
          f"{'✓' if r['meets_target'] else '✗'}")

print("\n" + "=" * 70)
print("ALL OUTPUTS SAVED TO:", OUT_DIR)
print("=" * 70)
for f in sorted(OUT_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(OUT_DIR)}")


2026-06-17 08:55:12.347767: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781686512.531530      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781686512.582090      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781686513.010542      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781686513.010582      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781686513.010585      57 computation_placer.cc:177] computation placer alr

Device: cuda | GPUs: 2

8-class setup:
  0: 0-NOISE_EMPTY
  1: 2-ZEBRA
  2: 3-OKAPI
  3: 4-CHEETAH
  4: 5-GRID
  5: 6-DIAGONAL
  6: 7-BLINDS
  7: 8-MORSE

Real data: 1146 total
  0-NOISE_EMPTY       :  1058  (92.3%)
  2-ZEBRA             :    11  (1.0%)
  3-OKAPI             :    10  (0.9%)
  4-CHEETAH           :    14  (1.2%)
  5-GRID              :    23  (2.0%)
  6-DIAGONAL          :    10  (0.9%)
  7-BLINDS            :     9  (0.8%)
  8-MORSE             :    11  (1.0%)

GENERATING MIXED TEST SET
  100 new synthetic images per class (seed+999 — never seen in training)
  Real-only:   1146
  Synth-only:   800  (100/class × 8)
  Mixed:       1946

BLOCK 0: REAL-ONLY FLOOR
Train on real images ONLY. No synthetic data.
This is the baseline to compare all synth configs against.

  Architecture: BaselineCNN
    Fold 1: acc=0.9547  bal=0.5361
    Fold 2: acc=0.9756  bal=0.8731
    Fold 3: acc=0.9860  bal=0.8329
    Fold 4: acc=0.9615  bal=0.6757
  → BaselineCNN REAL-ONLY: acc=0.9695±0.0

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/thesis_pipeline",
    "zip",
    "/kaggle/working/thesis_pipeline"
)

print("ZIP created!")

In [1]:
# ============================================================
# COMPLETE THESIS EVALUATION PIPELINE — FULL SWEEP VERSION
# UHasselt DSI × Gpixel — Nuthi Raviteja Pediredla
#
# UPDATED:
#   - TRASH class: exactly 10 real images (downsampled from 1,054)
#   - Perturbations: NO blur/intensity (binary matrix constraints)
#   - All 10 configs + controls run fresh this session
#   - FIXED: Added labels=list(range(NUM_CLASSES)) to classification_report to 
#     prevent the ValueError when test sets are missing a class.
#   - FIXED: Moved TripletDS class up to prevent NameError.
#
# EXPECTED RUNTIME on Kaggle T4 × 2 GPU: ~5.5 hours
# ============================================================

import os, random, shutil, time, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageFilter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

# ── Optional packages ────────────────────────────────────────────────────────
try:
    import umap as umap_lib
    HAS_UMAP = True
except ImportError:
    os.system("pip install umap-learn -q")
    try:
        import umap as umap_lib
        HAS_UMAP = True
    except:
        HAS_UMAP = False
        print("WARNING: umap-learn not available — UMAP skipped.")

try:
    from scipy.stats import wilcoxon as scipy_wilcoxon
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

SEED        = 42
N_SPLITS    = 4
EPOCHS      = 15
LR          = 1e-3
BATCH_SIZE  = 128
NUM_WORKERS = 2

# Full sweep — ALL 10 configurations
SWEEP_CONFIGS = [
    {"arch": "BaselineCNN",  "synth": 250},
    {"arch": "BaselineCNN",  "synth": 500},
    {"arch": "BaselineCNN",  "synth": 750},
    {"arch": "BaselineCNN",  "synth": 1000},
    {"arch": "BaselineCNN",  "synth": 1250},
    {"arch": "ResNet18_1ch", "synth": 250},
    {"arch": "ResNet18_1ch", "synth": 500},
    {"arch": "ResNet18_1ch", "synth": 750},
    {"arch": "ResNet18_1ch", "synth": 1000},
    {"arch": "ResNet18_1ch", "synth": 1250},
]

# For mixed test set: top N configs by balanced accuracy
TOP_N_FOR_MIXED_TEST = 3

# For t-SNE/UMAP: synthetic images to embed per class (visual only)
EMBED_SYNTH_N = 80
# For mixed test set generation
MIXED_SYNTH_N = 100

# Grad-CAM and perturbation settings
GRADCAM_N_CORRECT   = 16
GRADCAM_N_INCORRECT = 9

# ═══════════════════════════════════════════════════════════════════════════════
# TRASH CLASS DOWNSAMPLING: exactly 10 real images (downsampled from 1,054)
# ═══════════════════════════════════════════════════════════════════════════════
TRASH_DOWNSAMPLE_N = 10

DATA_ROOT = Path("/kaggle/input/datasets/teja19129/g-pixel/96x192")
WORK_DIR  = Path("/kaggle/working")
OUT_DIR   = WORK_DIR / "thesis_pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
scaler  = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def get_raw(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def strip_dp(state_dict):
    return {
        (k[len("module."):] if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }


# ══════════════════════════════════════════════════════════════════════════════
# DATA SETUP — WITH TRASH DOWNSAMPLING
# ══════════════════════════════════════════════════════════════════════════════

valid_exts     = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TRASH_FOLDERS  = {"0-EMPTY", "1-NOISE"}
all_folders    = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
STRUCT_FOLDERS = [f for f in all_folders if f not in TRASH_FOLDERS]

# ── STEP 1: Collect ALL trash images ────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 1: TRASH CLASS DOWNSAMPLING")
print(f"Target: exactly {TRASH_DOWNSAMPLE_N} images from combined NOISE+EMPTY")
print("=" * 70)

all_trash_paths = []
for folder in TRASH_FOLDERS:
    cls_dir = DATA_ROOT / folder
    if cls_dir.exists():
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in valid_exts:
                all_trash_paths.append(str(p))

print(f"Total NOISE + EMPTY images: {len(all_trash_paths)}")
print(f"Downsampling to: {TRASH_DOWNSAMPLE_N}")

# Random downsample trash images
rng_trash = np.random.default_rng(SEED)
trash_indices = rng_trash.choice(len(all_trash_paths), size=TRASH_DOWNSAMPLE_N, replace=False)
selected_trash_paths = [all_trash_paths[i] for i in sorted(trash_indices)]

print(f"Selected {len(selected_trash_paths)} images for TRASH class")

# ── STEP 2: Build class structure with downsampled trash ──────────────────────
classes        = ["0-TRASH"] + STRUCT_FOLDERS
class_to_idx   = {cls: i for i, cls in enumerate(classes)}
idx_to_class   = {i: cls for cls, i in class_to_idx.items()}
NUM_CLASSES    = len(classes)
TRASH_IDX      = 0
STRUCTURED_IDX = list(range(1, NUM_CLASSES))

print(f"\n{NUM_CLASSES}-class setup (after downsampling):")
for cls, idx in class_to_idx.items():
    print(f"  {idx}: {cls}")

# ── STEP 3: Build all_samples with downsampled trash ──────────────────────────
all_samples = []
for p in selected_trash_paths:
    all_samples.append((p, 0))

for folder in STRUCT_FOLDERS:
    cls_dir = DATA_ROOT / folder
    if cls_dir.exists():
        idx = class_to_idx[folder]
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in valid_exts:
                all_samples.append((str(p), idx))

paths  = np.array([p for p, _ in all_samples])
labels = np.array([y for _, y in all_samples])

print(f"\nFinal dataset: {len(all_samples)} total images")
print("Distribution:")
for k, v in sorted(Counter(labels).items()):
    pct = 100*v/len(all_samples)
    print(f"  {idx_to_class[k]:20s}: {v:5d}  ({pct:5.1f}%)")

# ══════════════════════════════════════════════════════════════════════════════
# SYNTHETIC GENERATORS
# ══════════════════════════════════════════════════════════════════════════════

H, W = 96, 192

def save_img(arr, path):
    Image.fromarray((arr.astype(np.uint8) * 255)).save(path)

def bg(rng, prob=None):
    return (rng.random((H, W)) < (prob or rng.uniform(0.0001, 0.003))).astype(np.uint8)

def rand(arr, rng):
    img = Image.fromarray((arr.astype(np.uint8) * 255))
    if rng.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.2, 0.7))))
    arr = np.array(img).astype(np.float32) / 255.0
    if rng.random() < 0.5:
        arr = np.clip(arr + rng.normal(0, rng.uniform(0.01, 0.04), arr.shape), 0, 1)
    return (arr > rng.uniform(0.35, 0.65)).astype(np.uint8)

def gen_trash(rng):
    arr = ((rng.random((H, W)) < rng.uniform(0.001, 0.05)).astype(np.uint8)
           if rng.random() < 0.5 else np.zeros((H, W), dtype=np.uint8))
    return rand(arr, rng)

def gen_zebra(rng):
    arr = bg(rng); bw = rng.integers(120, 170); oy = rng.integers(0, 16)
    x0 = rng.integers(0, max(1, W - bw)); d = rng.uniform(0.45, 1.0)
    for y in np.arange(oy, H, 16): arr[y, x0:x0+bw] = rng.random(bw) < d
    return rand(arr, rng)

def gen_okapi(rng):
    arr = bg(rng); bw = rng.integers(55, 95); oy = rng.integers(0, 16)
    x0 = rng.integers(0, max(1, W - bw)); d = rng.uniform(0.40, 1.0)
    for y in np.arange(oy, H, 16): arr[y, x0:x0+bw] = rng.random(bw) < d
    return rand(arr, rng)

def gen_cheetah(rng):
    arr = bg(rng, rng.uniform(0.00005, 0.001))
    ox, oy = rng.integers(0, 2), rng.integers(0, 2)
    pat = (rng.random((H//2+1, W//2+1)) < rng.uniform(0.90, 0.995)).astype(np.uint8)
    arr[oy::2, ox::2] = pat[:arr[oy::2, ox::2].shape[0], :arr[oy::2, ox::2].shape[1]]
    if rng.random() < 0.7:
        bh = rng.integers(6, 10)
        for y in range(rng.integers(0, 16), H, 16): arr[y:y+bh, :] = 0
    return rand(arr, rng)

def gen_grid(rng):
    arr = bg(rng); pw = 12 * rng.integers(12, 14)
    x0 = rng.integers(0, max(1, W - pw)); y0 = rng.integers(0, 16)
    d = rng.uniform(0.85, 0.99)
    for y in np.arange(y0, H, 16):
        cols = np.arange(x0, x0+pw, 12); arr[y, cols] = rng.random(len(cols)) < d
    return rand(arr, rng)

def gen_diagonal(rng):
    arr = bg(rng, rng.uniform(0.00005, 0.0005))
    ox, oy = rng.integers(0, 12), rng.integers(0, 16)
    bp = np.array([1, 1, 0, 0] if rng.random() < 0.5 else [1, 0, 0, 0])
    pat = np.roll(np.tile(bp, rng.integers(3, 6)), rng.integers(0, len(bp)*3))
    cols = np.arange(ox, W, 12); n = min(len(cols), len(pat))
    cols, pat = cols[:n], pat[:n]
    for i, y in enumerate(np.arange(oy, H, 16)):
        sh = np.roll(pat, i); arr[y, cols] = rng.random(len(sh)) < (0.92 * sh)
    if rng.random() < 0.5: arr = np.fliplr(arr)
    return rand(arr, rng)

def gen_blinds(rng):
    arr = bg(rng); bb = rng.integers(2, 6)
    x1 = rng.choice([0, rng.integers(0, max(1, W//3))])
    x2 = rng.choice([W, W - rng.integers(0, max(1, W//3))])
    if x2 <= x1: x1, x2 = 0, W
    w = x2 - x1; oy = rng.integers(0, max(1, 16-bb))
    bp = np.linspace(rng.uniform(0.15, 0.90), rng.uniform(0.15, 0.90), bb)
    pb = np.array([rng.random(w) < bp[r] for r in range(bb)], dtype=np.uint8)
    con = rng.uniform(0.55, 0.95)
    for by in range(oy, H, 16):
        bh = min(bb, H - by)
        for r in range(bh): arr[by+r, x1:x2] = rng.random(w) < (con * pb[r])
    return rand(arr, rng)

def gen_morse(rng):
    arr = bg(rng); seg = rng.integers(4, 8); rep = rng.integers(6, 10)
    pw = min(seg*2*rep, W-1); x0 = rng.integers(0, max(1, W-pw))
    y0 = rng.integers(0, 16)
    for y in np.arange(y0, H, 16):
        bits = (rng.random(pw) < rng.uniform(0.85, 0.99)).astype(np.uint8)
        for s in range(0, pw, seg*2): bits[s:min(s+seg, pw)] = 1 - bits[s:min(s+seg, pw)]
        arr[y, x0:x0+pw] = bits
    return rand(arr, rng)

GEN_FUNCS = {
    "0-TRASH": gen_trash,        "2-ZEBRA": gen_zebra,
    "3-OKAPI": gen_okapi,        "4-CHEETAH": gen_cheetah,
    "5-GRID": gen_grid,          "6-DIAGONAL": gen_diagonal,
    "7-BLINDS": gen_blinds,      "8-MORSE": gen_morse,
}

def generate_synth(n_per_class, seed, struct_only=True, tag="train"):
    rng_l = np.random.default_rng(seed)
    root  = WORK_DIR / f"synth_{tag}_s{seed}_n{n_per_class}"
    if root.exists(): shutil.rmtree(root)
    root.mkdir(parents=True, exist_ok=True)
    cls_list = STRUCT_FOLDERS if struct_only else classes
    samples  = []
    for cls in cls_list:
        d = root / cls; d.mkdir()
        for i in range(n_per_class):
            save_img(GEN_FUNCS[cls](rng_l), d / f"{cls}_{i:05d}.png")
        idx = class_to_idx[cls]
        for p in sorted(d.iterdir()): samples.append((str(p), idx))
    return samples

# ══════════════════════════════════════════════════════════════════════════════
# DATASET + MODELS
# ══════════════════════════════════════════════════════════════════════════════

t1ch = transforms.Compose([
    transforms.Resize((96, 192)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

class DefectDS(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p, lbl = self.samples[idx]
        return t1ch(Image.open(p).convert("L")), lbl

# Added here globally to prevent NameError
class TripletDS(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y, real = self.items[i]
        return t1ch(Image.open(p).convert("L")), y, int(real)

def make_loader(samples, shuffle=False, drop_last=False):
    return DataLoader(
        DefectDS(samples), batch_size=BATCH_SIZE,
        shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=(NUM_WORKERS > 0),
        drop_last=drop_last,
    )

def create_model(arch):
    if arch == "BaselineCNN":
        class BaselineCNN(nn.Module):
            def __init__(self, n=NUM_CLASSES):
                super().__init__()
                self.features = nn.Sequential(
                    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                )
                self.classifier = nn.Sequential(
                    nn.Flatten(),
                    nn.Linear(64*12*24, 128), nn.ReLU(), nn.Dropout(0.3),
                    nn.Linear(128, n),
                )
            def forward(self, x): return self.classifier(self.features(x))
        m = BaselineCNN()
    elif arch == "ResNet18_1ch":
        m = models.resnet18(weights=None)
        orig = m.conv1
        m.conv1 = nn.Conv2d(1, orig.out_channels, orig.kernel_size,
                            orig.stride, orig.padding, bias=orig.bias)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        m = nn.DataParallel(m)
    return m.to(device)

# ══════════════════════════════════════════════════════════════════════════════
# METRICS + TRAINING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def full_metrics(y_true, y_pred, label=""):
    acc  = float(accuracy_score(y_true, y_pred))
    mf1  = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    bal  = float(balanced_accuracy_score(y_true, y_pred))
    _, rec, _, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    struct_recall = float(np.mean([rec[i] for i in STRUCTURED_IDX]))
    if label:
        print(f"\n  [{label}]")
        print(f"    Accuracy:           {acc:.4f}")
        print(f"    Micro F1:           {mf1:.4f}")
        print(f"    Balanced Accuracy:  {bal:.4f}")
        print(f"    Struct Mean Recall: {struct_recall:.4f}  ← KEY")
        for i, cls in enumerate(classes):
            tag = " [TRASH]" if i == TRASH_IDX else ""
            print(f"    {cls+tag:22s}: recall={rec[i]:.3f}  (n={int(sup[i])})")
    return {"accuracy": acc, "micro_f1": mf1, "balanced_accuracy": bal,
            "structured_recall": struct_recall,
            "per_class_recall": rec.tolist(), "support": sup.tolist()}


def run_epoch(model, loader, crit, opt=None):
    model.train() if opt else model.eval()
    ls, n, yt, yp = 0.0, 0, [], []
    for imgs, lbl in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbl  = lbl.to(device, non_blocking=True)
        if opt: opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(opt is not None):
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(imgs); loss = crit(out, lbl)
            if opt:
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
        ls += loss.item() * imgs.size(0); n += imgs.size(0)
        yp.extend(out.argmax(1).detach().cpu().tolist())
        yt.extend(lbl.cpu().tolist())
    return ls/n, accuracy_score(yt, yp), f1_score(yt, yp, average="micro", zero_division=0)


def evaluate_model(model, loader):
    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for imgs, lbl in loader:
            imgs = imgs.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(imgs)
            yp.extend(out.argmax(1).cpu().tolist())
            yt.extend(lbl.tolist())
    return np.array(yt), np.array(yp)


def save_cm(yt, yp, title, path):
    cm      = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    cm_norm = np.nan_to_num(cm.astype(float) / cm.sum(axis=1, keepdims=True))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, t in [(axes[0], cm, f"{title}\nRaw CM (n={len(yt)})"),
                        (axes[1], cm_norm, "Normalised")]:
        im = ax.imshow(data, cmap="Blues")
        plt.colorbar(im, ax=ax)
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(NUM_CLASSES))
        ax.set_yticklabels(classes, fontsize=8)
        ax.set_title(t, fontsize=11)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                v = data[i, j]
                txt = f"{v:.2f}" if isinstance(v, float) else str(int(v))
                ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                        color="white" if v > data.max() * 0.6 else "black")
    plt.tight_layout()
    plt.savefig(path, dpi=180); plt.close()


def run_one_config(arch, synth_n, seed=SEED):
    set_seed(seed)
    train_synth = generate_synth(synth_n, seed, struct_only=True, tag=f"tr{synth_n}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    best_overall_mf1 = -1
    best_model_state = None
    fold_metrics = []
    fold_accs, fold_bals = [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(paths, labels), 1):
        real_tr = [(paths[i], int(labels[i])) for i in tr_idx]
        real_va = [(paths[i], int(labels[i])) for i in va_idx]
        tr_loader = make_loader(real_tr + train_synth, shuffle=True, drop_last=True)
        va_loader = make_loader(real_va)

        model = create_model(arch)
        crit  = nn.CrossEntropyLoss()
        opt   = optim.Adam(model.parameters(), lr=LR)
        best_fold_mf1, best_fold_state = -1, None

        for epoch in range(EPOCHS):
            run_epoch(model, tr_loader, crit, opt)
            _, _, va_mf1 = run_epoch(model, va_loader, crit)
            print(f"    Ep {epoch+1:02d} | va_mf1={va_mf1:.4f}")
            if va_mf1 > best_fold_mf1:
                best_fold_mf1  = va_mf1
                _raw = get_raw(model)
                best_fold_state = {k: v.detach().cpu().clone()
                                   for k, v in _raw.state_dict().items()}

        get_raw(model).load_state_dict(best_fold_state)
        yt, yp = evaluate_model(model, va_loader)
        m = full_metrics(yt, yp, f"Fold {fold} val")
        fold_metrics.append(m)
        fold_accs.append(m["accuracy"])
        fold_bals.append(m["balanced_accuracy"])

        if best_fold_mf1 > best_overall_mf1:
            best_overall_mf1 = best_fold_mf1
            best_model_state = best_fold_state

        del model, opt, crit; torch.cuda.empty_cache()

    return best_model_state, fold_metrics, fold_accs, fold_bals

# ══════════════════════════════════════════════════════════════════════════════
# GENERATE MIXED TEST SET 
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("GENERATING MIXED TEST SET")
print(f"  {MIXED_SYNTH_N} new synthetic images per class (seed+999 — never seen in training)")
print("  Note: synthetic TRASH is NOT generated (only structured classes)")
print("=" * 70)

test_synth     = generate_synth(MIXED_SYNTH_N, SEED+999, struct_only=True, tag="test")
real_test_set  = list(zip(paths.tolist(), labels.tolist()))
synth_test_set = test_synth
mixed_test_set = real_test_set + synth_test_set

print(f"  Real-only:  {len(real_test_set):5d}  (all classes including {TRASH_DOWNSAMPLE_N} TRASH)")
print(f"  Synth-only: {len(synth_test_set):5d}  ({MIXED_SYNTH_N}/class × {len(STRUCT_FOLDERS)} structured)")
print(f"  Mixed:      {len(mixed_test_set):5d}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK 0: REAL-ONLY FLOOR 
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK 0: REAL-ONLY FLOOR")
print("=" * 70)

floor_results = {}
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    print(f"\n  Architecture: {arch}")
    set_seed(SEED)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_accs, fold_bals = [], []
    best_mf1, best_state = -1, None

    for fold, (tr_idx, va_idx) in enumerate(skf.split(paths, labels), 1):
        real_tr = [(paths[i], int(labels[i])) for i in tr_idx]
        real_va = [(paths[i], int(labels[i])) for i in va_idx]
        tr_loader = make_loader(real_tr, shuffle=True, drop_last=False)
        va_loader = make_loader(real_va)
        model = create_model(arch); crit = nn.CrossEntropyLoss()
        opt = optim.Adam(model.parameters(), lr=LR)
        bfm, bfs = -1, None
        for epoch in range(EPOCHS):
            run_epoch(model, tr_loader, crit, opt)
            _, _, va_mf1 = run_epoch(model, va_loader, crit)
            if va_mf1 > bfm:
                bfm = va_mf1
                _raw = get_raw(model)
                bfs = {k: v.detach().cpu().clone() for k, v in _raw.state_dict().items()}
        get_raw(model).load_state_dict(bfs)
        yt, yp = evaluate_model(model, va_loader)
        m = full_metrics(yt, yp)
        fold_accs.append(m["accuracy"]); fold_bals.append(m["balanced_accuracy"])
        if bfm > best_mf1: best_mf1 = bfm; best_state = bfs
        del model, opt, crit; torch.cuda.empty_cache()
        print(f"    Fold {fold}: acc={m['accuracy']:.4f}  bal={m['balanced_accuracy']:.4f}")

    floor_results[arch] = {
        "fold_accs": fold_accs, "fold_bals": fold_bals,
        "acc_mean": float(np.mean(fold_accs)), "acc_std": float(np.std(fold_accs)),
        "bal_mean": float(np.mean(fold_bals)), "bal_std": float(np.std(fold_bals)),
        "best_state": best_state,
    }
    print(f"  → {arch} REAL-ONLY: acc={np.mean(fold_accs):.4f}±{np.std(fold_accs):.4f} "
          f"bal={np.mean(fold_bals):.4f}±{np.std(fold_bals):.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# FULL SWEEP: ALL 10 CONFIGURATIONS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("FULL SWEEP: ALL 10 CONFIGURATIONS")
print("=" * 70)

sweep_results = []
saved_states  = {}

for cfg in SWEEP_CONFIGS:
    arch  = cfg["arch"]
    synth = cfg["synth"]
    label = f"{arch}_s{synth}"

    print(f"\n{'#'*70}")
    print(f"CONFIG: {label}  (synth={synth}/structured class)")
    print(f"{'#'*70}")

    best_state, fold_metrics, fold_accs, fold_bals = run_one_config(arch, synth)
    saved_states[label] = {"arch": arch, "state": best_state}

    acc_mean = float(np.mean(fold_accs))
    acc_std  = float(np.std(fold_accs))
    bal_mean = float(np.mean(fold_bals))
    bal_std  = float(np.std(fold_bals))
    sr_mean  = float(np.mean([m["structured_recall"] for m in fold_metrics]))

    sweep_results.append({
        "label": label, "arch": arch, "synth": synth,
        "acc_mean": acc_mean, "acc_std": acc_std,
        "bal_mean": bal_mean, "bal_std": bal_std,
        "struct_recall_mean": sr_mean,
        "fold_accs": fold_accs, "fold_bals": fold_bals,
    })
    print(f"\n  → {label}: acc={acc_mean:.4f}±{acc_std:.4f}  "
          f"bal={bal_mean:.4f}±{bal_std:.4f}  struct={sr_mean:.4f}")

sweep_df = pd.DataFrame([{k: v for k, v in r.items()
                           if k not in ("fold_accs", "fold_bals")}
                          for r in sweep_results])
sweep_df.to_csv(OUT_DIR / "sweep_results.csv", index=False)

sweep_sorted = sorted(sweep_results, key=lambda r: r["bal_mean"], reverse=True)
best_overall   = sweep_sorted[0]
top_n_configs  = sweep_sorted[:TOP_N_FOR_MIXED_TEST]

print("\n" + "=" * 70)
print("SWEEP COMPLETE — RESULTS RANKED BY BALANCED ACCURACY")
print("=" * 70)
print(f"\n{'Rank':4s} {'Config':22s} {'Acc Mean':>10} {'Acc Std':>9} "
      f"{'Bal Acc':>9} {'Bal Std':>9} {'Struct':>8}")
print("-" * 77)
for i, r in enumerate(sweep_sorted, 1):
    flag = " ← BEST" if i == 1 else (f" ← top {i}" if i <= TOP_N_FOR_MIXED_TEST else "")
    print(f"{i:4d} {r['label']:22s} {r['acc_mean']:>10.4f} {r['acc_std']:>9.4f} "
          f"{r['bal_mean']:>9.4f} {r['bal_std']:>9.4f} {r['struct_recall_mean']:>8.4f}{flag}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK A: MIXED TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK A: MIXED TEST SET — TOP {TOP_N_FOR_MIXED_TEST} CONFIGS")
print("=" * 70)

mixed_test_rows = []
for r in top_n_configs:
    label = r["label"]
    arch  = r["arch"]
    cfg_dir = OUT_DIR / label
    cfg_dir.mkdir(exist_ok=True)

    model = create_model(arch)
    get_raw(model).load_state_dict(saved_states[label]["state"])
    model.eval()

    for test_name, test_data in [
        ("real_only",  real_test_set),
        ("synth_only", synth_test_set),
        ("mixed",      mixed_test_set),
    ]:
        loader = make_loader(test_data)
        yt, yp = evaluate_model(model, loader)
        m = full_metrics(yt, yp, f"{label} | {test_name} (n={len(test_data)})")
        
        # FIXED: Enforce labels size explicitly to prevent 7 vs 8 mismatch errors
        report = classification_report(yt, yp, labels=list(range(NUM_CLASSES)), target_names=classes, zero_division=0)
        (cfg_dir / f"test_{test_name}_report.txt").write_text(report)
        save_cm(yt, yp, f"{label} | {test_name}", cfg_dir / f"cm_{test_name}.png")
        
        mixed_test_rows.append({
            "config": label, "test_set": test_name, "n": len(test_data),
            "accuracy": m["accuracy"], "balanced_acc": m["balanced_accuracy"],
            "struct_recall": m["structured_recall"],
            **{f"recall_{classes[i]}": m["per_class_recall"][i]
               for i in range(NUM_CLASSES)},
        })
    del model; torch.cuda.empty_cache()
pd.DataFrame(mixed_test_rows).to_csv(OUT_DIR / "mixed_test_results.csv", index=False)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK B + C: t-SNE AND UMAP
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK B+C: t-SNE / UMAP — Best model: {best_overall['label']}")
print("=" * 70)

COLORS = ["#455A64","#00796B","#E53935","#6A1B9A",
          "#F57C00","#1E2761","#2E7D32","#795548"]

best_arch = best_overall["arch"]
emb_model = create_model(best_arch)
emb_raw = get_raw(emb_model)
emb_raw.load_state_dict(saved_states[best_overall["label"]]["state"])
emb_raw.eval()

feats_collect = []
if best_arch == "BaselineCNN":
    hook_layer = emb_raw.classifier[2]
else:
    hook_layer = emb_raw.avgpool

hook = hook_layer.register_forward_hook(
    lambda m, inp, out: feats_collect.append(out.detach().cpu()))

Y_all, R_all = [], []
loader_emb = DataLoader(TripletDS(all_triplets), batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
with torch.no_grad():
    for imgs, yl, rl in loader_emb:
        imgs = imgs.to(device, non_blocking=True)
        emb_raw(imgs)
        Y_all.extend(yl.tolist()); R_all.extend(rl.tolist())

hook.remove()
del emb_model, emb_raw; torch.cuda.empty_cache()

F_all = torch.cat(feats_collect, 0).numpy()
if F_all.ndim > 2: F_all = F_all.reshape(F_all.shape[0], -1)
Y_all = np.array(Y_all); R_all = np.array(R_all)

def plot_embeddings(Z, Y, R, title, path):
    from matplotlib.lines import Line2D
    fig, ax = plt.subplots(figsize=(11, 9))
    for ci, cls in enumerate(classes):
        col = COLORS[ci % len(COLORS)]
        rm = (Y == ci) & (R == 1); sm = (Y == ci) & (R == 0)
        if rm.any(): ax.scatter(Z[rm,0], Z[rm,1], c=col, s=55, alpha=0.85, marker="o")
        if sm.any(): ax.scatter(Z[sm,0], Z[sm,1], c=col, s=28, alpha=0.50, marker="x")
    leg_cls  = [Line2D([0],[0],marker="o",color="w",markerfacecolor=COLORS[i],
                       markersize=9, label=classes[i]) for i in range(NUM_CLASSES)]
    leg_type = [Line2D([0],[0],marker="o",color="k",markersize=7,label="real (circles)"),
                Line2D([0],[0],marker="x",color="k",markersize=7,label="synth (crosses)")]
    ax.legend(handles=leg_cls+leg_type, loc="best", fontsize=8, ncol=2)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Component 1"); ax.set_ylabel("Component 2")
    ax.grid(True, alpha=0.2); plt.tight_layout()
    plt.savefig(path, dpi=180); plt.close()

print("  Running t-SNE...")
Z_tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=SEED, n_jobs=-1).fit_transform(F_all)
plot_embeddings(Z_tsne, Y_all, R_all, f"t-SNE — {best_overall['label']}", OUT_DIR / "tsne_embeddings.png")

if HAS_UMAP:
    print("  Running UMAP...")
    Z_umap = umap_lib.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED, n_jobs=-1).fit_transform(F_all)
    plot_embeddings(Z_umap, Y_all, R_all, f"UMAP — {best_overall['label']}", OUT_DIR / "umap_embeddings.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK D: GRAD-CAM
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK D: GRAD-CAM — Best model: {best_overall['label']}")
print("=" * 70)

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.grads = None; self.acts = None; self._h = []
        self._h.append(target_layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o)))
        self._h.append(target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0])))
    def compute(self, img_t, cls_idx):
        self.model.zero_grad()
        out = self.model(img_t)
        out[0, cls_idx].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(H, W), mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), out.argmax(1).item()
    def remove(self):
        for h in self._h: h.remove()

gc_arch  = best_overall["arch"]; gc_label = best_overall["label"]
if gc_arch != "BaselineCNN":
    best_cnn = next((r for r in sweep_sorted if r["arch"] == "BaselineCNN"), None)
    if best_cnn:
        gc_arch  = "BaselineCNN"
        gc_label = best_cnn["label"]

gc_model = create_model("BaselineCNN")
gc_raw = get_raw(gc_model)
gc_raw.load_state_dict(saved_states[gc_label]["state"])
gc_raw.eval()
gc = GradCAM(gc_raw, gc_raw.features[6])

correct_items, incorrect_items = [], []
all_pil = [Image.open(p).convert("L") for p, _ in real_test_set]

for pil_img, true_lbl in zip(all_pil, labels):
    tensor = t1ch(pil_img).unsqueeze(0).to(device)
    cam_map, pred = gc.compute(tensor, int(true_lbl))
    raw_arr = np.array(pil_img.resize((W, H)))
    item = (raw_arr, int(true_lbl), pred, cam_map)
    if pred == int(true_lbl): correct_items.append(item)
    else:                     incorrect_items.append(item)

gc.remove()
del gc_model, gc_raw; torch.cuda.empty_cache()

def save_gradcam_panel(items, n, title, path):
    items = items[:n]
    cols  = 4; rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols*2, figsize=(cols*4, rows*2.6))
    axes = axes.flatten()
    for k, (raw_img, true_lbl, pred_lbl, cam_map) in enumerate(items):
        ax_img = axes[k*2]; ax_cam = axes[k*2+1]
        ax_img.imshow(raw_img, cmap="gray", aspect="auto")
        ax_img.set_title(f"True: {classes[true_lbl]}", fontsize=7); ax_img.axis("off")
        ax_cam.imshow(raw_img, cmap="gray", alpha=0.55, aspect="auto")
        ax_cam.imshow(cam_map, cmap="jet", alpha=0.55, aspect="auto", vmin=0, vmax=1)
        col = "green" if pred_lbl == true_lbl else "red"
        ax_cam.set_title(f"Pred: {classes[pred_lbl]}", fontsize=7, color=col)
        ax_cam.axis("off")
    for j in range(len(items)*2, len(axes)): axes[j].axis("off")
    plt.suptitle(title, fontsize=11, fontweight="bold")
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()

save_gradcam_panel(correct_items, GRADCAM_N_CORRECT, f"Grad-CAM — Correct ({gc_label})", OUT_DIR / "gradcam_correct.png")
save_gradcam_panel(incorrect_items, GRADCAM_N_INCORRECT, f"Grad-CAM — Incorrect ({gc_label})", OUT_DIR / "gradcam_incorrect.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK E: PERTURBATION ROBUSTNESS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK E: PERTURBATION ROBUSTNESS — Best model: {best_overall['label']}")
print("=" * 70)

def apply_perturbation(img_tensor, ptype):
    arr = img_tensor.clone().squeeze().numpy()
    if ptype == "translation":
        arr = np.roll(arr, shift=3, axis=1)
    elif ptype == "rotation":
        from scipy.ndimage import rotate as ndrotate
        arr = np.clip(ndrotate(arr, angle=3, reshape=False, mode="nearest"), 0, 1)
    elif ptype == "threshold_shift":
        arr = (arr > 0.55).astype(np.float32)
    elif ptype == "salt_pepper":
        noise_mask = np.random.random(arr.shape) < 0.02
        arr = arr.copy()
        arr[noise_mask] = 1 - arr[noise_mask]
    elif ptype == "morphological_erosion":
        from scipy.ndimage import binary_erosion
        arr_binary = (arr > 0.5).astype(np.uint8)
        arr = binary_erosion(arr_binary, iterations=1).astype(np.float32)
    elif ptype == "morphological_dilation":
        from scipy.ndimage import binary_dilation
        arr_binary = (arr > 0.5).astype(np.uint8)
        arr = binary_dilation(arr_binary, iterations=1).astype(np.float32)
    return torch.tensor(arr).unsqueeze(0).unsqueeze(0).float()

PERTURBATIONS = [
    "clean", "translation", "rotation", "threshold_shift",
    "salt_pepper", "morphological_erosion", "morphological_dilation",
]

p_arch  = best_overall["arch"]
p_label = best_overall["label"]
p_model = create_model(p_arch)
p_raw = get_raw(p_model)
p_raw.load_state_dict(saved_states[p_label]["state"])
p_raw.eval()

skf_p = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
pert_fold_bals = defaultdict(list)

for fold, (_, va_idx) in enumerate(skf_p.split(paths, labels), 1):
    real_va     = [(paths[i], int(labels[i])) for i in va_idx]
    imgs_loaded = [t1ch(Image.open(p).convert("L")) for p, _ in real_va]
    true_labels = [y for _, y in real_va]
    
    for ptype in PERTURBATIONS:
        yt, yp = [], []
        for img_t, ty in zip(imgs_loaded, true_labels):
            pt = apply_perturbation(img_t.unsqueeze(0), ptype).to(device)
            with torch.no_grad(): 
                pred = p_raw(pt).argmax(1).item()
            yt.append(ty)
            yp.append(pred)
        bal_acc = balanced_accuracy_score(yt, yp)
        pert_fold_bals[ptype].append(bal_acc)

del p_model, p_raw; torch.cuda.empty_cache()

clean_mean  = np.mean(pert_fold_bals["clean"])
pert_rows   = []

for ptype in PERTURBATIONS:
    m, s = np.mean(pert_fold_bals[ptype]), np.std(pert_fold_bals[ptype])
    pert_rows.append({
        "perturbation": ptype,
        "bal_acc_mean": round(m, 4),
        "bal_acc_std": round(s, 4),
        "drop": round(clean_mean - m, 4)
    })
    print(f"  {ptype:28s}: bal={m:.4f}±{s:.4f}  drop={clean_mean-m:+.4f}")

pd.DataFrame(pert_rows).to_csv(OUT_DIR / "perturbation_results.csv", index=False)

fig, ax = plt.subplots(figsize=(13, 5))
names = [r["perturbation"] for r in pert_rows]
means = [r["bal_acc_mean"] for r in pert_rows]
stds  = [r["bal_acc_std"]  for r in pert_rows]

bar_colors = []
for n in names:
    if n == "clean": bar_colors.append("#2E7D32")
    elif n in ["morphological_erosion", "morphological_dilation", "salt_pepper"]: bar_colors.append("#E53935")
    else: bar_colors.append("#1E2761")

ax.bar(range(len(names)), means, yerr=stds, capsize=4, color=bar_colors, edgecolor="white", linewidth=0.5, alpha=0.9)
ax.axhline(clean_mean, color="gray", ls="--", lw=1.5, label=f"Clean ({clean_mean:.4f})")
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=25, ha="right", fontsize=10)
ax.set_ylabel("Balanced Accuracy", fontsize=11)
ax.set_ylim(0.4, 1.02)
ax.set_title(f"Perturbation Robustness — {p_label}", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(OUT_DIR / "perturbation_robustness.png", dpi=180)
plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK F: WILCOXON STATISTICAL TEST
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK F: WILCOXON SIGNED-RANK TEST")
print("=" * 70)

best_cnn_r = next((r for r in sweep_sorted if r["arch"] == "BaselineCNN"), None)
best_r18_r = next((r for r in sweep_sorted if r["arch"] == "ResNet18_1ch"), None)

wilcoxon_txt = []
if best_cnn_r and best_r18_r:
    cnn_f = best_cnn_r["fold_accs"]
    r18_f = best_r18_r["fold_accs"]
    wilcoxon_txt.append(f"Best CNN: {best_cnn_r['label']}  folds={cnn_f}  mean={np.mean(cnn_f):.4f}")
    wilcoxon_txt.append(f"Best R18: {best_r18_r['label']}  folds={r18_f}  mean={np.mean(r18_f):.4f}")
    if HAS_SCIPY:
        try:
            stat, p = scipy_wilcoxon(cnn_f, r18_f)
            sig = p < 0.05
            wilcoxon_txt.append(f"Wilcoxon W={stat:.4f}  p={p:.4f}  significant={sig}")
        except Exception as e:
            wilcoxon_txt.append(f"Wilcoxon failed: {e}")

(OUT_DIR / "statistical_test.txt").write_text("\n".join(wilcoxon_txt))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK G: MODEL EFFICIENCY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK G: MODEL EFFICIENCY")
print("=" * 70)

dummy = torch.zeros(1, 1, H, W).to(device)
eff_rows = []
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    m = create_model(arch)
    raw = m.module if isinstance(m, nn.DataParallel) else m
    params = sum(p.numel() for p in raw.parameters())
    tmp = WORK_DIR / f"tmp_{arch}.pt"
    torch.save(raw.state_dict(), tmp)
    size_mb = os.path.getsize(tmp)/1e6
    tmp.unlink()
    
    m.eval()
    with torch.no_grad():
        for _ in range(10): m(dummy)
    
    times = []
    with torch.no_grad():
        for _ in range(100):
            t0 = time.perf_counter()
            m(dummy)
            if device.type == "cuda": torch.cuda.synchronize()
            times.append((time.perf_counter()-t0)*1000)
    
    eff_rows.append({
        "architecture": arch,
        "parameters": params,
        "size_mb": round(size_mb, 2),
        "inference_mean_ms": round(float(np.mean(times)), 3),
        "inference_std_ms":  round(float(np.std(times)), 3),
        "meets_target": size_mb <= 10.0,
    })
    print(f"  {arch}: {params:,} params  {size_mb:.2f} MB")
    del m; torch.cuda.empty_cache()

pd.DataFrame(eff_rows).to_csv(OUT_DIR / "efficiency.csv", index=False)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK H: SWEEP CHARTS
# ══════════════════════════════════════════════════════════════════════════════

synth_amts = [250, 500, 750, 1000, 1250]

def get_vals(arch, metric):
    return [next(r[metric] for r in sweep_results
                 if r["arch"] == arch and r["synth"] == s)
            for s in synth_amts]

cnn_acc = get_vals("BaselineCNN",  "acc_mean")
r18_acc = get_vals("ResNet18_1ch", "acc_mean")
cnn_bal = get_vals("BaselineCNN",  "bal_mean")
r18_bal = get_vals("ResNet18_1ch", "bal_mean")

cnn_acc_std = get_vals("BaselineCNN",  "acc_std")
r18_acc_std = get_vals("ResNet18_1ch", "acc_std")
cnn_bal_std = get_vals("BaselineCNN",  "bal_std")
r18_bal_std = get_vals("ResNet18_1ch", "bal_std")

fl_cnn_acc = floor_results["BaselineCNN"]["acc_mean"]
fl_r18_acc = floor_results["ResNet18_1ch"]["acc_mean"]
fl_cnn_bal = floor_results["BaselineCNN"]["bal_mean"]
fl_r18_bal = floor_results["ResNet18_1ch"]["bal_mean"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (cy, ry, cstd, rstd, fl_c, fl_r, ylabel, title) in zip(axes, [
    (cnn_acc, r18_acc, cnn_acc_std, r18_acc_std, fl_cnn_acc, fl_r18_acc,
     "Accuracy (= Micro F1)", "Synth Sweep — Accuracy"),
    (cnn_bal, r18_bal, cnn_bal_std, r18_bal_std, fl_cnn_bal, fl_r18_bal,
     "Balanced Accuracy",     "Synth Sweep — Balanced Accuracy"),
]):
    ax.errorbar(synth_amts, cy, yerr=cstd, fmt="o-", color="#00796B",
                lw=2, capsize=4, label="BaselineCNN")
    ax.errorbar(synth_amts, ry, yerr=rstd, fmt="s-", color="#1E2761",
                lw=2, capsize=4, label="ResNet18_1ch")
    ax.axhline(fl_c, color="#00796B", ls="--", lw=1.5, alpha=0.7,
               label=f"CNN real-only floor ({fl_c:.3f})")
    ax.axhline(fl_r, color="#1E2761", ls=":",  lw=1.5, alpha=0.7,
               label=f"R18 real-only floor ({fl_r:.3f})")
    ax.set_xlabel("Synthetic images per structured class", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(synth_amts)

plt.suptitle("Synthetic Augmentation Sweep", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "sweep_charts_with_floor.png", dpi=180)
plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK I: SYNTHETIC EXAMPLES PANEL
# ══════════════════════════════════════════════════════════════════════════════

rng_ex = np.random.default_rng(2025)
n_var  = 3
fig, axes = plt.subplots(NUM_CLASSES, n_var, figsize=(n_var*2.2, NUM_CLASSES*1.4))
for row, cls in enumerate(classes):
    for col in range(n_var):
        arr = GEN_FUNCS[cls](rng_ex)
        axes[row, col].imshow(arr, cmap="gray", interpolation="nearest", aspect="auto")
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=9, rotation=0,
                                       ha="right", va="center", labelpad=50)

plt.suptitle("Synthetic Examples — 3 Random Variants per Class", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "synthetic_examples.png", dpi=160)
plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK J: SCATTERPLOT
# ══════════════════════════════════════════════════════════════════════════════

eff_df = pd.read_csv(OUT_DIR / "efficiency.csv")
best_per_arch = {}
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    best_per_arch[arch] = next(r for r in sweep_sorted if r["arch"] == arch)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_s = ["#00796B", "#1E2761"]

for ax, (y_key, ylabel, title) in zip(axes, [
    ("acc_mean", "Accuracy (Micro F1)", "Model Complexity vs Accuracy"),
    ("bal_mean", "Balanced Accuracy",   "Model Complexity vs Balanced Accuracy"),
]):
    for i, (arch, col) in enumerate(zip(["BaselineCNN","ResNet18_1ch"], colors_s)):
        r   = best_per_arch[arch]
        row = eff_df[eff_df["architecture"] == arch].iloc[0]
        ax.scatter(row.parameters/1e6, r[y_key], s=300, c=col, zorder=5, label=arch)
        ax.annotate(
            f"{arch}\n{row.parameters/1e6:.1f}M  {row.size_mb:.1f}MB\n"
            f"{row.inference_mean_ms:.1f}ms",
            (row.parameters/1e6, r[y_key]),
            xytext=(15, -40), textcoords="offset points", fontsize=9, color=col,
            arrowprops=dict(arrowstyle="->", color=col, lw=1.5))
    ax.set_xlabel("Parameters (millions)", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.88, 1.01)
    ax.set_xlim(0, 15)

plt.suptitle("Efficiency vs Performance", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "scatterplot_params_vs_accuracy.png", dpi=180)
plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK K: THESIS SUMMARY TABLES
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK K: THESIS SUMMARY — COPY INTO OVERLEAF")
print("=" * 70)

summary_txt = []
summary_txt.append("=" * 80)
summary_txt.append("THESIS EVALUATION PIPELINE — UPDATED RESULTS")
summary_txt.append("=" * 80)
summary_txt.append("")
summary_txt.append(f"DATASET CONFIGURATION:")
summary_txt.append(f"  TRASH (0-EMPTY + 1-NOISE combined): {TRASH_DOWNSAMPLE_N} real images")
summary_txt.append(f"  Total classes: {NUM_CLASSES}")
summary_txt.append(f"  Total real samples: {len(real_test_set)}")
summary_txt.append("")
summary_txt.append(f"SWEEP RESULTS (ranked by balanced accuracy):")
for i, r in enumerate(sweep_sorted[:5], 1):
    summary_txt.append(
        f"  {i}. {r['label']:22s}  "
        f"acc={r['acc_mean']:.4f}  bal={r['bal_mean']:.4f}  "
        f"struct={r['struct_recall_mean']:.4f}")
summary_txt.append("")
summary_txt.append(f"BEST OVERALL: {best_overall['label']}")
summary_txt.append(f"  Balanced Accuracy: {best_overall['bal_mean']:.4f} ± {best_overall['bal_std']:.4f}")
summary_txt.append(f"  Accuracy: {best_overall['acc_mean']:.4f} ± {best_overall['acc_std']:.4f}")
summary_txt.append("")
summary_txt.append(f"PERTURBATION ROBUSTNESS (binary-relevant):")
for r in pert_rows[:3]:
    summary_txt.append(f"  {r['perturbation']:25s}: {r['bal_acc_mean']:.4f}  (drop={r['drop']:+.4f})")
summary_txt.append("")
summary_txt.append("=" * 80)

(OUT_DIR / "SUMMARY.txt").write_text("\n".join(summary_txt))

print("\n" + "=" * 70)
print("PIPELINE COMPLETE ✓")
print("=" * 70)

2026-06-07 10:44:10.062737: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780829050.317848      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780829050.386421      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780829050.915786      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780829050.915857      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780829050.915860      57 computation_placer.cc:177] computation placer alr

Device: cuda | GPUs: 2

STEP 1: TRASH CLASS DOWNSAMPLING
Target: exactly 10 images from combined NOISE+EMPTY
Total NOISE + EMPTY images: 1058
Downsampling to: 10
Selected 10 images for TRASH class

8-class setup (after downsampling):
  0: 0-TRASH
  1: 2-ZEBRA
  2: 3-OKAPI
  3: 4-CHEETAH
  4: 5-GRID
  5: 6-DIAGONAL
  6: 7-BLINDS
  7: 8-MORSE

Final dataset: 98 total images
Distribution:
  0-TRASH             :    10  ( 10.2%)
  2-ZEBRA             :    11  ( 11.2%)
  3-OKAPI             :    10  ( 10.2%)
  4-CHEETAH           :    14  ( 14.3%)
  5-GRID              :    23  ( 23.5%)
  6-DIAGONAL          :    10  ( 10.2%)
  7-BLINDS            :     9  (  9.2%)
  8-MORSE             :    11  ( 11.2%)

GENERATING MIXED TEST SET
  100 new synthetic images per class (seed+999 — never seen in training)
  Note: synthetic TRASH is NOT generated (only structured classes)
  Real-only:     98  (all classes including 10 TRASH)
  Synth-only:   700  (100/class × 7 structured)
  Mixed:        798

B

NameError: name 'all_triplets' is not defined

In [2]:
# ============================================================
# COMPLETE THESIS EVALUATION PIPELINE — FULL SWEEP VERSION
# UHasselt DSI × Gpixel — Nuthi Raviteja Pediredla
#
# UPDATED:
#   - TRASH class: exactly 10 real images (downsampled from 1,054)
#   - Perturbations: NO blur/intensity (binary matrix constraints)
#   - All 10 configs + controls run fresh this session
#   - FIXED: Added labels=list(range(NUM_CLASSES)) to classification_report
#   - FIXED: Moved TripletDS class up to prevent NameError.
#   - FIXED: Restored 'embed_synth' and 'all_triplets' variable definitions in Block B+C.
#
# EXPECTED RUNTIME on Kaggle T4 × 2 GPU: ~5.5 hours
# ============================================================

import os, random, shutil, time, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageFilter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

# ── Optional packages ────────────────────────────────────────────────────────
try:
    import umap as umap_lib
    HAS_UMAP = True
except ImportError:
    os.system("pip install umap-learn -q")
    try:
        import umap as umap_lib
        HAS_UMAP = True
    except:
        HAS_UMAP = False
        print("WARNING: umap-learn not available — UMAP skipped.")

try:
    from scipy.stats import wilcoxon as scipy_wilcoxon
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

SEED        = 42
N_SPLITS    = 4
EPOCHS      = 15
LR          = 1e-3
BATCH_SIZE  = 128
NUM_WORKERS = 2

# Full sweep — ALL 10 configurations
SWEEP_CONFIGS = [
    {"arch": "BaselineCNN",  "synth": 250},
    {"arch": "BaselineCNN",  "synth": 500},
    {"arch": "BaselineCNN",  "synth": 750},
    {"arch": "BaselineCNN",  "synth": 1000},
    {"arch": "BaselineCNN",  "synth": 1250},
    {"arch": "ResNet18_1ch", "synth": 250},
    {"arch": "ResNet18_1ch", "synth": 500},
    {"arch": "ResNet18_1ch", "synth": 750},
    {"arch": "ResNet18_1ch", "synth": 1000},
    {"arch": "ResNet18_1ch", "synth": 1250},
]

# For mixed test set: top N configs by balanced accuracy
TOP_N_FOR_MIXED_TEST = 3

# For t-SNE/UMAP: synthetic images to embed per class (visual only)
EMBED_SYNTH_N = 80
# For mixed test set generation
MIXED_SYNTH_N = 100

# Grad-CAM and perturbation settings
GRADCAM_N_CORRECT   = 16
GRADCAM_N_INCORRECT = 9

# ═══════════════════════════════════════════════════════════════════════════════
# TRASH CLASS DOWNSAMPLING: exactly 10 real images (downsampled from 1,054)
# ═══════════════════════════════════════════════════════════════════════════════
TRASH_DOWNSAMPLE_N = 10

DATA_ROOT = Path("/kaggle/input/datasets/teja19129/g-pixel/96x192")
WORK_DIR  = Path("/kaggle/working")
OUT_DIR   = WORK_DIR / "thesis_pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
scaler  = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def get_raw(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def strip_dp(state_dict):
    return {
        (k[len("module."):] if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }


# ══════════════════════════════════════════════════════════════════════════════
# DATA SETUP — WITH TRASH DOWNSAMPLING
# ══════════════════════════════════════════════════════════════════════════════

valid_exts     = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TRASH_FOLDERS  = {"0-EMPTY", "1-NOISE"}
all_folders    = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
STRUCT_FOLDERS = [f for f in all_folders if f not in TRASH_FOLDERS]

# ── STEP 1: Collect ALL trash images ────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 1: TRASH CLASS DOWNSAMPLING")
print(f"Target: exactly {TRASH_DOWNSAMPLE_N} images from combined NOISE+EMPTY")
print("=" * 70)

all_trash_paths = []
for folder in TRASH_FOLDERS:
    cls_dir = DATA_ROOT / folder
    if cls_dir.exists():
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in valid_exts:
                all_trash_paths.append(str(p))

print(f"Total NOISE + EMPTY images: {len(all_trash_paths)}")
print(f"Downsampling to: {TRASH_DOWNSAMPLE_N}")

# Random downsample trash images
rng_trash = np.random.default_rng(SEED)
trash_indices = rng_trash.choice(len(all_trash_paths), size=TRASH_DOWNSAMPLE_N, replace=False)
selected_trash_paths = [all_trash_paths[i] for i in sorted(trash_indices)]

print(f"Selected {len(selected_trash_paths)} images for TRASH class")

# ── STEP 2: Build class structure with downsampled trash ──────────────────────
classes        = ["0-TRASH"] + STRUCT_FOLDERS
class_to_idx   = {cls: i for i, cls in enumerate(classes)}
idx_to_class   = {i: cls for cls, i in class_to_idx.items()}
NUM_CLASSES    = len(classes)
TRASH_IDX      = 0
STRUCTURED_IDX = list(range(1, NUM_CLASSES))

print(f"\n{NUM_CLASSES}-class setup (after downsampling):")
for cls, idx in class_to_idx.items():
    print(f"  {idx}: {cls}")

# ── STEP 3: Build all_samples with downsampled trash ──────────────────────────
all_samples = []
for p in selected_trash_paths:
    all_samples.append((p, 0))

for folder in STRUCT_FOLDERS:
    cls_dir = DATA_ROOT / folder
    if cls_dir.exists():
        idx = class_to_idx[folder]
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in valid_exts:
                all_samples.append((str(p), idx))

paths  = np.array([p for p, _ in all_samples])
labels = np.array([y for _, y in all_samples])

print(f"\nFinal dataset: {len(all_samples)} total images")
print("Distribution:")
for k, v in sorted(Counter(labels).items()):
    pct = 100*v/len(all_samples)
    print(f"  {idx_to_class[k]:20s}: {v:5d}  ({pct:5.1f}%)")

# ══════════════════════════════════════════════════════════════════════════════
# SYNTHETIC GENERATORS
# ══════════════════════════════════════════════════════════════════════════════

H, W = 96, 192

def save_img(arr, path):
    Image.fromarray((arr.astype(np.uint8) * 255)).save(path)

def bg(rng, prob=None):
    return (rng.random((H, W)) < (prob or rng.uniform(0.0001, 0.003))).astype(np.uint8)

def rand(arr, rng):
    img = Image.fromarray((arr.astype(np.uint8) * 255))
    if rng.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.2, 0.7))))
    arr = np.array(img).astype(np.float32) / 255.0
    if rng.random() < 0.5:
        arr = np.clip(arr + rng.normal(0, rng.uniform(0.01, 0.04), arr.shape), 0, 1)
    return (arr > rng.uniform(0.35, 0.65)).astype(np.uint8)

def gen_trash(rng):
    arr = ((rng.random((H, W)) < rng.uniform(0.001, 0.05)).astype(np.uint8)
           if rng.random() < 0.5 else np.zeros((H, W), dtype=np.uint8))
    return rand(arr, rng)

def gen_zebra(rng):
    arr = bg(rng); bw = rng.integers(120, 170); oy = rng.integers(0, 16)
    x0 = rng.integers(0, max(1, W - bw)); d = rng.uniform(0.45, 1.0)
    for y in np.arange(oy, H, 16): arr[y, x0:x0+bw] = rng.random(bw) < d
    return rand(arr, rng)

def gen_okapi(rng):
    arr = bg(rng); bw = rng.integers(55, 95); oy = rng.integers(0, 16)
    x0 = rng.integers(0, max(1, W - bw)); d = rng.uniform(0.40, 1.0)
    for y in np.arange(oy, H, 16): arr[y, x0:x0+bw] = rng.random(bw) < d
    return rand(arr, rng)

def gen_cheetah(rng):
    arr = bg(rng, rng.uniform(0.00005, 0.001))
    ox, oy = rng.integers(0, 2), rng.integers(0, 2)
    pat = (rng.random((H//2+1, W//2+1)) < rng.uniform(0.90, 0.995)).astype(np.uint8)
    arr[oy::2, ox::2] = pat[:arr[oy::2, ox::2].shape[0], :arr[oy::2, ox::2].shape[1]]
    if rng.random() < 0.7:
        bh = rng.integers(6, 10)
        for y in range(rng.integers(0, 16), H, 16): arr[y:y+bh, :] = 0
    return rand(arr, rng)

def gen_grid(rng):
    arr = bg(rng); pw = 12 * rng.integers(12, 14)
    x0 = rng.integers(0, max(1, W - pw)); y0 = rng.integers(0, 16)
    d = rng.uniform(0.85, 0.99)
    for y in np.arange(y0, H, 16):
        cols = np.arange(x0, x0+pw, 12); arr[y, cols] = rng.random(len(cols)) < d
    return rand(arr, rng)

def gen_diagonal(rng):
    arr = bg(rng, rng.uniform(0.00005, 0.0005))
    ox, oy = rng.integers(0, 12), rng.integers(0, 16)
    bp = np.array([1, 1, 0, 0] if rng.random() < 0.5 else [1, 0, 0, 0])
    pat = np.roll(np.tile(bp, rng.integers(3, 6)), rng.integers(0, len(bp)*3))
    cols = np.arange(ox, W, 12); n = min(len(cols), len(pat))
    cols, pat = cols[:n], pat[:n]
    for i, y in enumerate(np.arange(oy, H, 16)):
        sh = np.roll(pat, i); arr[y, cols] = rng.random(len(sh)) < (0.92 * sh)
    if rng.random() < 0.5: arr = np.fliplr(arr)
    return rand(arr, rng)

def gen_blinds(rng):
    arr = bg(rng); bb = rng.integers(2, 6)
    x1 = rng.choice([0, rng.integers(0, max(1, W//3))])
    x2 = rng.choice([W, W - rng.integers(0, max(1, W//3))])
    if x2 <= x1: x1, x2 = 0, W
    w = x2 - x1; oy = rng.integers(0, max(1, 16-bb))
    bp = np.linspace(rng.uniform(0.15, 0.90), rng.uniform(0.15, 0.90), bb)
    pb = np.array([rng.random(w) < bp[r] for r in range(bb)], dtype=np.uint8)
    con = rng.uniform(0.55, 0.95)
    for by in range(oy, H, 16):
        bh = min(bb, H - by)
        for r in range(bh): arr[by+r, x1:x2] = rng.random(w) < (con * pb[r])
    return rand(arr, rng)

def gen_morse(rng):
    arr = bg(rng); seg = rng.integers(4, 8); rep = rng.integers(6, 10)
    pw = min(seg*2*rep, W-1); x0 = rng.integers(0, max(1, W-pw))
    y0 = rng.integers(0, 16)
    for y in np.arange(y0, H, 16):
        bits = (rng.random(pw) < rng.uniform(0.85, 0.99)).astype(np.uint8)
        for s in range(0, pw, seg*2): bits[s:min(s+seg, pw)] = 1 - bits[s:min(s+seg, pw)]
        arr[y, x0:x0+pw] = bits
    return rand(arr, rng)

GEN_FUNCS = {
    "0-TRASH": gen_trash,        "2-ZEBRA": gen_zebra,
    "3-OKAPI": gen_okapi,        "4-CHEETAH": gen_cheetah,
    "5-GRID": gen_grid,          "6-DIAGONAL": gen_diagonal,
    "7-BLINDS": gen_blinds,      "8-MORSE": gen_morse,
}

def generate_synth(n_per_class, seed, struct_only=True, tag="train"):
    rng_l = np.random.default_rng(seed)
    root  = WORK_DIR / f"synth_{tag}_s{seed}_n{n_per_class}"
    if root.exists(): shutil.rmtree(root)
    root.mkdir(parents=True, exist_ok=True)
    cls_list = STRUCT_FOLDERS if struct_only else classes
    samples  = []
    for cls in cls_list:
        d = root / cls; d.mkdir()
        for i in range(n_per_class):
            save_img(GEN_FUNCS[cls](rng_l), d / f"{cls}_{i:05d}.png")
        idx = class_to_idx[cls]
        for p in sorted(d.iterdir()): samples.append((str(p), idx))
    return samples

# ══════════════════════════════════════════════════════════════════════════════
# DATASET + MODELS
# ══════════════════════════════════════════════════════════════════════════════

t1ch = transforms.Compose([
    transforms.Resize((96, 192)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

class DefectDS(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p, lbl = self.samples[idx]
        return t1ch(Image.open(p).convert("L")), lbl

class TripletDS(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y, real = self.items[i]
        return t1ch(Image.open(p).convert("L")), y, int(real)

def make_loader(samples, shuffle=False, drop_last=False):
    return DataLoader(
        DefectDS(samples), batch_size=BATCH_SIZE,
        shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=(NUM_WORKERS > 0),
        drop_last=drop_last,
    )

def create_model(arch):
    if arch == "BaselineCNN":
        class BaselineCNN(nn.Module):
            def __init__(self, n=NUM_CLASSES):
                super().__init__()
                self.features = nn.Sequential(
                    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                )
                self.classifier = nn.Sequential(
                    nn.Flatten(),
                    nn.Linear(64*12*24, 128), nn.ReLU(), nn.Dropout(0.3),
                    nn.Linear(128, n),
                )
            def forward(self, x): return self.classifier(self.features(x))
        m = BaselineCNN()
    elif arch == "ResNet18_1ch":
        m = models.resnet18(weights=None)
        orig = m.conv1
        m.conv1 = nn.Conv2d(1, orig.out_channels, orig.kernel_size,
                            orig.stride, orig.padding, bias=orig.bias)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        m = nn.DataParallel(m)
    return m.to(device)

# ══════════════════════════════════════════════════════════════════════════════
# METRICS + TRAINING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def full_metrics(y_true, y_pred, label=""):
    acc  = float(accuracy_score(y_true, y_pred))
    mf1  = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    bal  = float(balanced_accuracy_score(y_true, y_pred))
    _, rec, _, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    struct_recall = float(np.mean([rec[i] for i in STRUCTURED_IDX]))
    if label:
        print(f"\n  [{label}]")
        print(f"    Accuracy:           {acc:.4f}")
        print(f"    Micro F1:           {mf1:.4f}")
        print(f"    Balanced Accuracy:  {bal:.4f}")
        print(f"    Struct Mean Recall: {struct_recall:.4f}  ← KEY")
        for i, cls in enumerate(classes):
            tag = " [TRASH]" if i == TRASH_IDX else ""
            print(f"    {cls+tag:22s}: recall={rec[i]:.3f}  (n={int(sup[i])})")
    return {"accuracy": acc, "micro_f1": mf1, "balanced_accuracy": bal,
            "structured_recall": struct_recall,
            "per_class_recall": rec.tolist(), "support": sup.tolist()}


def run_epoch(model, loader, crit, opt=None):
    model.train() if opt else model.eval()
    ls, n, yt, yp = 0.0, 0, [], []
    for imgs, lbl in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbl  = lbl.to(device, non_blocking=True)
        if opt: opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(opt is not None):
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(imgs); loss = crit(out, lbl)
            if opt:
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
        ls += loss.item() * imgs.size(0); n += imgs.size(0)
        yp.extend(out.argmax(1).detach().cpu().tolist())
        yt.extend(lbl.cpu().tolist())
    return ls/n, accuracy_score(yt, yp), f1_score(yt, yp, average="micro", zero_division=0)


def evaluate_model(model, loader):
    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for imgs, lbl in loader:
            imgs = imgs.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(imgs)
            yp.extend(out.argmax(1).cpu().tolist())
            yt.extend(lbl.tolist())
    return np.array(yt), np.array(yp)


def save_cm(yt, yp, title, path):
    cm      = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    cm_norm = np.nan_to_num(cm.astype(float) / cm.sum(axis=1, keepdims=True))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, t in [(axes[0], cm, f"{title}\nRaw CM (n={len(yt)})"),
                        (axes[1], cm_norm, "Normalised")]:
        im = ax.imshow(data, cmap="Blues")
        plt.colorbar(im, ax=ax)
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(NUM_CLASSES))
        ax.set_yticklabels(classes, fontsize=8)
        ax.set_title(t, fontsize=11)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                v = data[i, j]
                txt = f"{v:.2f}" if isinstance(v, float) else str(int(v))
                ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                        color="white" if v > data.max() * 0.6 else "black")
    plt.tight_layout()
    plt.savefig(path, dpi=180); plt.close()


def run_one_config(arch, synth_n, seed=SEED):
    set_seed(seed)
    train_synth = generate_synth(synth_n, seed, struct_only=True, tag=f"tr{synth_n}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    best_overall_mf1 = -1
    best_model_state = None
    fold_metrics = []
    fold_accs, fold_bals = [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(paths, labels), 1):
        real_tr = [(paths[i], int(labels[i])) for i in tr_idx]
        real_va = [(paths[i], int(labels[i])) for i in va_idx]
        tr_loader = make_loader(real_tr + train_synth, shuffle=True, drop_last=True)
        va_loader = make_loader(real_va)

        model = create_model(arch)
        crit  = nn.CrossEntropyLoss()
        opt   = optim.Adam(model.parameters(), lr=LR)
        best_fold_mf1, best_fold_state = -1, None

        for epoch in range(EPOCHS):
            run_epoch(model, tr_loader, crit, opt)
            _, _, va_mf1 = run_epoch(model, va_loader, crit)
            if va_mf1 > best_fold_mf1:
                best_fold_mf1  = va_mf1
                _raw = get_raw(model)
                best_fold_state = {k: v.detach().cpu().clone()
                                   for k, v in _raw.state_dict().items()}

        get_raw(model).load_state_dict(best_fold_state)
        yt, yp = evaluate_model(model, va_loader)
        m = full_metrics(yt, yp, f"Fold {fold} val")
        fold_metrics.append(m)
        fold_accs.append(m["accuracy"])
        fold_bals.append(m["balanced_accuracy"])

        if best_fold_mf1 > best_overall_mf1:
            best_overall_mf1 = best_fold_mf1
            best_model_state = best_fold_state

        del model, opt, crit; torch.cuda.empty_cache()

    return best_model_state, fold_metrics, fold_accs, fold_bals

# ══════════════════════════════════════════════════════════════════════════════
# GENERATE MIXED TEST SET 
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("GENERATING MIXED TEST SET")
print(f"  {MIXED_SYNTH_N} new synthetic images per class (seed+999 — never seen in training)")
print("  Note: synthetic TRASH is NOT generated (only structured classes)")
print("=" * 70)

test_synth     = generate_synth(MIXED_SYNTH_N, SEED+999, struct_only=True, tag="test")
real_test_set  = list(zip(paths.tolist(), labels.tolist()))
synth_test_set = test_synth
mixed_test_set = real_test_set + synth_test_set

print(f"  Real-only:  {len(real_test_set):5d}  (all classes including {TRASH_DOWNSAMPLE_N} TRASH)")
print(f"  Synth-only: {len(synth_test_set):5d}  ({MIXED_SYNTH_N}/class × {len(STRUCT_FOLDERS)} structured)")
print(f"  Mixed:      {len(mixed_test_set):5d}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK 0: REAL-ONLY FLOOR 
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK 0: REAL-ONLY FLOOR")
print("=" * 70)

floor_results = {}
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    print(f"\n  Architecture: {arch}")
    set_seed(SEED)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_accs, fold_bals = [], []
    best_mf1, best_state = -1, None

    for fold, (tr_idx, va_idx) in enumerate(skf.split(paths, labels), 1):
        real_tr = [(paths[i], int(labels[i])) for i in tr_idx]
        real_va = [(paths[i], int(labels[i])) for i in va_idx]
        tr_loader = make_loader(real_tr, shuffle=True, drop_last=False)
        va_loader = make_loader(real_va)
        model = create_model(arch); crit = nn.CrossEntropyLoss()
        opt = optim.Adam(model.parameters(), lr=LR)
        bfm, bfs = -1, None
        for epoch in range(EPOCHS):
            run_epoch(model, tr_loader, crit, opt)
            _, _, va_mf1 = run_epoch(model, va_loader, crit)
            if va_mf1 > bfm:
                bfm = va_mf1
                _raw = get_raw(model)
                bfs = {k: v.detach().cpu().clone() for k, v in _raw.state_dict().items()}
        get_raw(model).load_state_dict(bfs)
        yt, yp = evaluate_model(model, va_loader)
        m = full_metrics(yt, yp)
        fold_accs.append(m["accuracy"]); fold_bals.append(m["balanced_accuracy"])
        if bfm > best_mf1: best_mf1 = bfm; best_state = bfs
        del model, opt, crit; torch.cuda.empty_cache()
        print(f"    Fold {fold}: acc={m['accuracy']:.4f}  bal={m['balanced_accuracy']:.4f}")

    floor_results[arch] = {
        "fold_accs": fold_accs, "fold_bals": fold_bals,
        "acc_mean": float(np.mean(fold_accs)), "acc_std": float(np.std(fold_accs)),
        "bal_mean": float(np.mean(fold_bals)), "bal_std": float(np.std(fold_bals)),
        "best_state": best_state,
    }
    print(f"  → {arch} REAL-ONLY: acc={np.mean(fold_accs):.4f}±{np.std(fold_accs):.4f} "
          f"bal={np.mean(fold_bals):.4f}±{np.std(fold_bals):.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# FULL SWEEP: ALL 10 CONFIGURATIONS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("FULL SWEEP: ALL 10 CONFIGURATIONS")
print("=" * 70)

sweep_results = []
saved_states  = {}

for cfg in SWEEP_CONFIGS:
    arch  = cfg["arch"]
    synth = cfg["synth"]
    label = f"{arch}_s{synth}"

    print(f"\n{'#'*70}")
    print(f"CONFIG: {label}  (synth={synth}/structured class)")
    print(f"{'#'*70}")

    best_state, fold_metrics, fold_accs, fold_bals = run_one_config(arch, synth)
    saved_states[label] = {"arch": arch, "state": best_state}

    acc_mean = float(np.mean(fold_accs))
    acc_std  = float(np.std(fold_accs))
    bal_mean = float(np.mean(fold_bals))
    bal_std  = float(np.std(fold_bals))
    sr_mean  = float(np.mean([m["structured_recall"] for m in fold_metrics]))

    sweep_results.append({
        "label": label, "arch": arch, "synth": synth,
        "acc_mean": acc_mean, "acc_std": acc_std,
        "bal_mean": bal_mean, "bal_std": bal_std,
        "struct_recall_mean": sr_mean,
        "fold_accs": fold_accs, "fold_bals": fold_bals,
    })
    print(f"\n  → {label}: acc={acc_mean:.4f}±{acc_std:.4f}  "
          f"bal={bal_mean:.4f}±{bal_std:.4f}  struct={sr_mean:.4f}")

sweep_df = pd.DataFrame([{k: v for k, v in r.items()
                           if k not in ("fold_accs", "fold_bals")}
                          for r in sweep_results])
sweep_df.to_csv(OUT_DIR / "sweep_results.csv", index=False)

sweep_sorted = sorted(sweep_results, key=lambda r: r["bal_mean"], reverse=True)
best_overall   = sweep_sorted[0]
top_n_configs  = sweep_sorted[:TOP_N_FOR_MIXED_TEST]

print("\n" + "=" * 70)
print("SWEEP COMPLETE — RESULTS RANKED BY BALANCED ACCURACY")
print("=" * 70)
print(f"\n{'Rank':4s} {'Config':22s} {'Acc Mean':>10} {'Acc Std':>9} "
      f"{'Bal Acc':>9} {'Bal Std':>9} {'Struct':>8}")
print("-" * 77)
for i, r in enumerate(sweep_sorted, 1):
    flag = " ← BEST" if i == 1 else (f" ← top {i}" if i <= TOP_N_FOR_MIXED_TEST else "")
    print(f"{i:4d} {r['label']:22s} {r['acc_mean']:>10.4f} {r['acc_std']:>9.4f} "
          f"{r['bal_mean']:>9.4f} {r['bal_std']:>9.4f} {r['struct_recall_mean']:>8.4f}{flag}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK A: MIXED TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK A: MIXED TEST SET — TOP {TOP_N_FOR_MIXED_TEST} CONFIGS")
print("=" * 70)

mixed_test_rows = []
for r in top_n_configs:
    label = r["label"]
    arch  = r["arch"]
    cfg_dir = OUT_DIR / label
    cfg_dir.mkdir(exist_ok=True)

    model = create_model(arch)
    get_raw(model).load_state_dict(saved_states[label]["state"])
    model.eval()

    for test_name, test_data in [
        ("real_only",  real_test_set),
        ("synth_only", synth_test_set),
        ("mixed",      mixed_test_set),
    ]:
        loader = make_loader(test_data)
        yt, yp = evaluate_model(model, loader)
        m = full_metrics(yt, yp, f"{label} | {test_name} (n={len(test_data)})")
        
        report = classification_report(yt, yp, labels=list(range(NUM_CLASSES)), target_names=classes, zero_division=0)
        (cfg_dir / f"test_{test_name}_report.txt").write_text(report)
        save_cm(yt, yp, f"{label} | {test_name}", cfg_dir / f"cm_{test_name}.png")
        
        mixed_test_rows.append({
            "config": label, "test_set": test_name, "n": len(test_data),
            "accuracy": m["accuracy"], "balanced_acc": m["balanced_accuracy"],
            "struct_recall": m["structured_recall"],
            **{f"recall_{classes[i]}": m["per_class_recall"][i]
               for i in range(NUM_CLASSES)},
        })
    del model; torch.cuda.empty_cache()
pd.DataFrame(mixed_test_rows).to_csv(OUT_DIR / "mixed_test_results.csv", index=False)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK B + C: t-SNE AND UMAP
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK B+C: t-SNE / UMAP — Best model: {best_overall['label']}")
print("=" * 70)

COLORS = ["#455A64","#00796B","#E53935","#6A1B9A",
          "#F57C00","#1E2761","#2E7D32","#795548"]

best_arch = best_overall["arch"]
emb_model = create_model(best_arch)
emb_raw = get_raw(emb_model)
emb_raw.load_state_dict(saved_states[best_overall["label"]]["state"])
emb_raw.eval()

feats_collect = []
if best_arch == "BaselineCNN":
    hook_layer = emb_raw.classifier[2]
else:
    hook_layer = emb_raw.avgpool

hook = hook_layer.register_forward_hook(
    lambda m, inp, out: feats_collect.append(out.detach().cpu()))

# Re-define embed_synth and all_triplets right before DataLoader
print("  Generating embeddings datasets...")
embed_synth = generate_synth(EMBED_SYNTH_N, SEED+1, struct_only=True, tag="embed")
all_triplets = ([(p, y, True) for p, y in real_test_set] + [(p, y, False) for p, y in embed_synth])

Y_all, R_all = [], []
loader_emb = DataLoader(TripletDS(all_triplets), batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
with torch.no_grad():
    for imgs, yl, rl in loader_emb:
        imgs = imgs.to(device, non_blocking=True)
        emb_raw(imgs)
        Y_all.extend(yl.tolist()); R_all.extend(rl.tolist())

hook.remove()
del emb_model, emb_raw; torch.cuda.empty_cache()

F_all = torch.cat(feats_collect, 0).numpy()
if F_all.ndim > 2: F_all = F_all.reshape(F_all.shape[0], -1)
Y_all = np.array(Y_all); R_all = np.array(R_all)

def plot_embeddings(Z, Y, R, title, path):
    from matplotlib.lines import Line2D
    fig, ax = plt.subplots(figsize=(11, 9))
    for ci, cls in enumerate(classes):
        col = COLORS[ci % len(COLORS)]
        rm = (Y == ci) & (R == 1); sm = (Y == ci) & (R == 0)
        if rm.any(): ax.scatter(Z[rm,0], Z[rm,1], c=col, s=55, alpha=0.85, marker="o")
        if sm.any(): ax.scatter(Z[sm,0], Z[sm,1], c=col, s=28, alpha=0.50, marker="x")
    leg_cls  = [Line2D([0],[0],marker="o",color="w",markerfacecolor=COLORS[i],
                       markersize=9, label=classes[i]) for i in range(NUM_CLASSES)]
    leg_type = [Line2D([0],[0],marker="o",color="k",markersize=7,label="real (circles)"),
                Line2D([0],[0],marker="x",color="k",markersize=7,label="synth (crosses)")]
    ax.legend(handles=leg_cls+leg_type, loc="best", fontsize=8, ncol=2)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Component 1"); ax.set_ylabel("Component 2")
    ax.grid(True, alpha=0.2); plt.tight_layout()
    plt.savefig(path, dpi=180); plt.close()

print("  Running t-SNE...")
Z_tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=SEED, n_jobs=-1).fit_transform(F_all)
plot_embeddings(Z_tsne, Y_all, R_all, f"t-SNE — {best_overall['label']}", OUT_DIR / "tsne_embeddings.png")

if HAS_UMAP:
    print("  Running UMAP...")
    Z_umap = umap_lib.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED, n_jobs=-1).fit_transform(F_all)
    plot_embeddings(Z_umap, Y_all, R_all, f"UMAP — {best_overall['label']}", OUT_DIR / "umap_embeddings.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK D: GRAD-CAM
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK D: GRAD-CAM — Best model: {best_overall['label']}")
print("=" * 70)

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.grads = None; self.acts = None; self._h = []
        self._h.append(target_layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o)))
        self._h.append(target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0])))
    def compute(self, img_t, cls_idx):
        self.model.zero_grad()
        out = self.model(img_t)
        out[0, cls_idx].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(H, W), mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), out.argmax(1).item()
    def remove(self):
        for h in self._h: h.remove()

gc_arch  = best_overall["arch"]; gc_label = best_overall["label"]
if gc_arch != "BaselineCNN":
    best_cnn = next((r for r in sweep_sorted if r["arch"] == "BaselineCNN"), None)
    if best_cnn:
        gc_arch  = "BaselineCNN"
        gc_label = best_cnn["label"]

gc_model = create_model("BaselineCNN")
gc_raw = get_raw(gc_model)
gc_raw.load_state_dict(saved_states[gc_label]["state"])
gc_raw.eval()
gc = GradCAM(gc_raw, gc_raw.features[6])

correct_items, incorrect_items = [], []
all_pil = [Image.open(p).convert("L") for p, _ in real_test_set]

for pil_img, true_lbl in zip(all_pil, labels):
    tensor = t1ch(pil_img).unsqueeze(0).to(device)
    cam_map, pred = gc.compute(tensor, int(true_lbl))
    raw_arr = np.array(pil_img.resize((W, H)))
    item = (raw_arr, int(true_lbl), pred, cam_map)
    if pred == int(true_lbl): correct_items.append(item)
    else:                     incorrect_items.append(item)

gc.remove()
del gc_model, gc_raw; torch.cuda.empty_cache()

def save_gradcam_panel(items, n, title, path):
    items = items[:n]
    cols  = 4; rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols*2, figsize=(cols*4, rows*2.6))
    axes = axes.flatten()
    for k, (raw_img, true_lbl, pred_lbl, cam_map) in enumerate(items):
        ax_img = axes[k*2]; ax_cam = axes[k*2+1]
        ax_img.imshow(raw_img, cmap="gray", aspect="auto")
        ax_img.set_title(f"True: {classes[true_lbl]}", fontsize=7); ax_img.axis("off")
        ax_cam.imshow(raw_img, cmap="gray", alpha=0.55, aspect="auto")
        ax_cam.imshow(cam_map, cmap="jet", alpha=0.55, aspect="auto", vmin=0, vmax=1)
        col = "green" if pred_lbl == true_lbl else "red"
        ax_cam.set_title(f"Pred: {classes[pred_lbl]}", fontsize=7, color=col)
        ax_cam.axis("off")
    for j in range(len(items)*2, len(axes)): axes[j].axis("off")
    plt.suptitle(title, fontsize=11, fontweight="bold")
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()

save_gradcam_panel(correct_items, GRADCAM_N_CORRECT, f"Grad-CAM — Correct ({gc_label})", OUT_DIR / "gradcam_correct.png")
save_gradcam_panel(incorrect_items, GRADCAM_N_INCORRECT, f"Grad-CAM — Incorrect ({gc_label})", OUT_DIR / "gradcam_incorrect.png")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK E: PERTURBATION ROBUSTNESS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK E: PERTURBATION ROBUSTNESS — Best model: {best_overall['label']}")
print("=" * 70)

def apply_perturbation(img_tensor, ptype):
    arr = img_tensor.clone().squeeze().numpy()
    if ptype == "translation":
        arr = np.roll(arr, shift=3, axis=1)
    elif ptype == "rotation":
        from scipy.ndimage import rotate as ndrotate
        arr = np.clip(ndrotate(arr, angle=3, reshape=False, mode="nearest"), 0, 1)
    elif ptype == "threshold_shift":
        arr = (arr > 0.55).astype(np.float32)
    elif ptype == "salt_pepper":
        noise_mask = np.random.random(arr.shape) < 0.02
        arr = arr.copy()
        arr[noise_mask] = 1 - arr[noise_mask]
    elif ptype == "morphological_erosion":
        from scipy.ndimage import binary_erosion
        arr_binary = (arr > 0.5).astype(np.uint8)
        arr = binary_erosion(arr_binary, iterations=1).astype(np.float32)
    elif ptype == "morphological_dilation":
        from scipy.ndimage import binary_dilation
        arr_binary = (arr > 0.5).astype(np.uint8)
        arr = binary_dilation(arr_binary, iterations=1).astype(np.float32)
    return torch.tensor(arr).unsqueeze(0).unsqueeze(0).float()

PERTURBATIONS = [
    "clean", "translation", "rotation", "threshold_shift",
    "salt_pepper", "morphological_erosion", "morphological_dilation",
]

p_arch  = best_overall["arch"]
p_label = best_overall["label"]
p_model = create_model(p_arch)
p_raw = get_raw(p_model)
p_raw.load_state_dict(saved_states[p_label]["state"])
p_raw.eval()

skf_p = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
pert_fold_bals = defaultdict(list)

for fold, (_, va_idx) in enumerate(skf_p.split(paths, labels), 1):
    real_va     = [(paths[i], int(labels[i])) for i in va_idx]
    imgs_loaded = [t1ch(Image.open(p).convert("L")) for p, _ in real_va]
    true_labels = [y for _, y in real_va]
    
    for ptype in PERTURBATIONS:
        yt, yp = [], []
        for img_t, ty in zip(imgs_loaded, true_labels):
            pt = apply_perturbation(img_t.unsqueeze(0), ptype).to(device)
            with torch.no_grad(): 
                pred = p_raw(pt).argmax(1).item()
            yt.append(ty)
            yp.append(pred)
        bal_acc = balanced_accuracy_score(yt, yp)
        pert_fold_bals[ptype].append(bal_acc)

del p_model, p_raw; torch.cuda.empty_cache()

clean_mean  = np.mean(pert_fold_bals["clean"])
pert_rows   = []

for ptype in PERTURBATIONS:
    m, s = np.mean(pert_fold_bals[ptype]), np.std(pert_fold_bals[ptype])
    pert_rows.append({
        "perturbation": ptype,
        "bal_acc_mean": round(m, 4),
        "bal_acc_std": round(s, 4),
        "drop": round(clean_mean - m, 4)
    })
    print(f"  {ptype:28s}: bal={m:.4f}±{s:.4f}  drop={clean_mean-m:+.4f}")

pd.DataFrame(pert_rows).to_csv(OUT_DIR / "perturbation_results.csv", index=False)

fig, ax = plt.subplots(figsize=(13, 5))
names = [r["perturbation"] for r in pert_rows]
means = [r["bal_acc_mean"] for r in pert_rows]
stds  = [r["bal_acc_std"]  for r in pert_rows]

bar_colors = []
for n in names:
    if n == "clean": bar_colors.append("#2E7D32")
    elif n in ["morphological_erosion", "morphological_dilation", "salt_pepper"]: bar_colors.append("#E53935")
    else: bar_colors.append("#1E2761")

ax.bar(range(len(names)), means, yerr=stds, capsize=4, color=bar_colors, edgecolor="white", linewidth=0.5, alpha=0.9)
ax.axhline(clean_mean, color="gray", ls="--", lw=1.5, label=f"Clean ({clean_mean:.4f})")
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=25, ha="right", fontsize=10)
ax.set_ylabel("Balanced Accuracy", fontsize=11)
ax.set_ylim(0.4, 1.02)
ax.set_title(f"Perturbation Robustness — {p_label}", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(OUT_DIR / "perturbation_robustness.png", dpi=180)
plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK F: WILCOXON STATISTICAL TEST
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK F: WILCOXON SIGNED-RANK TEST")
print("=" * 70)

best_cnn_r = next((r for r in sweep_sorted if r["arch"] == "BaselineCNN"), None)
best_r18_r = next((r for r in sweep_sorted if r["arch"] == "ResNet18_1ch"), None)

wilcoxon_txt = []
if best_cnn_r and best_r18_r:
    cnn_f = best_cnn_r["fold_accs"]
    r18_f = best_r18_r["fold_accs"]
    wilcoxon_txt.append(f"Best CNN: {best_cnn_r['label']}  folds={cnn_f}  mean={np.mean(cnn_f):.4f}")
    wilcoxon_txt.append(f"Best R18: {best_r18_r['label']}  folds={r18_f}  mean={np.mean(r18_f):.4f}")
    if HAS_SCIPY:
        try:
            stat, p = scipy_wilcoxon(cnn_f, r18_f)
            sig = p < 0.05
            wilcoxon_txt.append(f"Wilcoxon W={stat:.4f}  p={p:.4f}  significant={sig}")
        except Exception as e:
            wilcoxon_txt.append(f"Wilcoxon failed: {e}")

(OUT_DIR / "statistical_test.txt").write_text("\n".join(wilcoxon_txt))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK G: MODEL EFFICIENCY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK G: MODEL EFFICIENCY")
print("=" * 70)

dummy = torch.zeros(1, 1, H, W).to(device)
eff_rows = []
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    m = create_model(arch)
    raw = m.module if isinstance(m, nn.DataParallel) else m
    params = sum(p.numel() for p in raw.parameters())
    tmp = WORK_DIR / f"tmp_{arch}.pt"
    torch.save(raw.state_dict(), tmp)
    size_mb = os.path.getsize(tmp)/1e6
    tmp.unlink()
    
    m.eval()
    with torch.no_grad():
        for _ in range(10): m(dummy)
    
    times = []
    with torch.no_grad():
        for _ in range(100):
            t0 = time.perf_counter()
            m(dummy)
            if device.type == "cuda": torch.cuda.synchronize()
            times.append((time.perf_counter()-t0)*1000)
    
    eff_rows.append({
        "architecture": arch,
        "parameters": params,
        "size_mb": round(size_mb, 2),
        "inference_mean_ms": round(float(np.mean(times)), 3),
        "inference_std_ms":  round(float(np.std(times)), 3),
        "meets_target": size_mb <= 10.0,
    })
    print(f"  {arch}: {params:,} params  {size_mb:.2f} MB")
    del m; torch.cuda.empty_cache()

pd.DataFrame(eff_rows).to_csv(OUT_DIR / "efficiency.csv", index=False)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK H: SWEEP CHARTS
# ══════════════════════════════════════════════════════════════════════════════

synth_amts = [250, 500, 750, 1000, 1250]

def get_vals(arch, metric):
    return [next(r[metric] for r in sweep_results
                 if r["arch"] == arch and r["synth"] == s)
            for s in synth_amts]

cnn_acc = get_vals("BaselineCNN",  "acc_mean")
r18_acc = get_vals("ResNet18_1ch", "acc_mean")
cnn_bal = get_vals("BaselineCNN",  "bal_mean")
r18_bal = get_vals("ResNet18_1ch", "bal_mean")

cnn_acc_std = get_vals("BaselineCNN",  "acc_std")
r18_acc_std = get_vals("ResNet18_1ch", "acc_std")
cnn_bal_std = get_vals("BaselineCNN",  "bal_std")
r18_bal_std = get_vals("ResNet18_1ch", "bal_std")

fl_cnn_acc = floor_results["BaselineCNN"]["acc_mean"]
fl_r18_acc = floor_results["ResNet18_1ch"]["acc_mean"]
fl_cnn_bal = floor_results["BaselineCNN"]["bal_mean"]
fl_r18_bal = floor_results["ResNet18_1ch"]["bal_mean"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (cy, ry, cstd, rstd, fl_c, fl_r, ylabel, title) in zip(axes, [
    (cnn_acc, r18_acc, cnn_acc_std, r18_acc_std, fl_cnn_acc, fl_r18_acc,
     "Accuracy (= Micro F1)", "Synth Sweep — Accuracy"),
    (cnn_bal, r18_bal, cnn_bal_std, r18_bal_std, fl_cnn_bal, fl_r18_bal,
     "Balanced Accuracy",     "Synth Sweep — Balanced Accuracy"),
]):
    ax.errorbar(synth_amts, cy, yerr=cstd, fmt="o-", color="#00796B",
                lw=2, capsize=4, label="BaselineCNN")
    ax.errorbar(synth_amts, ry, yerr=rstd, fmt="s-", color="#1E2761",
                lw=2, capsize=4, label="ResNet18_1ch")
    ax.axhline(fl_c, color="#00796B", ls="--", lw=1.5, alpha=0.7,
               label=f"CNN real-only floor ({fl_c:.3f})")
    ax.axhline(fl_r, color="#1E2761", ls=":",  lw=1.5, alpha=0.7,
               label=f"R18 real-only floor ({fl_r:.3f})")
    ax.set_xlabel("Synthetic images per structured class", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(synth_amts)

plt.suptitle("Synthetic Augmentation Sweep", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "sweep_charts_with_floor.png", dpi=180)
plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK I: SYNTHETIC EXAMPLES PANEL
# ══════════════════════════════════════════════════════════════════════════════

rng_ex = np.random.default_rng(2025)
n_var  = 3
fig, axes = plt.subplots(NUM_CLASSES, n_var, figsize=(n_var*2.2, NUM_CLASSES*1.4))
for row, cls in enumerate(classes):
    for col in range(n_var):
        arr = GEN_FUNCS[cls](rng_ex)
        axes[row, col].imshow(arr, cmap="gray", interpolation="nearest", aspect="auto")
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=9, rotation=0,
                                       ha="right", va="center", labelpad=50)

plt.suptitle("Synthetic Examples — 3 Random Variants per Class", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "synthetic_examples.png", dpi=160)
plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK J: SCATTERPLOT
# ══════════════════════════════════════════════════════════════════════════════

eff_df = pd.read_csv(OUT_DIR / "efficiency.csv")
best_per_arch = {}
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    best_per_arch[arch] = next(r for r in sweep_sorted if r["arch"] == arch)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_s = ["#00796B", "#1E2761"]

for ax, (y_key, ylabel, title) in zip(axes, [
    ("acc_mean", "Accuracy (Micro F1)", "Model Complexity vs Accuracy"),
    ("bal_mean", "Balanced Accuracy",   "Model Complexity vs Balanced Accuracy"),
]):
    for i, (arch, col) in enumerate(zip(["BaselineCNN","ResNet18_1ch"], colors_s)):
        r   = best_per_arch[arch]
        row = eff_df[eff_df["architecture"] == arch].iloc[0]
        ax.scatter(row.parameters/1e6, r[y_key], s=300, c=col, zorder=5, label=arch)
        ax.annotate(
            f"{arch}\n{row.parameters/1e6:.1f}M  {row.size_mb:.1f}MB\n"
            f"{row.inference_mean_ms:.1f}ms",
            (row.parameters/1e6, r[y_key]),
            xytext=(15, -40), textcoords="offset points", fontsize=9, color=col,
            arrowprops=dict(arrowstyle="->", color=col, lw=1.5))
    ax.set_xlabel("Parameters (millions)", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.88, 1.01)
    ax.set_xlim(0, 15)

plt.suptitle("Efficiency vs Performance", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "scatterplot_params_vs_accuracy.png", dpi=180)
plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK K: THESIS SUMMARY TABLES
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK K: THESIS SUMMARY — COPY INTO OVERLEAF")
print("=" * 70)

summary_txt = []
summary_txt.append("=" * 80)
summary_txt.append("THESIS EVALUATION PIPELINE — UPDATED RESULTS")
summary_txt.append("=" * 80)
summary_txt.append("")
summary_txt.append(f"DATASET CONFIGURATION:")
summary_txt.append(f"  TRASH (0-EMPTY + 1-NOISE combined): {TRASH_DOWNSAMPLE_N} real images")
summary_txt.append(f"  Total classes: {NUM_CLASSES}")
summary_txt.append(f"  Total real samples: {len(real_test_set)}")
summary_txt.append("")
summary_txt.append(f"SWEEP RESULTS (ranked by balanced accuracy):")
for i, r in enumerate(sweep_sorted[:5], 1):
    summary_txt.append(
        f"  {i}. {r['label']:22s}  "
        f"acc={r['acc_mean']:.4f}  bal={r['bal_mean']:.4f}  "
        f"struct={r['struct_recall_mean']:.4f}")
summary_txt.append("")
summary_txt.append(f"BEST OVERALL: {best_overall['label']}")
summary_txt.append(f"  Balanced Accuracy: {best_overall['bal_mean']:.4f} ± {best_overall['bal_std']:.4f}")
summary_txt.append(f"  Accuracy: {best_overall['acc_mean']:.4f} ± {best_overall['acc_std']:.4f}")
summary_txt.append("")
summary_txt.append(f"PERTURBATION ROBUSTNESS (binary-relevant):")
for r in pert_rows[:3]:
    summary_txt.append(f"  {r['perturbation']:25s}: {r['bal_acc_mean']:.4f}  (drop={r['drop']:+.4f})")
summary_txt.append("")
summary_txt.append("=" * 80)

(OUT_DIR / "SUMMARY.txt").write_text("\n".join(summary_txt))

print("\n" + "=" * 70)
print("PIPELINE COMPLETE ✓")
print("=" * 70)

Device: cuda | GPUs: 2

STEP 1: TRASH CLASS DOWNSAMPLING
Target: exactly 10 images from combined NOISE+EMPTY
Total NOISE + EMPTY images: 1058
Downsampling to: 10
Selected 10 images for TRASH class

8-class setup (after downsampling):
  0: 0-TRASH
  1: 2-ZEBRA
  2: 3-OKAPI
  3: 4-CHEETAH
  4: 5-GRID
  5: 6-DIAGONAL
  6: 7-BLINDS
  7: 8-MORSE

Final dataset: 98 total images
Distribution:
  0-TRASH             :    10  ( 10.2%)
  2-ZEBRA             :    11  ( 11.2%)
  3-OKAPI             :    10  ( 10.2%)
  4-CHEETAH           :    14  ( 14.3%)
  5-GRID              :    23  ( 23.5%)
  6-DIAGONAL          :    10  ( 10.2%)
  7-BLINDS            :     9  (  9.2%)
  8-MORSE             :    11  ( 11.2%)

GENERATING MIXED TEST SET
  100 new synthetic images per class (seed+999 — never seen in training)
  Note: synthetic TRASH is NOT generated (only structured classes)
  Real-only:     98  (all classes including 10 TRASH)
  Synth-only:   700  (100/class × 7 structured)
  Mixed:        798

B

In [1]:
# ============================================================
# COMPLETE THESIS EVALUATION PIPELINE — FIXED VERSION
# UHasselt DSI × Gpixel — Nuthi Raviteja Pediredla
#
# FIXES IN THIS VERSION:
#   FIX 1: try/except around EVERY block — if one crashes, the rest still run
#   FIX 2: Perturbation chart ylim set to 0.0 (was 0.4, clipping erosion/dilation bars)
#   FIX 3: FINAL BLOCK zips ALL outputs and prints full manifest of saved files
#   FIX 4: Every plt.savefig now followed by explicit print confirming file saved
#   FIX 5: Best-model confusion matrices saved explicitly (not only inside Block A)
#   FIX 6: t-SNE/UMAP hook — avgpool output reshaped correctly for R18
#   FIX 7: Grad-CAM — torch.enable_grad() context added explicitly so backward works
#   FIX 8: Block F Wilcoxon — prints result to stdout (not only to file)
#   FIX 9: All blocks wrapped in section banners so failures are easy to spot in logs
#
# EXPECTED RUNTIME on Kaggle T4 × 2 GPU: ~5.5 hours
# ============================================================

import os, random, shutil, time, warnings, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageFilter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

# ── Optional packages ────────────────────────────────────────────────────────
try:
    import umap as umap_lib
    HAS_UMAP = True
except ImportError:
    os.system("pip install umap-learn -q")
    try:
        import umap as umap_lib
        HAS_UMAP = True
    except:
        HAS_UMAP = False
        print("WARNING: umap-learn not available — UMAP skipped.")

try:
    from scipy.stats import wilcoxon as scipy_wilcoxon
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    print("WARNING: scipy not available — Wilcoxon test skipped.")

try:
    from scipy.ndimage import rotate as ndrotate, binary_erosion, binary_dilation
    HAS_SCIPY_NDIMAGE = True
except ImportError:
    HAS_SCIPY_NDIMAGE = False
    print("WARNING: scipy.ndimage not available — rotation/morphology perturbations skipped.")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

SEED        = 42
N_SPLITS    = 4
EPOCHS      = 15
LR          = 1e-3
BATCH_SIZE  = 128
NUM_WORKERS = 2

SWEEP_CONFIGS = [
    {"arch": "BaselineCNN",  "synth": 250},
    {"arch": "BaselineCNN",  "synth": 500},
    {"arch": "BaselineCNN",  "synth": 750},
    {"arch": "BaselineCNN",  "synth": 1000},
    {"arch": "BaselineCNN",  "synth": 1250},
    {"arch": "ResNet18_1ch", "synth": 250},
    {"arch": "ResNet18_1ch", "synth": 500},
    {"arch": "ResNet18_1ch", "synth": 750},
    {"arch": "ResNet18_1ch", "synth": 1000},
    {"arch": "ResNet18_1ch", "synth": 1250},
]

TOP_N_FOR_MIXED_TEST = 3
EMBED_SYNTH_N        = 80
MIXED_SYNTH_N        = 100
GRADCAM_N_CORRECT    = 16
GRADCAM_N_INCORRECT  = 9
TRASH_DOWNSAMPLE_N   = 10

DATA_ROOT = Path("/kaggle/input/datasets/teja19129/g-pixel/96x192")
WORK_DIR  = Path("/kaggle/working")
OUT_DIR   = WORK_DIR / "thesis_pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
scaler  = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")
print(f"Output directory: {OUT_DIR}")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def get_raw(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def strip_dp(state_dict):
    return {
        (k[len("module."):] if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }


def confirm_saved(path):
    """Print a confirmation that a file was saved and its size."""
    p = Path(path)
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f"  ✓ SAVED: {p.name}  ({size_kb:.1f} KB)  →  {p}")
    else:
        print(f"  ✗ FAILED TO SAVE: {path}")


# ══════════════════════════════════════════════════════════════════════════════
# DATA SETUP — WITH TRASH DOWNSAMPLING
# ══════════════════════════════════════════════════════════════════════════════

valid_exts     = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TRASH_FOLDERS  = {"0-EMPTY", "1-NOISE"}
all_folders    = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
STRUCT_FOLDERS = [f for f in all_folders if f not in TRASH_FOLDERS]

print("\n" + "=" * 70)
print("STEP 1: TRASH CLASS DOWNSAMPLING")
print(f"Target: exactly {TRASH_DOWNSAMPLE_N} images from combined NOISE+EMPTY")
print("=" * 70)

all_trash_paths = []
for folder in TRASH_FOLDERS:
    cls_dir = DATA_ROOT / folder
    if cls_dir.exists():
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in valid_exts:
                all_trash_paths.append(str(p))

print(f"Total NOISE + EMPTY images: {len(all_trash_paths)}")
rng_trash = np.random.default_rng(SEED)
trash_indices = rng_trash.choice(len(all_trash_paths), size=TRASH_DOWNSAMPLE_N, replace=False)
selected_trash_paths = [all_trash_paths[i] for i in sorted(trash_indices)]
print(f"Selected {len(selected_trash_paths)} images for TRASH class")

classes        = ["0-TRASH"] + STRUCT_FOLDERS
class_to_idx   = {cls: i for i, cls in enumerate(classes)}
idx_to_class   = {i: cls for cls, i in class_to_idx.items()}
NUM_CLASSES    = len(classes)
TRASH_IDX      = 0
STRUCTURED_IDX = list(range(1, NUM_CLASSES))

print(f"\n{NUM_CLASSES}-class setup:")
for cls, idx in class_to_idx.items():
    print(f"  {idx}: {cls}")

all_samples = [(p, 0) for p in selected_trash_paths]
for folder in STRUCT_FOLDERS:
    cls_dir = DATA_ROOT / folder
    if cls_dir.exists():
        idx = class_to_idx[folder]
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in valid_exts:
                all_samples.append((str(p), idx))

paths  = np.array([p for p, _ in all_samples])
labels = np.array([y for _, y in all_samples])

print(f"\nFinal dataset: {len(all_samples)} total images")
for k, v in sorted(Counter(labels).items()):
    print(f"  {idx_to_class[k]:20s}: {v:5d}  ({100*v/len(all_samples):5.1f}%)")

# ══════════════════════════════════════════════════════════════════════════════
# SYNTHETIC GENERATORS
# ══════════════════════════════════════════════════════════════════════════════

H, W = 96, 192

def save_img(arr, path):
    Image.fromarray((arr.astype(np.uint8) * 255)).save(path)

def bg(rng, prob=None):
    return (rng.random((H, W)) < (prob or rng.uniform(0.0001, 0.003))).astype(np.uint8)

def rand(arr, rng):
    img = Image.fromarray((arr.astype(np.uint8) * 255))
    if rng.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.2, 0.7))))
    arr = np.array(img).astype(np.float32) / 255.0
    if rng.random() < 0.5:
        arr = np.clip(arr + rng.normal(0, rng.uniform(0.01, 0.04), arr.shape), 0, 1)
    return (arr > rng.uniform(0.35, 0.65)).astype(np.uint8)

def gen_trash(rng):
    arr = ((rng.random((H, W)) < rng.uniform(0.001, 0.05)).astype(np.uint8)
           if rng.random() < 0.5 else np.zeros((H, W), dtype=np.uint8))
    return rand(arr, rng)

def gen_zebra(rng):
    arr = bg(rng); bw = rng.integers(120, 170); oy = rng.integers(0, 16)
    x0 = rng.integers(0, max(1, W - bw)); d = rng.uniform(0.45, 1.0)
    for y in np.arange(oy, H, 16): arr[y, x0:x0+bw] = rng.random(bw) < d
    return rand(arr, rng)

def gen_okapi(rng):
    arr = bg(rng); bw = rng.integers(55, 95); oy = rng.integers(0, 16)
    x0 = rng.integers(0, max(1, W - bw)); d = rng.uniform(0.40, 1.0)
    for y in np.arange(oy, H, 16): arr[y, x0:x0+bw] = rng.random(bw) < d
    return rand(arr, rng)

def gen_cheetah(rng):
    arr = bg(rng, rng.uniform(0.00005, 0.001))
    ox, oy = rng.integers(0, 2), rng.integers(0, 2)
    pat = (rng.random((H//2+1, W//2+1)) < rng.uniform(0.90, 0.995)).astype(np.uint8)
    arr[oy::2, ox::2] = pat[:arr[oy::2, ox::2].shape[0], :arr[oy::2, ox::2].shape[1]]
    if rng.random() < 0.7:
        bh = rng.integers(6, 10)
        for y in range(rng.integers(0, 16), H, 16): arr[y:y+bh, :] = 0
    return rand(arr, rng)

def gen_grid(rng):
    arr = bg(rng); pw = 12 * rng.integers(12, 14)
    x0 = rng.integers(0, max(1, W - pw)); y0 = rng.integers(0, 16)
    d = rng.uniform(0.85, 0.99)
    for y in np.arange(y0, H, 16):
        cols = np.arange(x0, x0+pw, 12); arr[y, cols] = rng.random(len(cols)) < d
    return rand(arr, rng)

def gen_diagonal(rng):
    arr = bg(rng, rng.uniform(0.00005, 0.0005))
    ox, oy = rng.integers(0, 12), rng.integers(0, 16)
    bp = np.array([1, 1, 0, 0] if rng.random() < 0.5 else [1, 0, 0, 0])
    pat = np.roll(np.tile(bp, rng.integers(3, 6)), rng.integers(0, len(bp)*3))
    cols = np.arange(ox, W, 12); n = min(len(cols), len(pat))
    cols, pat = cols[:n], pat[:n]
    for i, y in enumerate(np.arange(oy, H, 16)):
        sh = np.roll(pat, i); arr[y, cols] = rng.random(len(sh)) < (0.92 * sh)
    if rng.random() < 0.5: arr = np.fliplr(arr)
    return rand(arr, rng)

def gen_blinds(rng):
    arr = bg(rng); bb = rng.integers(2, 6)
    x1 = rng.choice([0, rng.integers(0, max(1, W//3))])
    x2 = rng.choice([W, W - rng.integers(0, max(1, W//3))])
    if x2 <= x1: x1, x2 = 0, W
    w = x2 - x1; oy = rng.integers(0, max(1, 16-bb))
    bp = np.linspace(rng.uniform(0.15, 0.90), rng.uniform(0.15, 0.90), bb)
    pb = np.array([rng.random(w) < bp[r] for r in range(bb)], dtype=np.uint8)
    con = rng.uniform(0.55, 0.95)
    for by in range(oy, H, 16):
        bh = min(bb, H - by)
        for r in range(bh): arr[by+r, x1:x2] = rng.random(w) < (con * pb[r])
    return rand(arr, rng)

def gen_morse(rng):
    arr = bg(rng); seg = rng.integers(4, 8); rep = rng.integers(6, 10)
    pw = min(seg*2*rep, W-1); x0 = rng.integers(0, max(1, W-pw))
    y0 = rng.integers(0, 16)
    for y in np.arange(y0, H, 16):
        bits = (rng.random(pw) < rng.uniform(0.85, 0.99)).astype(np.uint8)
        for s in range(0, pw, seg*2): bits[s:min(s+seg, pw)] = 1 - bits[s:min(s+seg, pw)]
        arr[y, x0:x0+pw] = bits
    return rand(arr, rng)

GEN_FUNCS = {
    "0-TRASH":    gen_trash,
    "2-ZEBRA":    gen_zebra,
    "3-OKAPI":    gen_okapi,
    "4-CHEETAH":  gen_cheetah,
    "5-GRID":     gen_grid,
    "6-DIAGONAL": gen_diagonal,
    "7-BLINDS":   gen_blinds,
    "8-MORSE":    gen_morse,
}

def generate_synth(n_per_class, seed, struct_only=True, tag="train"):
    rng_l = np.random.default_rng(seed)
    root  = WORK_DIR / f"synth_{tag}_s{seed}_n{n_per_class}"
    if root.exists(): shutil.rmtree(root)
    root.mkdir(parents=True, exist_ok=True)
    cls_list = STRUCT_FOLDERS if struct_only else classes
    samples  = []
    for cls in cls_list:
        d = root / cls; d.mkdir()
        for i in range(n_per_class):
            save_img(GEN_FUNCS[cls](rng_l), d / f"{cls}_{i:05d}.png")
        idx = class_to_idx[cls]
        for p in sorted(d.iterdir()): samples.append((str(p), idx))
    return samples

# ══════════════════════════════════════════════════════════════════════════════
# DATASETS + MODELS
# ══════════════════════════════════════════════════════════════════════════════

t1ch = transforms.Compose([
    transforms.Resize((96, 192)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

class DefectDS(Dataset):
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p, lbl = self.samples[idx]
        return t1ch(Image.open(p).convert("L")), lbl

class TripletDS(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y, real = self.items[i]
        return t1ch(Image.open(p).convert("L")), y, int(real)

def make_loader(samples, shuffle=False, drop_last=False):
    return DataLoader(
        DefectDS(samples), batch_size=BATCH_SIZE,
        shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=(NUM_WORKERS > 0),
        drop_last=drop_last,
    )

def create_model(arch):
    if arch == "BaselineCNN":
        class BaselineCNN(nn.Module):
            def __init__(self, n=NUM_CLASSES):
                super().__init__()
                self.features = nn.Sequential(
                    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                )
                self.classifier = nn.Sequential(
                    nn.Flatten(),
                    nn.Linear(64*12*24, 128), nn.ReLU(), nn.Dropout(0.3),
                    nn.Linear(128, n),
                )
            def forward(self, x): return self.classifier(self.features(x))
        m = BaselineCNN()
    elif arch == "ResNet18_1ch":
        m = models.resnet18(weights=None)
        orig = m.conv1
        m.conv1 = nn.Conv2d(1, orig.out_channels, orig.kernel_size,
                            orig.stride, orig.padding, bias=orig.bias)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        m = nn.DataParallel(m)
    return m.to(device)

# ══════════════════════════════════════════════════════════════════════════════
# METRICS + TRAINING
# ══════════════════════════════════════════════════════════════════════════════

def full_metrics(y_true, y_pred, label=""):
    acc  = float(accuracy_score(y_true, y_pred))
    mf1  = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    bal  = float(balanced_accuracy_score(y_true, y_pred))
    _, rec, _, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    struct_recall = float(np.mean([rec[i] for i in STRUCTURED_IDX]))
    if label:
        print(f"\n  [{label}]")
        print(f"    Accuracy:           {acc:.4f}")
        print(f"    Micro F1:           {mf1:.4f}")
        print(f"    Balanced Accuracy:  {bal:.4f}")
        print(f"    Struct Mean Recall: {struct_recall:.4f}  ← KEY")
        for i, cls in enumerate(classes):
            tag = " [TRASH]" if i == TRASH_IDX else ""
            print(f"    {cls+tag:22s}: recall={rec[i]:.3f}  (n={int(sup[i])})")
    return {"accuracy": acc, "micro_f1": mf1, "balanced_accuracy": bal,
            "structured_recall": struct_recall,
            "per_class_recall": rec.tolist(), "support": sup.tolist()}

def run_epoch(model, loader, crit, opt=None):
    model.train() if opt else model.eval()
    ls, n, yt, yp = 0.0, 0, [], []
    for imgs, lbl in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbl  = lbl.to(device, non_blocking=True)
        if opt: opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(opt is not None):
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(imgs); loss = crit(out, lbl)
            if opt:
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
        ls += loss.item() * imgs.size(0); n += imgs.size(0)
        yp.extend(out.argmax(1).detach().cpu().tolist())
        yt.extend(lbl.cpu().tolist())
    return ls/n, accuracy_score(yt, yp), f1_score(yt, yp, average="micro", zero_division=0)

def evaluate_model(model, loader):
    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for imgs, lbl in loader:
            imgs = imgs.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(imgs)
            yp.extend(out.argmax(1).cpu().tolist())
            yt.extend(lbl.tolist())
    return np.array(yt), np.array(yp)

def save_cm(yt, yp, title, path):
    cm      = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    cm_norm = np.nan_to_num(cm.astype(float) / cm.sum(axis=1, keepdims=True))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, t in [(axes[0], cm, f"{title}\nRaw CM (n={len(yt)})"),
                        (axes[1], cm_norm, "Normalised")]:
        im = ax.imshow(data, cmap="Blues")
        plt.colorbar(im, ax=ax)
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(NUM_CLASSES))
        ax.set_yticklabels(classes, fontsize=8)
        ax.set_title(t, fontsize=11)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                v = data[i, j]
                txt = f"{v:.2f}" if isinstance(v, float) else str(int(v))
                ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                        color="white" if v > data.max() * 0.6 else "black")
    plt.tight_layout()
    plt.savefig(path, dpi=180); plt.close()

def run_one_config(arch, synth_n, seed=SEED):
    set_seed(seed)
    train_synth = generate_synth(synth_n, seed, struct_only=True, tag=f"tr{synth_n}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    best_overall_mf1 = -1
    best_model_state = None
    fold_metrics_list = []
    fold_accs, fold_bals = [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(paths, labels), 1):
        real_tr = [(paths[i], int(labels[i])) for i in tr_idx]
        real_va = [(paths[i], int(labels[i])) for i in va_idx]
        tr_loader = make_loader(real_tr + train_synth, shuffle=True, drop_last=True)
        va_loader = make_loader(real_va)
        model = create_model(arch)
        crit  = nn.CrossEntropyLoss()
        opt   = optim.Adam(model.parameters(), lr=LR)
        best_fold_mf1, best_fold_state = -1, None

        for epoch in range(EPOCHS):
            run_epoch(model, tr_loader, crit, opt)
            _, _, va_mf1 = run_epoch(model, va_loader, crit)
            if va_mf1 > best_fold_mf1:
                best_fold_mf1 = va_mf1
                _raw = get_raw(model)
                best_fold_state = {k: v.detach().cpu().clone()
                                   for k, v in _raw.state_dict().items()}

        get_raw(model).load_state_dict(best_fold_state)
        yt, yp = evaluate_model(model, va_loader)
        m = full_metrics(yt, yp, f"Fold {fold} val")
        fold_metrics_list.append(m)
        fold_accs.append(m["accuracy"])
        fold_bals.append(m["balanced_accuracy"])

        if best_fold_mf1 > best_overall_mf1:
            best_overall_mf1 = best_fold_mf1
            best_model_state = best_fold_state

        del model, opt, crit; torch.cuda.empty_cache()

    return best_model_state, fold_metrics_list, fold_accs, fold_bals

# ══════════════════════════════════════════════════════════════════════════════
# GENERATE MIXED TEST SET
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("GENERATING MIXED TEST SET")
print(f"  {MIXED_SYNTH_N} new synthetic images per structured class (seed+999)")
print("=" * 70)

test_synth     = generate_synth(MIXED_SYNTH_N, SEED+999, struct_only=True, tag="test")
real_test_set  = list(zip(paths.tolist(), labels.tolist()))
synth_test_set = test_synth
mixed_test_set = real_test_set + synth_test_set

print(f"  Real-only:  {len(real_test_set):5d}  ({TRASH_DOWNSAMPLE_N} TRASH + {len(real_test_set)-TRASH_DOWNSAMPLE_N} structured)")
print(f"  Synth-only: {len(synth_test_set):5d}  ({MIXED_SYNTH_N}/class × {len(STRUCT_FOLDERS)} structured)")
print(f"  Mixed:      {len(mixed_test_set):5d}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK 0: REAL-ONLY FLOOR
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK 0: REAL-ONLY FLOOR")
print("=" * 70)

floor_results = {}
for arch in ["BaselineCNN", "ResNet18_1ch"]:
    print(f"\n  Architecture: {arch}")
    set_seed(SEED)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_accs, fold_bals = [], []
    best_mf1, best_state = -1, None

    for fold, (tr_idx, va_idx) in enumerate(skf.split(paths, labels), 1):
        real_tr = [(paths[i], int(labels[i])) for i in tr_idx]
        real_va = [(paths[i], int(labels[i])) for i in va_idx]
        tr_loader = make_loader(real_tr, shuffle=True, drop_last=False)
        va_loader = make_loader(real_va)
        model = create_model(arch); crit = nn.CrossEntropyLoss()
        opt = optim.Adam(model.parameters(), lr=LR)
        bfm, bfs = -1, None
        for epoch in range(EPOCHS):
            run_epoch(model, tr_loader, crit, opt)
            _, _, va_mf1 = run_epoch(model, va_loader, crit)
            if va_mf1 > bfm:
                bfm = va_mf1
                _raw = get_raw(model)
                bfs = {k: v.detach().cpu().clone() for k, v in _raw.state_dict().items()}
        get_raw(model).load_state_dict(bfs)
        yt, yp = evaluate_model(model, va_loader)
        m = full_metrics(yt, yp)
        fold_accs.append(m["accuracy"]); fold_bals.append(m["balanced_accuracy"])
        if bfm > best_mf1: best_mf1 = bfm; best_state = bfs
        del model, opt, crit; torch.cuda.empty_cache()
        print(f"    Fold {fold}: acc={m['accuracy']:.4f}  bal={m['balanced_accuracy']:.4f}")

    floor_results[arch] = {
        "fold_accs": fold_accs, "fold_bals": fold_bals,
        "acc_mean": float(np.mean(fold_accs)), "acc_std": float(np.std(fold_accs)),
        "bal_mean": float(np.mean(fold_bals)), "bal_std": float(np.std(fold_bals)),
        "best_state": best_state,
    }
    print(f"  → {arch} FLOOR: acc={np.mean(fold_accs):.4f}±{np.std(fold_accs):.4f} "
          f"bal={np.mean(fold_bals):.4f}±{np.std(fold_bals):.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# FULL SWEEP: ALL 10 CONFIGURATIONS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("FULL SWEEP: ALL 10 CONFIGURATIONS")
print("=" * 70)

sweep_results = []
saved_states  = {}

for cfg in SWEEP_CONFIGS:
    arch  = cfg["arch"]
    synth = cfg["synth"]
    label = f"{arch}_s{synth}"
    print(f"\n{'#'*70}")
    print(f"CONFIG: {label}  (synth={synth}/structured class)")
    print(f"{'#'*70}")

    best_state, fold_metrics_list, fold_accs, fold_bals = run_one_config(arch, synth)
    saved_states[label] = {"arch": arch, "state": best_state}

    acc_mean = float(np.mean(fold_accs)); acc_std = float(np.std(fold_accs))
    bal_mean = float(np.mean(fold_bals)); bal_std = float(np.std(fold_bals))
    sr_mean  = float(np.mean([m["structured_recall"] for m in fold_metrics_list]))

    sweep_results.append({
        "label": label, "arch": arch, "synth": synth,
        "acc_mean": acc_mean, "acc_std": acc_std,
        "bal_mean": bal_mean, "bal_std": bal_std,
        "struct_recall_mean": sr_mean,
        "fold_accs": fold_accs, "fold_bals": fold_bals,
    })
    print(f"\n  → {label}: acc={acc_mean:.4f}±{acc_std:.4f}  "
          f"bal={bal_mean:.4f}±{bal_std:.4f}  struct={sr_mean:.4f}")

sweep_df = pd.DataFrame([{k: v for k, v in r.items()
                           if k not in ("fold_accs", "fold_bals")}
                          for r in sweep_results])
sweep_df.to_csv(OUT_DIR / "sweep_results.csv", index=False)

sweep_sorted   = sorted(sweep_results, key=lambda r: r["bal_mean"], reverse=True)
best_overall   = sweep_sorted[0]
top_n_configs  = sweep_sorted[:TOP_N_FOR_MIXED_TEST]

print("\n" + "=" * 70)
print("SWEEP COMPLETE — RESULTS RANKED BY BALANCED ACCURACY")
print("=" * 70)
print(f"\n{'Rank':4s} {'Config':22s} {'Acc Mean':>10} {'Acc Std':>9} "
      f"{'Bal Acc':>9} {'Bal Std':>9} {'Struct':>8}")
print("-" * 77)
for i, r in enumerate(sweep_sorted, 1):
    flag = " ← BEST" if i == 1 else (f" ← top {i}" if i <= TOP_N_FOR_MIXED_TEST else "")
    print(f"{i:4d} {r['label']:22s} {r['acc_mean']:>10.4f} {r['acc_std']:>9.4f} "
          f"{r['bal_mean']:>9.4f} {r['bal_std']:>9.4f} {r['struct_recall_mean']:>8.4f}{flag}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK A: MIXED TEST EVALUATION + CONFUSION MATRICES
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK A: MIXED TEST SET — TOP {TOP_N_FOR_MIXED_TEST} CONFIGS + CONFUSION MATRICES")
print("=" * 70)

mixed_test_rows = []
try:
    for r in top_n_configs:
        label   = r["label"]
        arch    = r["arch"]
        cfg_dir = OUT_DIR / label
        cfg_dir.mkdir(exist_ok=True)

        model = create_model(arch)
        get_raw(model).load_state_dict(saved_states[label]["state"])
        model.eval()

        for test_name, test_data in [
            ("real_only",  real_test_set),
            ("synth_only", synth_test_set),
            ("mixed",      mixed_test_set),
        ]:
            loader = make_loader(test_data)
            yt, yp = evaluate_model(model, loader)
            m = full_metrics(yt, yp, f"{label} | {test_name} (n={len(test_data)})")

            report = classification_report(yt, yp, labels=list(range(NUM_CLASSES)),
                                           target_names=classes, zero_division=0)
            (cfg_dir / f"test_{test_name}_report.txt").write_text(report)

            cm_path = cfg_dir / f"cm_{test_name}.png"
            save_cm(yt, yp, f"{label} | {test_name}", cm_path)
            confirm_saved(cm_path)

            mixed_test_rows.append({
                "config": label, "test_set": test_name, "n": len(test_data),
                "accuracy": m["accuracy"], "balanced_acc": m["balanced_accuracy"],
                "struct_recall": m["structured_recall"],
                **{f"recall_{classes[i]}": m["per_class_recall"][i]
                   for i in range(NUM_CLASSES)},
            })
        del model; torch.cuda.empty_cache()

    pd.DataFrame(mixed_test_rows).to_csv(OUT_DIR / "mixed_test_results.csv", index=False)
    print(f"\n  Block A complete — {len(mixed_test_rows)} rows saved to mixed_test_results.csv")

except Exception as e:
    print(f"\n  ✗ BLOCK A FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK B + C: t-SNE AND UMAP
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK B+C: t-SNE / UMAP — Best model: {best_overall['label']}")
print("=" * 70)

COLORS = ["#455A64","#00796B","#E53935","#6A1B9A",
          "#F57C00","#1E2761","#2E7D32","#795548"]

try:
    best_arch = best_overall["arch"]
    emb_model = create_model(best_arch)
    emb_raw   = get_raw(emb_model)
    emb_raw.load_state_dict(saved_states[best_overall["label"]]["state"])
    emb_raw.eval()

    feats_collect = []

    # FIX: Use a layer that reliably outputs a 2D (batch, features) tensor for both archs
    if best_arch == "BaselineCNN":
        # classifier[1] is Linear(64*12*24, 128) — after Flatten, before ReLU
        # hook on classifier[2] (ReLU) gives (batch, 128) — confirmed 2D
        hook_layer = emb_raw.classifier[2]  # ReLU after first linear
    else:
        # For ResNet18, hook on avgpool gives (batch, 512, 1, 1)
        # We handle 4D → 2D in the reshape step below
        hook_layer = emb_raw.avgpool

    def collect_feat(m, inp, out):
        # Flatten to 2D before appending — handles both (batch,128) and (batch,512,1,1)
        f = out.detach().cpu()
        if f.ndim > 2:
            f = f.reshape(f.shape[0], -1)
        feats_collect.append(f)

    hook = hook_layer.register_forward_hook(collect_feat)

    print("  Generating embedding dataset...")
    embed_synth  = generate_synth(EMBED_SYNTH_N, SEED+1, struct_only=True, tag="embed")
    all_triplets = ([(p, y, True) for p, y in real_test_set] +
                    [(p, y, False) for p, y in embed_synth])
    print(f"  Total embedding samples: {len(all_triplets)} "
          f"({len(real_test_set)} real + {len(embed_synth)} synth)")

    Y_all, R_all = [], []
    loader_emb = DataLoader(TripletDS(all_triplets), batch_size=64,
                            shuffle=False, num_workers=NUM_WORKERS)
    with torch.no_grad():
        for imgs, yl, rl in loader_emb:
            imgs = imgs.to(device, non_blocking=True)
            emb_raw(imgs)
            Y_all.extend(yl.tolist())
            R_all.extend(rl.tolist())

    hook.remove()
    del emb_model, emb_raw; torch.cuda.empty_cache()

    # Concatenate — all tensors are already 2D after collect_feat
    F_all = torch.cat(feats_collect, 0).numpy()
    Y_all = np.array(Y_all)
    R_all = np.array(R_all)
    print(f"  Embedding matrix shape: {F_all.shape}  "
          f"({F_all.shape[0]} samples × {F_all.shape[1]} features)")

    def plot_embeddings(Z, Y, R, title, path):
        from matplotlib.lines import Line2D
        fig, ax = plt.subplots(figsize=(11, 9))
        for ci, cls in enumerate(classes):
            col = COLORS[ci % len(COLORS)]
            rm = (Y == ci) & (R == 1)
            sm = (Y == ci) & (R == 0)
            if rm.any(): ax.scatter(Z[rm,0], Z[rm,1], c=col, s=55, alpha=0.85, marker="o")
            if sm.any(): ax.scatter(Z[sm,0], Z[sm,1], c=col, s=28, alpha=0.50, marker="x")
        leg_cls  = [Line2D([0],[0], marker="o", color="w", markerfacecolor=COLORS[i],
                            markersize=9, label=classes[i]) for i in range(NUM_CLASSES)]
        leg_type = [Line2D([0],[0], marker="o", color="k", markersize=7, label="real (circles)"),
                    Line2D([0],[0], marker="x", color="k", markersize=7, label="synth (crosses)")]
        ax.legend(handles=leg_cls + leg_type, loc="best", fontsize=8, ncol=2)
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Component 1"); ax.set_ylabel("Component 2")
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.savefig(path, dpi=180); plt.close()

    # t-SNE
    print("  Running t-SNE (perplexity=30, n_iter=1000)...")
    Z_tsne = TSNE(n_components=2, perplexity=30, n_iter=1000,
                  random_state=SEED, n_jobs=-1).fit_transform(F_all)
    tsne_path = OUT_DIR / "tsne_embeddings.png"
    plot_embeddings(Z_tsne, Y_all, R_all,
                    f"t-SNE — {best_overall['label']}", tsne_path)
    confirm_saved(tsne_path)

    # UMAP
    if HAS_UMAP:
        print("  Running UMAP (n_neighbors=15, min_dist=0.1)...")
        Z_umap = umap_lib.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                                random_state=SEED, n_jobs=-1).fit_transform(F_all)
        umap_path = OUT_DIR / "umap_embeddings.png"
        plot_embeddings(Z_umap, Y_all, R_all,
                        f"UMAP — {best_overall['label']}", umap_path)
        confirm_saved(umap_path)
    else:
        print("  UMAP skipped (umap-learn not available)")

    print(f"\n  Block B+C complete.")

except Exception as e:
    print(f"\n  ✗ BLOCK B+C FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK D: GRAD-CAM
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK D: GRAD-CAM — Using best BaselineCNN (features[6] = Conv 32→64)")
print("=" * 70)

try:
    # Always use best BaselineCNN for Grad-CAM (features[6] is the target layer)
    gc_label = next(r["label"] for r in sweep_sorted if r["arch"] == "BaselineCNN")
    print(f"  Grad-CAM model: {gc_label}")

    gc_model = create_model("BaselineCNN")
    gc_raw   = get_raw(gc_model)
    gc_raw.load_state_dict(saved_states[gc_label]["state"])
    gc_raw.eval()

    class GradCAM:
        def __init__(self, model, target_layer):
            self.model = model
            self.grads = None
            self.acts  = None
            self._h    = []
            self._h.append(target_layer.register_forward_hook(
                lambda m, i, o: setattr(self, "acts", o)))
            self._h.append(target_layer.register_full_backward_hook(
                lambda m, gi, go: setattr(self, "grads", go[0])))

        def compute(self, img_t, cls_idx):
            # FIX: Use torch.enable_grad() explicitly — model is in eval() but we
            # need gradients for backward pass
            with torch.enable_grad():
                self.model.zero_grad()
                out = self.model(img_t)
                out[0, cls_idx].backward()
            w   = self.grads.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((w * self.acts).sum(dim=1, keepdim=True))
            cam = F.interpolate(cam, size=(H, W), mode="bilinear", align_corners=False)
            cam = cam.squeeze().detach().cpu().numpy()
            return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), out.argmax(1).item()

        def remove(self):
            for h in self._h: h.remove()

    gc = GradCAM(gc_raw, gc_raw.features[6])

    correct_items, incorrect_items = [], []
    all_pil = [Image.open(p).convert("L") for p, _ in real_test_set]

    for pil_img, true_lbl in zip(all_pil, labels):
        tensor = t1ch(pil_img).unsqueeze(0).to(device)
        cam_map, pred = gc.compute(tensor, int(true_lbl))
        raw_arr = np.array(pil_img.resize((W, H)))
        item = (raw_arr, int(true_lbl), pred, cam_map)
        if pred == int(true_lbl):
            correct_items.append(item)
        else:
            incorrect_items.append(item)

    gc.remove()
    del gc_model, gc_raw; torch.cuda.empty_cache()

    print(f"  Correct predictions:   {len(correct_items)}")
    print(f"  Incorrect predictions: {len(incorrect_items)}")

    def save_gradcam_panel(items, n, title, path):
        items = items[:n]
        if not items:
            print(f"  WARNING: no items for {title}, skipping panel")
            return
        cols = 4
        rows = (len(items) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols*2, figsize=(cols*4, rows*2.6))
        axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
        for k, (raw_img, true_lbl, pred_lbl, cam_map) in enumerate(items):
            ax_img = axes[k*2]; ax_cam = axes[k*2+1]
            ax_img.imshow(raw_img, cmap="gray", aspect="auto")
            ax_img.set_title(f"True: {classes[true_lbl]}", fontsize=7)
            ax_img.axis("off")
            ax_cam.imshow(raw_img, cmap="gray", alpha=0.55, aspect="auto")
            ax_cam.imshow(cam_map, cmap="jet",  alpha=0.55, aspect="auto", vmin=0, vmax=1)
            col = "green" if pred_lbl == true_lbl else "red"
            ax_cam.set_title(f"Pred: {classes[pred_lbl]}", fontsize=7, color=col)
            ax_cam.axis("off")
        for j in range(len(items)*2, len(axes)):
            axes[j].axis("off")
        plt.suptitle(title, fontsize=11, fontweight="bold")
        plt.tight_layout()
        plt.savefig(path, dpi=160); plt.close()

    gc_correct_path   = OUT_DIR / "gradcam_correct.png"
    gc_incorrect_path = OUT_DIR / "gradcam_incorrect.png"

    save_gradcam_panel(correct_items,   GRADCAM_N_CORRECT,
                       f"Grad-CAM — Correct ({gc_label})",   gc_correct_path)
    confirm_saved(gc_correct_path)

    save_gradcam_panel(incorrect_items, GRADCAM_N_INCORRECT,
                       f"Grad-CAM — Incorrect ({gc_label})", gc_incorrect_path)
    confirm_saved(gc_incorrect_path)

    print(f"\n  Block D complete.")

except Exception as e:
    print(f"\n  ✗ BLOCK D FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK E: PERTURBATION ROBUSTNESS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"BLOCK E: PERTURBATION ROBUSTNESS — Best model: {best_overall['label']}")
print("=" * 70)

try:
    def apply_perturbation(img_tensor, ptype):
        arr = img_tensor.clone().squeeze().numpy()
        if ptype == "clean":
            pass
        elif ptype == "translation":
            arr = np.roll(arr, shift=3, axis=1)
        elif ptype == "rotation":
            if HAS_SCIPY_NDIMAGE:
                arr = np.clip(ndrotate(arr, angle=3, reshape=False, mode="nearest"), 0, 1)
        elif ptype == "threshold_shift":
            arr = (arr > 0.55).astype(np.float32)
        elif ptype == "salt_pepper":
            rng_p = np.random.default_rng(SEED)
            noise_mask = rng_p.random(arr.shape) < 0.02
            arr = arr.copy()
            arr[noise_mask] = 1 - arr[noise_mask]
        elif ptype == "morphological_erosion":
            if HAS_SCIPY_NDIMAGE:
                arr = binary_erosion(
                    (arr > 0.5).astype(np.uint8), iterations=1).astype(np.float32)
        elif ptype == "morphological_dilation":
            if HAS_SCIPY_NDIMAGE:
                arr = binary_dilation(
                    (arr > 0.5).astype(np.uint8), iterations=1).astype(np.float32)
        return torch.tensor(arr).unsqueeze(0).unsqueeze(0).float()

    PERTURBATIONS = [
        "clean", "translation", "rotation", "threshold_shift",
        "salt_pepper", "morphological_erosion", "morphological_dilation",
    ]

    p_arch  = best_overall["arch"]
    p_label = best_overall["label"]
    p_model = create_model(p_arch)
    p_raw   = get_raw(p_model)
    p_raw.load_state_dict(saved_states[p_label]["state"])
    p_raw.eval()

    skf_p = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    pert_fold_bals = defaultdict(list)

    for fold, (_, va_idx) in enumerate(skf_p.split(paths, labels), 1):
        real_va     = [(paths[i], int(labels[i])) for i in va_idx]
        imgs_loaded = [t1ch(Image.open(p).convert("L")) for p, _ in real_va]
        true_labels = [y for _, y in real_va]

        for ptype in PERTURBATIONS:
            yt, yp = [], []
            for img_t, ty in zip(imgs_loaded, true_labels):
                pt = apply_perturbation(img_t.unsqueeze(0), ptype).to(device)
                with torch.no_grad():
                    pred = p_raw(pt).argmax(1).item()
                yt.append(ty); yp.append(pred)
            pert_fold_bals[ptype].append(balanced_accuracy_score(yt, yp))

    del p_model, p_raw; torch.cuda.empty_cache()

    clean_mean = np.mean(pert_fold_bals["clean"])
    pert_rows  = []
    for ptype in PERTURBATIONS:
        m, s = np.mean(pert_fold_bals[ptype]), np.std(pert_fold_bals[ptype])
        pert_rows.append({
            "perturbation": ptype,
            "bal_acc_mean": round(m, 4),
            "bal_acc_std":  round(s, 4),
            "drop":         round(clean_mean - m, 4),
        })
        print(f"  {ptype:28s}: bal={m:.4f}±{s:.4f}  drop={clean_mean-m:+.4f}")

    pd.DataFrame(pert_rows).to_csv(OUT_DIR / "perturbation_results.csv", index=False)

    # FIX: ylim starts at 0.0 (not 0.4) so erosion/dilation bars at 0.14/0.24 are visible
    fig, ax = plt.subplots(figsize=(13, 6))
    names = [r["perturbation"] for r in pert_rows]
    means = [r["bal_acc_mean"] for r in pert_rows]
    stds  = [r["bal_acc_std"]  for r in pert_rows]

    bar_colors = []
    for n in names:
        if n == "clean":
            bar_colors.append("#2E7D32")
        elif n in ["morphological_erosion", "morphological_dilation", "salt_pepper"]:
            bar_colors.append("#E53935")
        else:
            bar_colors.append("#1E2761")

    bars = ax.bar(range(len(names)), means, yerr=stds, capsize=4,
                  color=bar_colors, edgecolor="white", linewidth=0.5, alpha=0.9)

    # Add value labels on top of each bar
    for bar, m_val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f"{m_val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.axhline(clean_mean, color="gray", ls="--", lw=1.5,
               label=f"Clean baseline ({clean_mean:.4f})")
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=25, ha="right", fontsize=10)
    ax.set_ylabel("Balanced Accuracy", fontsize=11)
    ax.set_ylim(0.0, 1.10)   # FIX: was 0.4, now 0.0 — all bars fully visible
    ax.set_title(f"Perturbation Robustness — {p_label}\n"
                 f"Green = baseline  |  Blue = robust  |  Red = sensitive",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()

    pert_path = OUT_DIR / "perturbation_robustness.png"
    plt.savefig(pert_path, dpi=180); plt.close()
    confirm_saved(pert_path)

    print(f"\n  Block E complete.")

except Exception as e:
    print(f"\n  ✗ BLOCK E FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK F: WILCOXON STATISTICAL TEST
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK F: WILCOXON SIGNED-RANK TEST")
print("=" * 70)

try:
    best_cnn_r = next((r for r in sweep_sorted if r["arch"] == "BaselineCNN"), None)
    best_r18_r = next((r for r in sweep_sorted if r["arch"] == "ResNet18_1ch"), None)

    wilcoxon_txt = []
    if best_cnn_r and best_r18_r:
        cnn_f = best_cnn_r["fold_accs"]
        r18_f = best_r18_r["fold_accs"]
        wilcoxon_txt.append(f"Best CNN: {best_cnn_r['label']}  folds={cnn_f}  mean={np.mean(cnn_f):.4f}")
        wilcoxon_txt.append(f"Best R18: {best_r18_r['label']}  folds={r18_f}  mean={np.mean(r18_f):.4f}")
        print(f"  CNN folds: {[round(f,4) for f in cnn_f]}  mean={np.mean(cnn_f):.4f}")
        print(f"  R18 folds: {[round(f,4) for f in r18_f]}  mean={np.mean(r18_f):.4f}")

        if HAS_SCIPY:
            try:
                stat, p_val = scipy_wilcoxon(cnn_f, r18_f)
                sig = p_val < 0.05
                result_str = f"Wilcoxon W={stat:.4f}  p={p_val:.4f}  significant={sig}"
                wilcoxon_txt.append(result_str)
                print(f"  {result_str}")
                if not sig:
                    print("  → No significant difference between architectures (p ≥ 0.05)")
                    print("  → Both architectures are statistically equivalent on this dataset")
                else:
                    print("  → Significant difference detected (p < 0.05)")
            except Exception as e:
                msg = f"Wilcoxon failed: {e}"
                wilcoxon_txt.append(msg)
                print(f"  {msg}")
        else:
            msg = "scipy not available — Wilcoxon test skipped"
            wilcoxon_txt.append(msg)
            print(f"  {msg}")

    wil_path = OUT_DIR / "statistical_test.txt"
    wil_path.write_text("\n".join(wilcoxon_txt))
    confirm_saved(wil_path)

    print(f"\n  Block F complete.")

except Exception as e:
    print(f"\n  ✗ BLOCK F FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK G: MODEL EFFICIENCY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK G: MODEL EFFICIENCY")
print("=" * 70)

try:
    dummy = torch.zeros(1, 1, H, W).to(device)
    eff_rows = []
    for arch in ["BaselineCNN", "ResNet18_1ch"]:
        m   = create_model(arch)
        raw = get_raw(m)
        params = sum(p.numel() for p in raw.parameters())
        tmp = WORK_DIR / f"tmp_{arch}.pt"
        torch.save(raw.state_dict(), tmp)
        size_mb = os.path.getsize(tmp) / 1e6
        tmp.unlink()
        m.eval()
        with torch.no_grad():
            for _ in range(10): m(dummy)  # warmup
        times = []
        with torch.no_grad():
            for _ in range(100):
                t0 = time.perf_counter()
                m(dummy)
                if device.type == "cuda": torch.cuda.synchronize()
                times.append((time.perf_counter() - t0) * 1000)
        eff_rows.append({
            "architecture":      arch,
            "parameters":        params,
            "size_mb":           round(size_mb, 2),
            "inference_mean_ms": round(float(np.mean(times)), 3),
            "inference_std_ms":  round(float(np.std(times)),  3),
            "meets_target":      size_mb <= 10.0,
        })
        print(f"  {arch}: {params:,} params  {size_mb:.2f} MB  "
              f"{np.mean(times):.2f}±{np.std(times):.2f} ms")
        del m; torch.cuda.empty_cache()

    eff_path = OUT_DIR / "efficiency.csv"
    pd.DataFrame(eff_rows).to_csv(eff_path, index=False)
    confirm_saved(eff_path)
    print(f"\n  Block G complete.")

except Exception as e:
    print(f"\n  ✗ BLOCK G FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK H: SWEEP CHARTS WITH FLOOR LINES
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK H: SWEEP CHARTS")
print("=" * 70)

try:
    synth_amts = [250, 500, 750, 1000, 1250]

    def get_vals(arch, metric):
        return [next(r[metric] for r in sweep_results
                     if r["arch"] == arch and r["synth"] == s)
                for s in synth_amts]

    cnn_acc = get_vals("BaselineCNN",  "acc_mean")
    r18_acc = get_vals("ResNet18_1ch", "acc_mean")
    cnn_bal = get_vals("BaselineCNN",  "bal_mean")
    r18_bal = get_vals("ResNet18_1ch", "bal_mean")
    cnn_acc_std = get_vals("BaselineCNN",  "acc_std")
    r18_acc_std = get_vals("ResNet18_1ch", "acc_std")
    cnn_bal_std = get_vals("BaselineCNN",  "bal_std")
    r18_bal_std = get_vals("ResNet18_1ch", "bal_std")

    fl_cnn_acc = floor_results["BaselineCNN"]["acc_mean"]
    fl_r18_acc = floor_results["ResNet18_1ch"]["acc_mean"]
    fl_cnn_bal = floor_results["BaselineCNN"]["bal_mean"]
    fl_r18_bal = floor_results["ResNet18_1ch"]["bal_mean"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (cy, ry, cstd, rstd, fl_c, fl_r, ylabel, title) in zip(axes, [
        (cnn_acc, r18_acc, cnn_acc_std, r18_acc_std, fl_cnn_acc, fl_r18_acc,
         "Accuracy (= Micro F1)", "Synth Sweep — Accuracy"),
        (cnn_bal, r18_bal, cnn_bal_std, r18_bal_std, fl_cnn_bal, fl_r18_bal,
         "Balanced Accuracy",     "Synth Sweep — Balanced Accuracy"),
    ]):
        ax.errorbar(synth_amts, cy, yerr=cstd, fmt="o-", color="#00796B",
                    lw=2, capsize=4, label="BaselineCNN")
        ax.errorbar(synth_amts, ry, yerr=rstd, fmt="s-", color="#1E2761",
                    lw=2, capsize=4, label="ResNet18_1ch")
        ax.axhline(fl_c, color="#00796B", ls="--", lw=1.5, alpha=0.7,
                   label=f"CNN real-only floor ({fl_c:.3f})")
        ax.axhline(fl_r, color="#1E2761", ls=":",  lw=1.5, alpha=0.7,
                   label=f"R18 real-only floor ({fl_r:.3f})")
        ax.set_xlabel("Synthetic images per structured class", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(title, fontweight="bold", fontsize=12)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(synth_amts)

    plt.suptitle("Synthetic Augmentation Sweep", fontsize=13, fontweight="bold")
    plt.tight_layout()
    sweep_path = OUT_DIR / "sweep_charts_with_floor.png"
    plt.savefig(sweep_path, dpi=180); plt.close()
    confirm_saved(sweep_path)
    print(f"\n  Block H complete.")

except Exception as e:
    print(f"\n  ✗ BLOCK H FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK I: SYNTHETIC EXAMPLES PANEL
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK I: SYNTHETIC EXAMPLES PANEL")
print("=" * 70)

try:
    rng_ex = np.random.default_rng(2025)
    n_var  = 3
    fig, axes = plt.subplots(NUM_CLASSES, n_var, figsize=(n_var*2.2, NUM_CLASSES*1.4))
    for row, cls in enumerate(classes):
        for col in range(n_var):
            arr = GEN_FUNCS[cls](rng_ex)
            axes[row, col].imshow(arr, cmap="gray", interpolation="nearest", aspect="auto")
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_ylabel(cls, fontsize=9, rotation=0,
                                           ha="right", va="center", labelpad=50)
    plt.suptitle("Synthetic Examples — 3 Random Variants per Class",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    synth_ex_path = OUT_DIR / "synthetic_examples.png"
    plt.savefig(synth_ex_path, dpi=160); plt.close()
    confirm_saved(synth_ex_path)
    print(f"\n  Block I complete.")

except Exception as e:
    print(f"\n  ✗ BLOCK I FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK J: SCATTERPLOT (Efficiency vs Performance)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK J: EFFICIENCY SCATTERPLOT")
print("=" * 70)

try:
    eff_df = pd.read_csv(OUT_DIR / "efficiency.csv")
    best_per_arch = {
        arch: next(r for r in sweep_sorted if r["arch"] == arch)
        for arch in ["BaselineCNN", "ResNet18_1ch"]
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors_s = ["#00796B", "#1E2761"]

    for ax, (y_key, ylabel, title) in zip(axes, [
        ("acc_mean", "Accuracy (Micro F1)", "Model Complexity vs Accuracy"),
        ("bal_mean", "Balanced Accuracy",   "Model Complexity vs Balanced Accuracy"),
    ]):
        for arch, col in zip(["BaselineCNN", "ResNet18_1ch"], colors_s):
            r   = best_per_arch[arch]
            row = eff_df[eff_df["architecture"] == arch].iloc[0]
            ax.scatter(row.parameters/1e6, r[y_key], s=300, c=col, zorder=5, label=arch)
            ax.annotate(
                f"{arch}\n{row.parameters/1e6:.1f}M  {row.size_mb:.1f}MB\n"
                f"{row.inference_mean_ms:.1f}ms",
                (row.parameters/1e6, r[y_key]),
                xytext=(15, -40), textcoords="offset points", fontsize=9, color=col,
                arrowprops=dict(arrowstyle="->", color=col, lw=1.5))
        ax.set_xlabel("Parameters (millions)", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(title, fontweight="bold", fontsize=12)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0.85, 1.01); ax.set_xlim(0, 15)

    plt.suptitle("Efficiency vs Performance", fontsize=12, fontweight="bold")
    plt.tight_layout()
    scatter_path = OUT_DIR / "scatterplot_params_vs_accuracy.png"
    plt.savefig(scatter_path, dpi=180); plt.close()
    confirm_saved(scatter_path)
    print(f"\n  Block J complete.")

except Exception as e:
    print(f"\n  ✗ BLOCK J FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK K: THESIS SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("BLOCK K: THESIS SUMMARY")
print("=" * 70)

try:
    summary_txt = [
        "=" * 80,
        "THESIS EVALUATION PIPELINE — COMPLETE RESULTS",
        "UHasselt DSI × Gpixel — Nuthi Raviteja Pediredla",
        "=" * 80, "",
        f"DATASET CONFIGURATION:",
        f"  TRASH (0-EMPTY + 1-NOISE combined, downsampled): {TRASH_DOWNSAMPLE_N} real images",
        f"  Structured classes: {len(STRUCT_FOLDERS)}  ({', '.join(STRUCT_FOLDERS)})",
        f"  Total real samples: {len(real_test_set)}",
        f"  Total classes: {NUM_CLASSES}", "",
        "REAL-ONLY FLOOR:",
        f"  BaselineCNN:  acc={floor_results['BaselineCNN']['acc_mean']:.4f}  "
        f"bal={floor_results['BaselineCNN']['bal_mean']:.4f}",
        f"  ResNet18_1ch: acc={floor_results['ResNet18_1ch']['acc_mean']:.4f}  "
        f"bal={floor_results['ResNet18_1ch']['bal_mean']:.4f}", "",
        "SWEEP RESULTS (ranked by balanced accuracy):",
    ]
    for i, r in enumerate(sweep_sorted, 1):
        summary_txt.append(
            f"  {i:2d}. {r['label']:22s}  "
            f"acc={r['acc_mean']:.4f}±{r['acc_std']:.4f}  "
            f"bal={r['bal_mean']:.4f}±{r['bal_std']:.4f}  "
            f"struct={r['struct_recall_mean']:.4f}")
    summary_txt += [
        "",
        f"BEST OVERALL: {best_overall['label']}",
        f"  Balanced Accuracy: {best_overall['bal_mean']:.4f} ± {best_overall['bal_std']:.4f}",
        f"  Accuracy:          {best_overall['acc_mean']:.4f} ± {best_overall['acc_std']:.4f}",
        f"  Struct Recall:     {best_overall['struct_recall_mean']:.4f}", "",
        "=" * 80,
    ]

    summary_path = OUT_DIR / "SUMMARY.txt"
    summary_path.write_text("\n".join(summary_txt))
    confirm_saved(summary_path)

    # Print top results to stdout
    print(f"\n  BEST MODEL: {best_overall['label']}")
    print(f"  Bal Acc: {best_overall['bal_mean']:.4f} ± {best_overall['bal_std']:.4f}")
    print(f"  Accuracy: {best_overall['acc_mean']:.4f} ± {best_overall['acc_std']:.4f}")

except Exception as e:
    print(f"\n  ✗ BLOCK K FAILED: {e}")
    import traceback; traceback.print_exc()

# ══════════════════════════════════════════════════════════════════════════════
# FINAL BLOCK: ZIP ALL OUTPUTS AND PRINT COMPLETE FILE MANIFEST
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("FINAL BLOCK: ZIPPING ALL OUTPUTS")
print("=" * 70)

zip_path = WORK_DIR / "thesis_outputs.zip"
all_output_files = []

try:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUT_DIR.rglob("*")):
            if f.is_file():
                arcname = f.relative_to(WORK_DIR)
                zf.write(f, arcname)
                all_output_files.append(f)

    zip_size_mb = zip_path.stat().st_size / 1e6
    print(f"\n  ZIP created: {zip_path}")
    print(f"  ZIP size:    {zip_size_mb:.1f} MB")
    print(f"  Files zipped: {len(all_output_files)}")

except Exception as e:
    print(f"  ✗ ZIP FAILED: {e}")

# Print complete manifest
print("\n" + "=" * 70)
print("COMPLETE OUTPUT FILE MANIFEST")
print("=" * 70)

# Key images first
key_images = [
    "tsne_embeddings.png",
    "umap_embeddings.png",
    "gradcam_correct.png",
    "gradcam_incorrect.png",
    "perturbation_robustness.png",
    "sweep_charts_with_floor.png",
    "synthetic_examples.png",
    "scatterplot_params_vs_accuracy.png",
]

print("\n  ── KEY IMAGES ──")
for fname in key_images:
    p = OUT_DIR / fname
    if p.exists():
        print(f"  ✓  {fname:45s}  {p.stat().st_size/1024:.1f} KB")
    else:
        print(f"  ✗  {fname:45s}  MISSING")

print("\n  ── CONFUSION MATRICES (per top config, per test set) ──")
for r in top_n_configs:
    for test_name in ["real_only", "synth_only", "mixed"]:
        p = OUT_DIR / r["label"] / f"cm_{test_name}.png"
        if p.exists():
            print(f"  ✓  {r['label']}/cm_{test_name}.png  ({p.stat().st_size/1024:.1f} KB)")
        else:
            print(f"  ✗  {r['label']}/cm_{test_name}.png  MISSING")

print("\n  ── CSV / TEXT FILES ──")
for fname in ["sweep_results.csv", "mixed_test_results.csv", "perturbation_results.csv",
              "efficiency.csv", "statistical_test.txt", "SUMMARY.txt"]:
    p = OUT_DIR / fname
    if p.exists():
        print(f"  ✓  {fname:45s}  {p.stat().st_size/1024:.1f} KB")
    else:
        print(f"  ✗  {fname:45s}  MISSING")

print("\n" + "=" * 70)
print("PIPELINE COMPLETE ✓")
print(f"Download your results: {zip_path}")
print("=" * 70)

2026-06-08 06:21:19.854296: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780899680.115786      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780899680.183670      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780899680.753304      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780899680.753346      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780899680.753349      57 computation_placer.cc:177] computation placer alr

Device: cuda | GPUs: 2
Output directory: /kaggle/working/thesis_pipeline

STEP 1: TRASH CLASS DOWNSAMPLING
Target: exactly 10 images from combined NOISE+EMPTY
Total NOISE + EMPTY images: 1058
Selected 10 images for TRASH class

8-class setup:
  0: 0-TRASH
  1: 2-ZEBRA
  2: 3-OKAPI
  3: 4-CHEETAH
  4: 5-GRID
  5: 6-DIAGONAL
  6: 7-BLINDS
  7: 8-MORSE

Final dataset: 98 total images
  0-TRASH             :    10  ( 10.2%)
  2-ZEBRA             :    11  ( 11.2%)
  3-OKAPI             :    10  ( 10.2%)
  4-CHEETAH           :    14  ( 14.3%)
  5-GRID              :    23  ( 23.5%)
  6-DIAGONAL          :    10  ( 10.2%)
  7-BLINDS            :     9  (  9.2%)
  8-MORSE             :    11  ( 11.2%)

GENERATING MIXED TEST SET
  100 new synthetic images per structured class (seed+999)
  Real-only:     98  (10 TRASH + 88 structured)
  Synth-only:   700  (100/class × 7 structured)
  Mixed:        798

BLOCK 0: REAL-ONLY FLOOR

  Architecture: BaselineCNN
    Fold 1: acc=0.6400  bal=0.6146
    

In [1]:
# ============================================================
# THESIS FIGURE GENERATOR
# Generates two publication-quality figures for the thesis:
#   1. real_defect_examples.png  — panel of real defect maps
#   2. pipeline_diagram.png      — methodology pipeline flowchart
#
# Run this in a Kaggle notebook cell AFTER your main pipeline.
# Output saved to /kaggle/working/
# ============================================================

import os
import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from pathlib import Path
from PIL import Image

# ── Paths ───────────────────────────────────────────────────
DATA_ROOT = Path("/kaggle/input/datasets/teja19129/g-pixel/96x192")
OUT_DIR   = Path("/kaggle/working")

# ── Class map (matches your pipeline exactly) ───────────────
CLASS_MAP = {
    "0-TRASH":    {"folders": ["0-EMPTY", "1-NOISE"], "label": "TRASH\n(NOISE+EMPTY)"},
    "2-ZEBRA":    {"folders": ["2-ZEBRA"],             "label": "ZEBRA"},
    "3-OKAPI":    {"folders": ["3-OKAPI"],             "label": "OKAPI"},
    "4-CHEETAH":  {"folders": ["4-CHEETAH"],           "label": "CHEETAH"},
    "5-GRID":     {"folders": ["5-GRID"],              "label": "GRID"},
    "6-DIAGONAL": {"folders": ["6-DIAGONAL"],          "label": "DIAGONAL"},
    "7-BLINDS":   {"folders": ["7-BLINDS"],            "label": "BLINDS"},
    "8-MORSE":    {"folders": ["8-MORSE"],             "label": "MORSE"},
}

VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ── Colour palette (matches thesis uhblue / teal) ───────────
UHBLUE  = "#1E2761"
TEAL    = "#00796B"
CORAL   = "#C62828"
AMBER   = "#E65100"
GRAY    = "#455A64"

# ════════════════════════════════════════════════════════════
# FIGURE 1 — real_defect_examples.png
# 2 rows × 4 cols, one real image per class
# ════════════════════════════════════════════════════════════

def pick_real_image(folders, seed=42):
    """Return a PIL image from one of the given class folders."""
    rng = random.Random(seed)
    candidates = []
    for folder in folders:
        d = DATA_ROOT / folder
        if d.exists():
            for p in sorted(d.iterdir()):
                if p.suffix.lower() in VALID_EXT:
                    candidates.append(p)
    if not candidates:
        return None
    return Image.open(rng.choice(candidates)).convert("L")


def make_real_defect_panel():
    classes  = list(CLASS_MAP.items())     # 8 entries
    n_cols, n_rows = 4, 2

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 3.2, n_rows * 2.4),
        facecolor="white",
    )
    fig.subplots_adjust(left=0.01, right=0.99, top=0.88, bottom=0.08,
                        wspace=0.08, hspace=0.30)

    for ax, (cls_key, info) in zip(axes.flatten(), classes):
        img = pick_real_image(info["folders"], seed=42)
        if img is not None:
            arr = np.array(img.resize((192, 96)))
            ax.imshow(arr, cmap="gray", interpolation="nearest",
                      aspect="auto", vmin=0, vmax=255)
        else:
            ax.set_facecolor("#eeeeee")
            ax.text(0.5, 0.5, "not found", ha="center", va="center",
                    transform=ax.transAxes, fontsize=10, color="red")

        # Styled label box at bottom of each subplot
        ax.set_xlabel(
            info["label"],
            fontsize=11, fontweight="bold",
            color=UHBLUE, labelpad=4,
        )
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(UHBLUE)
            spine.set_linewidth(1.2)

    # Super-title
    fig.suptitle(
        "Real $96\\times192$ binary defect maps — one example per class",
        fontsize=13, fontweight="bold", color=UHBLUE, y=0.97,
    )

    out = OUT_DIR / "real_defect_examples.png"
    fig.savefig(out, dpi=220, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"✓  Saved: {out}  ({out.stat().st_size/1024:.1f} KB)")
    return out


# ════════════════════════════════════════════════════════════
# FIGURE 2 — pipeline_diagram.png
# Clean flowchart of the full methodology pipeline
# ════════════════════════════════════════════════════════════

def draw_rounded_box(ax, x, y, w, h, label, sublabel=None,
                     fc=UHBLUE, ec=UHBLUE, tc="white", fontsize=11):
    """Draw a rounded rectangle with a centred label."""
    box = FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.04",
        facecolor=fc, edgecolor=ec, linewidth=1.4, zorder=3,
    )
    ax.add_patch(box)
    if sublabel:
        ax.text(x, y + 0.09, label, ha="center", va="center",
                fontsize=fontsize, fontweight="bold", color=tc, zorder=4)
        ax.text(x, y - 0.14, sublabel, ha="center", va="center",
                fontsize=fontsize - 2, color=tc, alpha=0.90, zorder=4)
    else:
        ax.text(x, y, label, ha="center", va="center",
                fontsize=fontsize, fontweight="bold", color=tc, zorder=4)


def draw_arrow(ax, x0, y0, x1, y1, label=None, color=GRAY):
    ax.annotate(
        "", xy=(x1, y1), xytext=(x0, y0),
        arrowprops=dict(
            arrowstyle="-|>", color=color,
            lw=1.6, mutation_scale=16,
        ),
        zorder=2,
    )
    if label:
        mx, my = (x0+x1)/2, (y0+y1)/2
        ax.text(mx + 0.03, my, label, ha="left", va="center",
                fontsize=9, color=color, style="italic")


def make_pipeline_diagram():
    fig, ax = plt.subplots(figsize=(14, 7), facecolor="white")
    ax.set_xlim(0, 14); ax.set_ylim(0, 7)
    ax.axis("off")
    fig.subplots_adjust(left=0.01, right=0.99, top=0.94, bottom=0.02)

    # ── Row 1: DATA SOURCES ──────────────────────────────────
    # Real dataset box
    draw_rounded_box(ax, 2.0, 5.8, 2.8, 0.90,
                     "Real dataset",
                     "98 images · 8 classes",
                     fc=UHBLUE)

    # Synthetic generator box
    draw_rounded_box(ax, 6.0, 5.8, 2.8, 0.90,
                     "Synthetic generator",
                     "Physics-based · 7 structured classes",
                     fc=TEAL)

    # ── Row 2: TRASH DOWNSAMPLING note ──────────────────────
    ax.text(2.0, 4.95, "Trash class downsampled\nto 10 real images",
            ha="center", va="center", fontsize=8.5,
            color=GRAY, style="italic",
            bbox=dict(boxstyle="round,pad=0.3", fc="#f4f7ff",
                      ec=GRAY, lw=0.8))

    # ── Row 2: SYNTH CONFIG box ──────────────────────────────
    ax.text(6.0, 4.95, "250 / 500 / 750 / 1000 / 1250\nimages per structured class",
            ha="center", va="center", fontsize=8.5,
            color=GRAY, style="italic",
            bbox=dict(boxstyle="round,pad=0.3", fc="#e8f5e9",
                      ec=TEAL, lw=0.8))

    # ── Arrows into merge ────────────────────────────────────
    draw_arrow(ax, 2.0, 5.35, 2.0, 4.35, color=UHBLUE)
    draw_arrow(ax, 6.0, 5.35, 6.0, 4.35, color=TEAL)

    # Lines converging to merge point
    # Real → merge
    ax.annotate("", xy=(4.0, 3.85), xytext=(2.0, 4.35),
                arrowprops=dict(arrowstyle="-", color=UHBLUE, lw=1.4))
    # Synth → merge
    ax.annotate("", xy=(4.0, 3.85), xytext=(6.0, 4.35),
                arrowprops=dict(arrowstyle="-|>", color=TEAL,
                                lw=1.4, mutation_scale=14))

    # ── Row 3: TRAINING SET ──────────────────────────────────
    draw_rounded_box(ax, 4.0, 3.35, 3.2, 0.90,
                     "Training set",
                     "Real (train fold) + Synthetic images",
                     fc="#37474F")

    # ── Arrow to CV ─────────────────────────────────────────
    draw_arrow(ax, 4.0, 2.90, 4.0, 2.35, color=GRAY)

    # ── Row 4: 4-FOLD CV ────────────────────────────────────
    draw_rounded_box(ax, 4.0, 1.90, 3.4, 0.80,
                     "4-fold StratifiedKFold CV",
                     "Validation: real images only",
                     fc=AMBER, ec=AMBER)

    # ── Arrow to models ─────────────────────────────────────
    draw_arrow(ax, 4.0, 1.50, 4.0, 1.00, color=GRAY)

    # ── Row 5: MODELS side by side ──────────────────────────
    draw_rounded_box(ax, 2.4, 0.55, 2.4, 0.72,
                     "BaselineCNN",
                     "2.38M params · 9.54 MB",
                     fc=TEAL, fontsize=10)
    draw_rounded_box(ax, 5.6, 0.55, 2.4, 0.72,
                     "ResNet18\\_1ch",
                     "11.17M params · 44.78 MB",
                     fc=UHBLUE, fontsize=10)

    # branching arrows
    ax.annotate("", xy=(2.4, 0.91), xytext=(3.7, 1.00),
                arrowprops=dict(arrowstyle="-|>", color=TEAL,
                                lw=1.4, mutation_scale=14))
    ax.annotate("", xy=(5.6, 0.91), xytext=(4.3, 1.00),
                arrowprops=dict(arrowstyle="-|>", color=UHBLUE,
                                lw=1.4, mutation_scale=14))

    # ── RIGHT SIDE: EVALUATION ──────────────────────────────
    # Real-only eval
    draw_rounded_box(ax, 10.5, 5.2, 2.8, 0.80,
                     "Real-only evaluation",
                     "98 images · all 8 classes",
                     fc="#5C6BC0", ec="#5C6BC0", fontsize=10)

    # Mixed test set
    draw_rounded_box(ax, 10.5, 3.8, 2.8, 0.80,
                     "Mixed test set",
                     "98 real + 700 new synthetic = 798",
                     fc=CORAL, ec=CORAL, fontsize=10)

    # Floor experiment
    draw_rounded_box(ax, 10.5, 2.4, 2.8, 0.80,
                     "Real-only floor",
                     "No synth · measures gain",
                     fc=GRAY, ec=GRAY, fontsize=10)

    # Metric box
    draw_rounded_box(ax, 10.5, 1.0, 2.8, 0.80,
                     "Metrics",
                     "Accuracy · Micro F1 · Bal Acc · Struct Recall",
                     fc="#263238", ec="#263238", fontsize=9)

    # Right-side arrows
    draw_arrow(ax, 10.5, 4.80, 10.5, 4.20, color=CORAL)
    draw_arrow(ax, 10.5, 3.40, 10.5, 2.80, color=GRAY)
    draw_arrow(ax, 10.5, 2.00, 10.5, 1.40, color="#263238")

    # Connector from CV block to right-side evals
    draw_arrow(ax, 5.70, 1.90, 9.10, 4.85,
               label="best model weights", color="#5C6BC0")
    draw_arrow(ax, 5.70, 1.90, 9.10, 3.80, color=CORAL)
    ax.annotate("", xy=(9.10, 2.40), xytext=(5.70, 1.90),
                arrowprops=dict(arrowstyle="-|>", color=GRAY,
                                lw=1.2, mutation_scale=13))

    # ── Statistical test box ────────────────────────────────
    ax.text(10.5, 0.22,
            "Wilcoxon test  W = 0.0  p = 0.500",
            ha="center", va="center", fontsize=9,
            color=GRAY, style="italic",
            bbox=dict(boxstyle="round,pad=0.3", fc="#f4f7ff",
                      ec=GRAY, lw=0.8))

    # ── Legend ───────────────────────────────────────────────
    legend_items = [
        mpatches.Patch(fc=UHBLUE,    label="Real data / ResNet18"),
        mpatches.Patch(fc=TEAL,      label="Synthetic data / BaselineCNN"),
        mpatches.Patch(fc=AMBER,     label="Cross-validation"),
        mpatches.Patch(fc=CORAL,     label="Mixed test evaluation"),
        mpatches.Patch(fc="#5C6BC0", label="Real-only evaluation"),
        mpatches.Patch(fc=GRAY,      label="Floor / metrics"),
    ]
    ax.legend(
        handles=legend_items,
        loc="lower left", bbox_to_anchor=(0.0, 0.0),
        fontsize=8.5, ncol=3, framealpha=0.92,
        edgecolor=GRAY, fancybox=True,
    )

    # ── Title ────────────────────────────────────────────────
    ax.set_title(
        "Experimental pipeline — UHasselt DSI × Gpixel NV",
        fontsize=13, fontweight="bold", color=UHBLUE, pad=8,
    )

    out = OUT_DIR / "pipeline_diagram.png"
    fig.savefig(out, dpi=220, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"✓  Saved: {out}  ({out.stat().st_size/1024:.1f} KB)")
    return out


# ════════════════════════════════════════════════════════════
# RUN BOTH
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("Generating thesis figures...\n")
    p1 = make_real_defect_panel()
    p2 = make_pipeline_diagram()
    print("\nDone. Download these two files from /kaggle/working/:")
    print(f"  {p1.name}")
    print(f"  {p2.name}")
    print("\nUpload both to the root of your Overleaf project.")
    print("Then replace the \\imagebox placeholders with:")
    print("  \\includegraphics[width=0.95\\textwidth]{real_defect_examples.png}")
    print("  \\includegraphics[width=0.90\\textwidth]{pipeline_diagram.png}")

Generating thesis figures...

✓  Saved: /kaggle/working/real_defect_examples.png  (63.1 KB)
✓  Saved: /kaggle/working/pipeline_diagram.png  (295.1 KB)

Done. Download these two files from /kaggle/working/:
  real_defect_examples.png
  pipeline_diagram.png

Upload both to the root of your Overleaf project.
Then replace the \imagebox placeholders with:
  \includegraphics[width=0.95\textwidth]{real_defect_examples.png}
  \includegraphics[width=0.90\textwidth]{pipeline_diagram.png}
